## 基本操作

In [1]:
import re
import os
import gc
import dask
import pymysql
import warnings
import numpy as np
import pandas as pd
import dask.dataframe as dd
from dask.distributed import Client
from sqlalchemy import create_engine, exc as sa_exc
from datetime import datetime, timedelta
from pandas.errors import SettingWithCopyWarning
warnings.filterwarnings("ignore", category = UserWarning, module='openpyxl')
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, message="pandas only supports SQLAlchemy connectable")
warnings.filterwarnings('ignore', category=FutureWarning, module='pandas')
warnings.filterwarnings('ignore', category=SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=sa_exc.SAWarning, message="Unrecognized server version info")

In [3]:
from datetime import datetime
import pymysql

# 固定 2026年2月 核心变量（不改变你要的变量名）
db_name = "2026年2月盈亏表数据"
file_time_str = "2026-02-28"
file_date = datetime.strptime(file_time_str, '%Y-%m-%d')
month = "02"

# 连接数据库
try:
    conn = pymysql.connect(
        host='192.168.30.51',
        user='mysql',
        password='QL132.465',
        database=db_name,
        charset='utf8mb4'
    )
    print("✅ 成功连接数据库：", db_name)

except pymysql.MySQLError as e:
    print("❌ 数据库连接失败：", e)
# 连接数据库
try:
    # 2. 配置 SQL Server 连接信息
    server = "192.168.30.46"    # 你的服务器名
    database = "BI数据"     # 数据库名
    username = "QLBP"         # 用户名
    password = "QL.132.BP"     # 密码
    driver = "ODBC Driver 17 for SQL Server"  # 驱动（几乎都用这个）
    # 3. 创建连接引擎
    connection_string = (
        f"mssql+pyodbc://{username}:{password}@{server}/{database}"
        f"?driver={driver}&TrustServerCertificate=yes"
    )
    conn2 = create_engine(connection_string)
    # 测试连接是否真的可用
    with conn2.connect() as test_conn:
        print("✅ 成功连接数据库：", 'BI数据')

except Exception  as e:
    print("❌ 数据库连接失败：", e)

✅ 成功连接数据库： 2026年2月盈亏表数据
✅ 成功连接数据库： BI数据


In [4]:
dp_bg = pd.read_sql(f'select * from 店铺表格修改后', conn)

## 函数

In [5]:
def convert_date_simple(series):
    # 定义去重后的格式列表
    formats = [
        '%Y-%m-%d', '%Y/%m/%d',
        '%Y-%m-%d %H:%M', '%Y/%m/%d %H:%M',
        '%Y-%m-%d %H:%M:%S', '%Y/%m/%d %H:%M:%S'
    ]

    # 初始化结果为原数据
    result = series.copy()

    # 依次尝试每种格式，只转换未成功的部分
    for fmt in formats:
        # 筛选出还未转换成功的非空值
        mask = pd.notna(result) & (result == series)
        if not mask.any():
            break

        # 向量化转换
        converted = pd.to_datetime(series[mask], format=fmt, errors='coerce')
        # 只更新转换成功的值
        result[mask & pd.notna(converted)] = converted.dt.strftime('%Y-%m-%d')

    return result

In [6]:
def optimize_field_by_date(df, field_name):
    """通用函数：按日期规则处理指定字段（合伙人/品牌）"""
    # 1. 转换日期（批量向量化）
    df['发生时间_dt'] = pd.to_datetime(df['发生时间'], errors='coerce')
    df['日期_dt'] = pd.to_datetime(df['日期'], errors='coerce')

    # 2. 筛选有效行（日期非空 + 字段含/）
    mask = (
        ~df['发生时间_dt'].isna()
        & ~df['日期_dt'].isna()
        & df[field_name].astype(str).str.contains('/', na=False)
    )

    # 3. 批量处理有效行
    if mask.any():
        split_vals = df.loc[mask, field_name].str.split('/', expand=True)
        cond = df.loc[mask, '发生时间_dt'] <= df.loc[mask, '日期_dt']
        df.loc[mask, field_name] = np.where(cond, split_vals[0], split_vals[1])

    # 清理临时列
    df = df.drop(columns=['发生时间_dt', '日期_dt'])
    return df

In [7]:
def save_dataframes(dataframes, file_paths, overwrite=True):
    """
    保存多个DataFrame到指定路径
    - 只在DataFrame有数据时生成文件
    - 如果DataFrame为空，删除已存在的对应文件

    参数:
        dataframes: DataFrame列表
        file_paths: 文件路径列表
        overwrite: 是否覆盖已存在的文件
    """
    # 确保输入的列表长度相同
    if len(dataframes) != len(file_paths):
        raise ValueError("DataFrames和文件路径数量不匹配")

    for df, path in zip(dataframes, file_paths):
        # 先创建目录（确保路径合法，即使删除文件也需要目录存在）
        dir_path = os.path.dirname(path)
        os.makedirs(dir_path, exist_ok=True)

        # 核心逻辑：先判断数据是否为空
        if df is None or df.empty:
            # 数据为空时，删除已存在的文件
            if os.path.exists(path):
                os.remove(path)
                print(f"数据为空，已删除旧文件: {path}")
            else:
                print(f"跳过空数据: {path} (无旧文件可删除)")
            continue  # 空数据处理完直接进入下一个循环

        # 数据非空时，执行覆盖和保存逻辑
        if overwrite and os.path.exists(path):
            os.remove(path)

        # 保存DataFrame到CSV
        df.to_csv(path, index=False)
        print(f"已保存数据到: {path}")

In [8]:
def get_new_subject(row):
    # 1. 安全解析日期，过滤无效值
    date_strings = [d.strip() for d in row['变更日期'].split(',') if d.strip()]
    dates = []
    for d in date_strings:
        try:
            parsed_date = pd.to_datetime(d)
            dates.append(parsed_date)
        except (ValueError, TypeError):
            continue  # 跳过无法解析的日期

    # 2. 安全解析主体
    subjects = [s.strip() for s in row['主体'].split('-') if s.strip()]
    month = row['发生时间']

    # 3. 按原逻辑匹配
    if len(dates) == 1 and len(subjects) == 2:
        idx = 0 if month < dates[0] else 1
        return subjects[idx]
    elif len(dates) == 2 and len(subjects) == 3:
        idx = (month >= dates[0]) + (month >= dates[1])
        return subjects[idx]
    else:
        return row['主体']

## 主体

In [9]:
zt2 = pd.read_sql('select * from 主体身份', conn).rename(columns = {'公司':'主体'})

In [10]:
import pandas as pd

# 读取 + 基础处理
zt1 = pd.read_excel(r'../主体.xlsx', sheet_name = 'Sheet1').rename(columns = {'公司':'主体'})
zt1['变更日期'] = zt1['变更日期'].dt.strftime('%Y-%m-%d').fillna('0').astype(str).str.strip().replace('', '0')
zt1['店铺ID'] = zt1['聚水潭店铺编号'].astype(str).str.replace(' ', '')

# 极简分组聚合
def agg_func(lst):
    v = [x for x in lst if x not in ('0', '', 'nan')]
    return ','.join(sorted(set(v))) if v else '0'

zt3 = zt1.groupby('店铺ID', as_index=False).agg(
    主体=('主体', lambda x: '-'.join(x.dropna().astype(str).unique())),
    变更日期=('变更日期', agg_func)
)

## 拼多多后台

### 对账中心

In [11]:
ff = pd.read_sql(f'select * from `对账中心(原导)`', conn)

In [12]:
ff = ff.replace(' ', '').replace('', np.nan).fillna(0)

In [13]:
ff['店铺'] = ff['店铺'].str.split('_').str[0]

In [14]:
ff = ff[(ff['发生时间'].str[:7] == f'2026-{month}')]

In [15]:
ff['发生时间'] = convert_date_simple(ff['发生时间'])

In [16]:
ff_1 = ff.copy()

In [17]:
ff_1['收入金额（+元）'] = ff_1['收入金额（+元）'].astype(float)
ff_1['支出金额（_元）'] = ff_1['支出金额（_元）'].astype(float)

In [18]:
ff_1.rename(columns = {'收入金额（+元）':'收入金额', '支出金额（_元）':'支出金额'}, inplace = True)

#### 区分类型

In [19]:
# #  ====================== 初始化 ======================
# ff_1['类型'] = '未知'
# ff_1['合计'] = ff_1['收入金额'] - ff_1['支出金额']
#
# # ====================== 工具函数：一行定义规则 ======================
# def set_rule(df, cond, type_name, amount_val):
#     """满足cond，就自动赋值 类型 和 合计"""
#     df.loc[cond, '类型'] = type_name
#     df.loc[cond, '合计'] = amount_val
#
# # ====================== 【核心：所有规则，一行一个！】 ======================
# # 格式：set_rule(数据表, 条件, 类型名称, 金额)
# # 顺序随便放！新增只需要加一行！
#
# # 销售额
# set_rule(ff_1, ff_1['账务类型']=='交易收入', '销售额-订单收入', ff_1['收入金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='优惠券结算')&(ff_1['业务描述'].str.contains('交易收入-优惠券结算',na=False)), '销售额-优惠券收入', ff_1['收入金额'].abs())
# set_rule(ff_1, ff_1['账务类型'].isin(['其他','其它'])&(ff_1['业务描述'].str.contains('其他收入-多单立减',na=False)), '销售额-多单立减', ff_1['收入金额'].abs())
# set_rule(ff_1, ff_1['账务类型'].isin(['其他','其它'])&(ff_1['业务描述'].str.contains('平台补贴-补贴汇入',na=False)), '销售额-平台补贴收入', ff_1['收入金额'].abs())
#
# # 退款
# set_rule(ff_1, (ff_1['账务类型']=='优惠券结算')&(ff_1['业务描述'].str.contains('交易退款-优惠券结算',na=False)), '退款-优惠券退款', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='退款')&(ff_1['业务描述'].str.contains('交易退款-订单退款',na=False)), '退款-订单退款', ff_1['支出金额'].abs())
# set_rule(ff_1, ff_1['账务类型'].isin(['其他','其它'])&(ff_1['业务描述'].str.contains('其他支出-多单立减',na=False)), '退款-多单立减', ff_1['支出金额'].abs())
#
# # 扣点 - 技术服务费
# set_rule(ff_1, (ff_1['账务类型']=='技术服务费')&ff_1['业务描述'].str.contains('基础技术服务费',na=False), '扣点-技术服务费', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='技术服务费')&(ff_1['业务描述'].str.contains('基础技术服务费',na=False)) &
#          (ff_1['备注'].str.contains('返还',na=False)), '扣点-技术服务费', -ff_1['收入金额'])
#
# set_rule(ff_1, (ff_1['账务类型']=='技术服务费')&ff_1['业务描述'].str.contains('黑标技术服务费',na=False), '扣点-技术服务费', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='技术服务费')&(ff_1['业务描述'].str.contains('黑标技术服务费',na=False)) &
#          (ff_1['备注'].str.contains('返还',na=False)), '扣点-技术服务费', -ff_1['收入金额'])
#
# set_rule(ff_1, (ff_1['账务类型']=='技术服务费') & (ff_1['业务描述'].str.contains('直播技术服务费',na=False)), '扣点-技术服务费',ff_1['支出金额'].abs())
# set_rule(ff_1,(ff_1['账务类型']=='技术服务费') &(ff_1['业务描述'].str.contains('直播技术服务费',na=False)) &
#     (ff_1['备注'].str.contains('返还',na=False)),'扣点-技术服务费',-ff_1['收入金额'])
#
# set_rule(ff_1, ff_1['账务类型'].isin(['其他','其它'])&(ff_1['业务描述'].str.contains('技术服务费', na  = False)), '扣点-技术服务费', ff_1['支出金额'].abs())
# set_rule(ff_1, ff_1['账务类型'].isin(['其他','其它'])&(ff_1['业务描述'].str.contains('技术服务费', na  = False)) &
#          ff_1['备注'].str.contains('返还',na=False), '扣点-技术服务费', -ff_1['收入金额'])
#
# # 扣点 - 服务支出
# set_rule(ff_1, (ff_1['账务类型']=='服务消费')&(ff_1['业务描述'].str.contains('交易付款-服务消费',na=False)), '扣点-服务支出', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='服务消费')&(ff_1['业务描述'].str.contains('交易退回-服务消费退款',na=False)), '扣点-服务支出', -ff_1['收入金额'])
#
# set_rule(ff_1, (ff_1['账务类型']=='其他服务')&(ff_1['业务描述'].str.contains('服务支出-物流提醒短信|服务支出-24小时发货|服务支出-消费者体验提升计划',na=False)), '扣点-服务支出', ff_1['支出金额'].abs())
#
# set_rule(ff_1, (ff_1['账务类型']=='转账')&ff_1['备注'].str.contains('评价有礼金服务',na=False)&
#          (ff_1['业务描述'].str.contains('转账-广告账户',na=False)), '扣点-评价有礼金服务', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='转账')&ff_1['备注'].str.contains('任务神器',na=False)&
#          (ff_1['业务描述'].str.contains('转账-广告账户',na=False)), '扣点-任务神器', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='转账')&ff_1['备注'].str.contains('转账到短信',na=False)&
#          (ff_1['业务描述'].str.contains('转账-广告账户',na=False)), '扣点-转账短信', ff_1['支出金额'].abs())
#
# # 扣款
# set_rule(ff_1, (ff_1['账务类型']=='多多进宝')&~ff_1['备注'].str.contains('返还',na=False), '扣款-多多进宝', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='多多进宝')&ff_1['备注'].str.contains('返还',na=False), '扣款-多多进宝', -ff_1['收入金额'])
#
# set_rule(ff_1, (ff_1['账务类型'].isin(['退款', '扣款']))&(ff_1['业务描述'].str. contains('运费补偿|售后费用-更高物流履约承诺未兑现', na  = False)), '扣款-运费补偿', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='扣款')&(ff_1['业务描述'].str. contains('运费补偿', na  = False))&ff_1['备注'].str.contains('退回',na=False), '扣款-运费补偿', -ff_1['收入金额'])
# set_rule(ff_1, ff_1['账务类型'].isin(['其他','其它'])&(ff_1['业务描述'].str. contains('申诉补回', na  = False)), '扣款-申诉补回', -ff_1['收入金额'])
# set_rule(ff_1, ff_1['账务类型'].isin(['其他','其它'])&(ff_1['业务描述'].str. contains('欺诈发货', na  = False)), '扣款-欺诈发货', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='扣款')&(ff_1['业务描述'].str. contains('售后补偿', na  = False)), '扣款-售后补偿消费者', ff_1['支出金额'].abs())
#
# set_rule(ff_1, (ff_1['账务类型']=='扣款')&(ff_1['业务描述'].str. contains('延迟发货', na  = False)), '扣款-延迟发货', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='扣款')&(ff_1['业务描述'].str. contains('虚假发货', na  = False)), '扣款-虚假发货', ff_1['支出金额'].abs())
#
# set_rule(ff_1, ff_1['账务类型'].isin(['其他','其它'])&(ff_1['业务描述'].str. contains('小额打款', na  = False)), '扣款-小额打款', ff_1['支出金额'].abs())
# set_rule(ff_1, ff_1['账务类型'].isin(['其他','其它'])&(ff_1['业务描述'].str. contains('小额打款', na  = False)) &
#          ff_1['备注'].str.contains('打款失败',na=False), '扣款-小额打款', -ff_1['收入金额'])
#
# set_rule(ff_1, (ff_1['账务类型']=='转账')&ff_1['备注'].str.contains('单店满返',na=False)&
#          (ff_1['业务描述'].str.contains('转账-营销账户',na=False)), '扣款-单店满返', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='转账')&ff_1['备注'].str.contains('评价有礼',na=False)&
#          (ff_1['业务描述'].str.contains('转账-营销账户',na=False)), '扣款-评价有礼', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='转账')&ff_1['备注'].str.contains('跨店满返',na=False)&
#          (ff_1['业务描述'].str.contains('转账-营销账户',na=False)), '扣款-日常跨店满返', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='转账')&ff_1['备注'].str.contains('评价有礼金',na=False)&
#          (ff_1['业务描述'].str.contains('转账-广告账户',na=False)), '扣款-评价有礼', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='转账')&ff_1['备注'].str.contains('粉丝抢福利',na=False)&
#          (ff_1['业务描述'].str.contains('转账-广告账户',na=False)), '扣款-粉丝福利', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='扣款')&(ff_1['业务描述'].str. contains('延迟送达', na  = False)), '扣款-延迟送达', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='扣款')&(ff_1['业务描述'].str. contains('缺货', na  = False)), '扣款-缺货', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='扣款')&(ff_1['业务描述'].str. contains('售后费用-发票服务承诺未兑现', na  = False)), '扣款-发票服务承诺未兑现', ff_1['支出金额'].abs())
#
# # 保证金
# set_rule(ff_1, ff_1['账务类型'].isin(['其他','其它'])&(ff_1['业务描述'].str. contains('店铺保证金', na  = False))&
#          ff_1['备注'].str.contains('转入货款',na=False), '保证金-店铺保证金', -ff_1['收入金额'])
# set_rule(ff_1, (ff_1['账务类型']=='转账')&(ff_1['业务描述'].str. contains('活动保证金', na  = False)), '保证金-活动保证金', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='转账')&(ff_1['业务描述'].str. contains('店铺保证金', na  = False))&
#          ff_1['备注'].str.contains('充值|转账',na=False), '保证金-店铺保证金', ff_1['支出金额'].abs())
#
# # 直通车
# set_rule(ff_1, (ff_1['账务类型']=='转账')&ff_1['备注'].str.contains('货款转账到推广账户',na=False)&
#          (ff_1['业务描述'].str. contains('转账-广告账户', na  = False)), '直通车-充值', ff_1['支出金额'].abs())
#
# # 基础映射（最后兜底）
# set_rule(ff_1, (ff_1['账务类型']=='分账') & (ff_1['业务描述']==0) & (ff_1['备注']==0), '扣点-物流费', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='提现') & (ff_1['业务描述']==0) & (ff_1['备注']==0), '提现', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='其他') & (ff_1['业务描述']==0) & (ff_1['备注']==0), '扣款-其他', ff_1['支出金额'].abs())
# set_rule(ff_1, (ff_1['账务类型']=='交易收入') & (ff_1['业务描述']==0) & (ff_1['备注']==0), '扣款-其他', ff_1['支出金额'].abs())

In [20]:
import pandas as pd
import numpy as np

# ====================== 初始化 ======================
ff_1['类型'] = '未知'
# 合计列后续单独计算，此处不做规则赋值

# 列别名
tx = ff_1['账务类型']
desc = ff_1['业务描述']
note = ff_1['备注']

# 通用判断简写
note_has = lambda s: note.str.contains(s, na=False)
desc_has = lambda s: desc.str.contains(s, na=False)
tx_in = lambda lst: tx.isin(lst)

# ===================== 规则：仅赋值 类型 列 ======================
# 销售额
ff_1.loc[(tx == '优惠券结算') & desc_has('交易收入-优惠券结算'), '类型'] = '销售额-优惠券收入'
ff_1.loc[tx_in(['其他','其它']) & desc_has('其他收入-多单立减'), '类型'] = '销售额-多单立减'
ff_1.loc[tx_in(['其他','其它']) & desc_has('平台补贴-补贴汇入'), '类型'] = '销售额-平台补贴收入'

# 退款
ff_1.loc[(tx == '优惠券结算') & desc_has('交易退款-优惠券结算'), '类型'] = '退款-优惠券退款'
ff_1.loc[(tx == '退款') & desc_has('交易退款-订单退款'), '类型'] = '退款-订单退款'
ff_1.loc[tx_in(['其他','其它']) & desc_has('其他支出-多单立减'), '类型'] = '退款-多单立减'

# 扣点 - 技术服务费
ff_1.loc[(tx == '技术服务费') & desc_has('基础技术服务费'), '类型'] = '扣点-技术服务费'
ff_1.loc[(tx == '技术服务费') & desc_has('基础技术服务费') & note_has('返还'), '类型'] = '扣点-技术服务费'

ff_1.loc[(tx == '技术服务费') & desc_has('黑标技术服务费'), '类型'] = '扣点-技术服务费'
ff_1.loc[(tx == '技术服务费') & desc_has('黑标技术服务费') & note_has('返还'), '类型'] = '扣点-技术服务费'

ff_1.loc[(tx == '技术服务费') & desc_has('直播技术服务费'), '类型'] = '扣点-技术服务费'
ff_1.loc[(tx == '技术服务费') & desc_has('直播技术服务费') & note_has('返还'), '类型'] = '扣点-技术服务费'

ff_1.loc[tx_in(['其他','其它']) & desc_has('技术服务费'), '类型'] = '扣点-技术服务费'
ff_1.loc[tx_in(['其他','其它']) & desc_has('技术服务费') & note_has('返还'), '类型'] = '扣点-技术服务费'

# 扣点 - 服务支出
ff_1.loc[(tx == '服务消费') & desc_has('交易付款-服务消费'), '类型'] = '扣点-服务支出'
ff_1.loc[(tx == '服务消费') & desc_has('交易退回-服务消费退款'), '类型'] = '扣点-服务支出'

ff_1.loc[(tx == '其他服务') & desc_has('服务支出-物流提醒短信|服务支出-24小时发货|服务支出-消费者体验提升计划'), '类型'] = '扣点-服务支出'

ff_1.loc[(tx == '转账') & note_has('评价有礼金服务') & desc_has('转账-广告账户'), '类型'] = '扣点-评价有礼金服务'
ff_1.loc[(tx == '转账') & note_has('任务神器') & desc_has('转账-广告账户'), '类型'] = '扣点-任务神器'
ff_1.loc[(tx == '转账') & note_has('转账到短信') & desc_has('转账-广告账户'), '类型'] = '扣点-转账短信'

# 扣款
ff_1.loc[(tx == '多多进宝') & ~note_has('返还'), '类型'] = '扣款-多多进宝'
ff_1.loc[(tx == '多多进宝') & note_has('返还'), '类型'] = '扣款-多多进宝'

ff_1.loc[tx_in(['退款', '扣款']) & desc_has('运费补偿|售后费用-更高物流履约承诺未兑现'), '类型'] = '扣款-运费补偿'
ff_1.loc[(tx == '扣款') & desc_has('运费补偿') & note_has('退回'), '类型'] = '扣款-运费补偿'
ff_1.loc[tx_in(['其他','其它']) & desc_has('申诉补回'), '类型'] = '扣款-申诉补回'
ff_1.loc[tx_in(['其他','其它']) & desc_has('欺诈发货'), '类型'] = '扣款-欺诈发货'
ff_1.loc[(tx == '扣款') & desc_has('售后补偿'), '类型'] = '扣款-售后补偿消费者'

ff_1.loc[(tx == '扣款') & desc_has('延迟发货'), '类型'] = '扣款-延迟发货'
ff_1.loc[(tx == '扣款') & desc_has('虚假发货'), '类型'] = '扣款-虚假发货'

ff_1.loc[tx_in(['其他','其它']) & desc_has('小额打款'), '类型'] = '扣款-小额打款'
ff_1.loc[tx_in(['其他','其它']) & desc_has('小额打款') & note_has('打款失败'), '类型'] = '扣款-小额打款'

ff_1.loc[(tx == '转账') & note_has('单店满返') & desc_has('转账-营销账户'), '类型'] = '扣款-单店满返'
ff_1.loc[(tx == '转账') & note_has('评价有礼') & desc_has('转账-营销账户'), '类型'] = '扣款-评价有礼'
ff_1.loc[(tx == '转账') & note_has('跨店满返') & desc_has('转账-营销账户'), '类型'] = '扣款-日常跨店满返'
ff_1.loc[(tx == '转账') & note_has('评价有礼金') & desc_has('转账-广告账户'), '类型'] = '扣款-评价有礼'
ff_1.loc[(tx == '转账') & note_has('粉丝抢福利') & desc_has('转账-广告账户'), '类型'] = '扣款-粉丝福利'
ff_1.loc[(tx == '扣款') & desc_has('延迟送达'), '类型'] = '扣款-延迟送达'
ff_1.loc[(tx == '扣款') & desc_has('缺货'), '类型'] = '扣款-缺货'
ff_1.loc[(tx == '扣款') & desc_has('售后费用-发票服务承诺未兑现'), '类型'] = '扣款-发票服务承诺未兑现'

# 保证金
ff_1.loc[tx_in(['其他','其它']) & desc_has('店铺保证金') & note_has('转入货款'), '类型'] = '保证金-店铺保证金'
ff_1.loc[(tx == '转账') & desc_has('活动保证金'), '类型'] = '保证金-活动保证金'
ff_1.loc[(tx == '转账') & desc_has('店铺保证金') & note_has('充值|转账'), '类型'] = '保证金-店铺保证金'

# 直通车
ff_1.loc[(tx == '转账') & note_has('货款转账到推广账户') & desc_has('转账-广告账户'), '类型'] = '直通车-充值'

# 兜底规则
ff_1.loc[(tx == '分账') & (desc == 0) & (note == 0), '类型'] = '扣点-物流费'
ff_1.loc[(tx == '提现') & (desc == 0) & (note == 0), '类型'] = '提现'
ff_1.loc[(tx == '其他') & (desc == 0) & (note == 0), '类型'] = '扣款-其他'
ff_1.loc[(tx == '交易收入') & (desc == 0) & (note == 0), '类型'] = '扣款-其他'

In [21]:
import pandas as pd

# ===================== ff_1 专用 金额规则配置 =====================
# 支出类固定类型（取绝对值）
EXPENSE_TYPES = {
    '扣点-技术服务费', '扣点-服务支出', '扣点-评价有礼金服务',
    '扣点-任务神器', '扣点-转账短信', '扣款-多多进宝', '扣款-运费补偿',
    '扣款-申诉补回', '扣款-欺诈发货', '扣款-售后补偿消费者',
    '扣款-延迟发货', '扣款-虚假发货', '扣款-小额打款',
    '扣款-单店满返', '扣款-评价有礼', '扣款-日常跨店满返',
    '扣款-粉丝福利', '扣款-延迟送达', '扣款-缺货',
    '扣款-发票服务承诺未兑现', '保证金-活动保证金',
    '保证金-店铺保证金', '直通车-充值', '扣点-物流费',
    '提现', '扣款-其他'
}

# 收入需要取负的类型
INCOME_NEG_TYPES = {
    '扣点-技术服务费', '扣款-多多进宝', '扣款-运费补偿',
    '扣款-小额打款', '保证金-店铺保证金'
}

# 销售额固定类型（收入取绝对值）
SALES_FIXED = {
    '销售额-订单收入', '销售额-优惠券收入', '销售额-多单立减',
    '销售额-平台补贴收入'
}

# ===================== 核心金额逻辑（和你给的格式完全一致） =====================
ap = ''  # 没有就留空，不影响运行

# 三大掩码（逻辑完全一样）
mask_abs = (ff_1['类型'].isin(SALES_FIXED)) & ff_1['收入金额'].ne(0)
mask_neg = (ff_1['类型'].isin(INCOME_NEG_TYPES) | (ff_1['类型'] == ap)) & ff_1['收入金额'].ne(0)
mask_type = (ff_1['类型'].isin(EXPENSE_TYPES) | (ff_1['类型'] == ap)) & ff_1['支出金额'].ne(0)

# 金额赋值
ff_1.loc[mask_neg, '收入金额'] *= -1
ff_1.loc[mask_abs, '收入金额'] = ff_1['收入金额'].abs()
ff_1.loc[mask_type, '支出金额'] = ff_1['支出金额'].abs()

In [22]:
ff_1['合计'] = round(ff_1['收入金额'] + ff_1['支出金额'],3)

In [23]:
## 未知
ff_1.loc[
    ff_1['类型'].isin([
        '未知'
    ]), '合计'] = abs(ff_1['合计'])

In [24]:
ff_1[ff_1['类型'] == '扣点-技术服务费']

,店铺,商户订单号,发生时间,收入金额,支出金额,账务类型,备注,业务描述,类型,合计
6,11931578,260224-001971351080638,2026-02-28,0.00,0.17,技术服务费,-,0030002|技术服务费-基础技术服务费,扣点-技术服务费,0.17
7,11931578,260221-676509806230115,2026-02-28,0.00,0.17,技术服务费,-,0030002|技术服务费-基础技术服务费,扣点-技术服务费,0.17
11,11931578,260228-039625756973582,2026-02-28,0.00,0.11,技术服务费,-,0030002|技术服务费-基础技术服务费,扣点-技术服务费,0.11
12,11931578,260218-224080740731377,2026-02-28,0.00,0.22,技术服务费,-,0030002|技术服务费-基础技术服务费,扣点-技术服务费,0.22
20,11931578,260228-493092902511170,2026-02-28,-0.17,0.00,技术服务费,基础技术服务费返还,0030002|技术服务费-基础技术服务费,扣点-技术服务费,-0.17
...,...,...,...,...,...,...,...,...,...,...
1254722,18230774,260128-445141533082130,2026-02-02,0.00,0.24,技术服务费,-,0030002|技术服务费-基础技术服务费,扣点-技术服务费,0.24
1254723,18230774,260124-583280893441704,2026-02-01,-0.36,0.00,技术服务费,基础技术服务费返还,0030002|技术服务费-基础技术服务费,扣点-技术服务费,-0.36
1254726,18230774,260129-645964766852178,2026-02-01,0.00,0.42,技术服务费,-,0030002|技术服务费-基础技术服务费,扣点-技术服务费,0.42
1254730,18230774,260130-139806718313401,2026-02-01,0.00,0.36,技术服务费,-,0030002|技术服务费-基础技术服务费,扣点-技术服务费,0.36


In [25]:
ff_1['类型'].value_counts()

类型
扣点-技术服务费        448798
未知              362152
扣点-服务支出         244482
销售额-优惠券收入       112941
退款-订单退款          50889
退款-优惠券退款         17704
直通车-充值            8249
扣款-小额打款           2811
扣款-售后补偿消费者        2212
扣款-延迟发货           1431
扣款-运费补偿            922
扣款-评价有礼            506
扣款-虚假发货            422
扣款-多多进宝            242
扣款-申诉补回            139
扣款-缺货               66
销售额-多单立减            17
扣款-单店满返             13
扣款-发票服务承诺未兑现         1
Name: count, dtype: int64

#### 店铺表格

In [26]:
ff_1['店铺'] = ff_1['店铺'].astype(int)
dp_bg['店铺ID'] = dp_bg['店铺ID'].astype(int)

In [27]:
ff_2 = pd.merge(ff_1, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [28]:
# 主执行逻辑（仅2行核心调用）
ff_2 = ff_2.copy()
for field in ['合伙人', '品牌']:
    ff_2 = optimize_field_by_date(ff_2, field)

#### 未知类型

In [29]:
ff_2_wz = ff_2[(ff_2['类型'] == '未知') & (ff_2['合计'] != 0)]
ff_2_wz

,店铺,商户订单号,发生时间,收入金额,支出金额,账务类型,备注,业务描述,类型,合计,店铺名称（聚水潭）,店铺ID,平台,合伙人,品牌,状态,日期
5,11931578,260224-001971351080638,2026-02-28,27.80,0.0,交易收入,-,0010002|交易收入-订单收入,未知,27.80,白珍珠依酷专营店（白腾）,11931578,拼多多,许文腾,白珍珠,正常,None
8,11931578,260221-676509806230115,2026-02-28,27.80,0.0,交易收入,-,0010002|交易收入-订单收入,未知,27.80,白珍珠依酷专营店（白腾）,11931578,拼多多,许文腾,白珍珠,正常,None
10,11931578,260228-039625756973582,2026-02-28,17.91,0.0,交易收入,-,0010002|交易收入-订单收入,未知,17.91,白珍珠依酷专营店（白腾）,11931578,拼多多,许文腾,白珍珠,正常,None
13,11931578,260218-224080740731377,2026-02-28,36.80,0.0,交易收入,-,0010002|交易收入-订单收入,未知,36.80,白珍珠依酷专营店（白腾）,11931578,拼多多,许文腾,白珍珠,正常,None
19,11931578,260228-493092902511170,2026-02-28,27.80,0.0,交易收入,-,0010002|交易收入-订单收入,未知,27.80,白珍珠依酷专营店（白腾）,11931578,拼多多,许文腾,白珍珠,正常,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1253979,18230774,260122-065431160450798,2026-02-02,39.90,0.0,交易收入,-,0010002|交易收入-订单收入,未知,39.90,红蜻蜓柔薇专卖店（红晓),18230774,拼多多,陈晓斌,红蜻蜓,正常,None
1253981,18230774,260128-445141533082130,2026-02-02,39.90,0.0,交易收入,-,0010002|交易收入-订单收入,未知,39.90,红蜻蜓柔薇专卖店（红晓),18230774,拼多多,陈晓斌,红蜻蜓,正常,None
1253987,18230774,260129-645964766852178,2026-02-01,69.90,0.0,交易收入,-,0010002|交易收入-订单收入,未知,69.90,红蜻蜓柔薇专卖店（红晓),18230774,拼多多,陈晓斌,红蜻蜓,正常,None
1253988,18230774,260130-139806718313401,2026-02-01,41.93,0.0,交易收入,-,0010002|交易收入-订单收入,未知,41.93,红蜻蜓柔薇专卖店（红晓),18230774,拼多多,陈晓斌,红蜻蜓,正常,None


#### 汇总-直通车

In [85]:
ff_4 = ff_2[ff_2['类型'].str.contains('直通车', na = False)]

In [86]:
ff_5 = ff_4.groupby(['店铺名称（聚水潭）', '店铺ID', '发生时间', '平台', '合伙人', '品牌'])['合计'].sum().reset_index().rename(columns = {'店铺名称（聚水潭）':'店铺名称'})

### 拼多多推广

In [87]:
fn_tg = pd.read_sql(f'select * from 拼多多_推广', conn)

In [88]:
# 一行调用，直接处理整列
fn_tg['时间'] = convert_date_simple(fn_tg['时间'])

In [89]:
fn_tg.rename(columns = {'时间': '发生时间'}, inplace = True)

In [90]:
fn_tg['店铺'] = fn_tg['店铺'].astype(int)
fn_tg['交易金额'] = fn_tg['交易金额'].astype(float)

In [91]:
fn_tg['类型'] = np.where((fn_tg['资金类型'].str.contains('现金', na = False)) &
                         (fn_tg['流水类型'].str.contains('支出', na = False)), '直通车-支出',

               np.where((fn_tg['资金类型'].str.contains('现金', na = False)) &
                         (fn_tg['流水类型'].str.contains('收入', na = False)), '直通车-收入',

               np.where(fn_tg['资金类型'].str.contains('红包', na = False), '不算', '未知')))

#### 店铺表格

In [92]:
fn_tg1 = pd.merge(fn_tg, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')
fn_tg1

,店铺,发生时间,资金类型,流水类型,店铺名称,交易金额,余额,交易摘要,类型,店铺名称（聚水潭）,店铺ID,平台,合伙人,品牌,状态,日期
0,11931578,2026-02-02,现金,支出,依酷内衣专营店,129.55,370.79,推广支出： 商品推广129.55元；,直通车-支出,白珍珠依酷专营店（白腾）,11931578,拼多多,许文腾,白珍珠,正常,None
1,11931578,2026-02-02,现金,收入,依酷内衣专营店,400.00,500.34,现金充值-账户实时余额低于设定值，自动充值生效,直通车-收入,白珍珠依酷专营店（白腾）,11931578,拼多多,许文腾,白珍珠,正常,None
2,11931578,2026-02-01,红包,支出,依酷内衣专营店,5.17,0.00,推广支出【券ID：217501789】： 商品推广5.17元；,不算,白珍珠依酷专营店（白腾）,11931578,拼多多,许文腾,白珍珠,正常,None
3,11931578,2026-02-01,现金,支出,依酷内衣专营店,105.20,100.34,推广支出： 商品推广105.20元；,直通车-支出,白珍珠依酷专营店（白腾）,11931578,拼多多,许文腾,白珍珠,正常,None
4,11931578,2026-02-01,红包,收入,依酷内衣专营店,5.17,--,红包发放【券ID：217501789】,不算,白珍珠依酷专营店（白腾）,11931578,拼多多,许文腾,白珍珠,正常,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14662,18230774,2026-02-04,现金,支出,红蜻蜓柔薇专卖店,260.94,541.41,推广支出： 商品推广260.94元；,直通车-支出,红蜻蜓柔薇专卖店（红晓),18230774,拼多多,陈晓斌,红蜻蜓,正常,None
14663,18230774,2026-02-04,现金,收入,红蜻蜓柔薇专卖店,500.00,802.35,现金充值-账户实时余额低于设定值，自动充值生效,直通车-收入,红蜻蜓柔薇专卖店（红晓),18230774,拼多多,陈晓斌,红蜻蜓,正常,None
14664,18230774,2026-02-03,现金,支出,红蜻蜓柔薇专卖店,106.82,302.35,推广支出： 商品推广106.82元；,直通车-支出,红蜻蜓柔薇专卖店（红晓),18230774,拼多多,陈晓斌,红蜻蜓,正常,None
14665,18230774,2026-02-02,现金,支出,红蜻蜓柔薇专卖店,136.11,409.17,推广支出： 商品推广136.11元；,直通车-支出,红蜻蜓柔薇专卖店（红晓),18230774,拼多多,陈晓斌,红蜻蜓,正常,None


#### 未知类型

In [93]:
fn_tg_wz = fn_tg1[fn_tg1['类型'] == '未知']
fn_tg_wz

,店铺,发生时间,资金类型,流水类型,店铺名称,交易金额,余额,交易摘要,类型,店铺名称（聚水潭）,店铺ID,平台,合伙人,品牌,状态,日期


#### 计算差额

In [94]:
fn_tg_1 = fn_tg1[fn_tg1['类型'] == '直通车-收入']
fn_tg_1['时间'] = file_time_str
fn_tg_1_1 = fn_tg_1.groupby(['店铺ID', '发生时间'])['交易金额'].sum().reset_index()

In [95]:
fn_cz_grouped = pd.merge(ff_5, fn_tg_1_1, left_on = ['店铺ID', '发生时间'], right_on = ['店铺ID', '发生时间'], how = 'left')
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
fn_cz_grouped = fn_cz_grouped.fillna(0).infer_objects()

In [96]:
fn_cz_grouped['差额'] = round(fn_cz_grouped['交易金额'] - fn_cz_grouped['合计'],3)
fn_cz_grouped1 = fn_cz_grouped[['店铺名称', '店铺ID', '发生时间', '平台', '合伙人', '品牌', '差额']].copy()
fn_cz_grouped1.rename(columns={'店铺名称':'店铺名称（聚水潭）', '差额': '合计'}, inplace=True)
fn_cz_grouped1['类型'] = '扣款_现金抵减推广'

In [97]:
fn_cz_grouped1[fn_cz_grouped1['合计'] != 0]

,店铺名称（聚水潭）,店铺ID,发生时间,平台,合伙人,品牌,合计,类型
131,白珍珠旗舰店（白培）,12043012,2026-02-03,拼多多,李培城,白珍珠,50.00,扣款_现金抵减推广
154,白珍珠月织光专卖店（白邹）,11931620,2026-02-02,拼多多,邹慧,白珍珠,50.00,扣款_现金抵减推广
155,白珍珠月织光专卖店（白邹）,11931620,2026-02-06,拼多多,邹慧,白珍珠,50.00,扣款_现金抵减推广
1888,红蜻蜓配件旗舰店（红鸿）,15316100,2026-02-03,拼多多,陈镇鸿,红蜻蜓,1000.00,扣款_现金抵减推广
2167,舒柔专卖店（南晓）,11931767,2026-02-26,拼多多,陈晓斌,白珍珠,382.47,扣款_现金抵减推广


### 拼多多_货款明细

In [98]:
fn_cz_grouped2 = pd.concat([
    ff_2,
    fn_cz_grouped1
]).sort_values(by = ['店铺ID', '发生时间'])

In [99]:
fn_cz_grouped2['店铺'] = np.where(fn_cz_grouped2['店铺'].isnull(), fn_cz_grouped2['店铺ID'], fn_cz_grouped2['店铺'])
fn_cz_grouped2 = fn_cz_grouped2[~((fn_cz_grouped2['类型'] == '扣款_现金抵减推广') & (fn_cz_grouped2['合计'] == 0))]
fn_cz_grouped2['来源文件'] = np.where((fn_cz_grouped2['类型'] == '扣款_现金抵减推广'), '其他', '拼多多_货款明细')

In [100]:
fn_cz_grouped2 = fn_cz_grouped2[['店铺名称（聚水潭）', '店铺ID', '平台', '合伙人', '品牌', '商户订单号', '发生时间', '收入金额', '支出金额', '账务类型', '备注', '业务描述', '类型', '合计', '来源文件']]

#### 主体

In [101]:
fn_cz_grouped2['店铺ID'] = fn_cz_grouped2['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
fn_cz_grouped3 = pd.merge(fn_cz_grouped2, zt3, on = '店铺ID', how = 'left')

In [102]:
def vectorized_get_new_subject(df):
    """
    向量化版本：根据变更日期和发生时间匹配新主体
    完全保留原逻辑，无循环，速度提升10~100倍
    """
    df = df.copy()

    # ===================== 1. 向量化解析【变更日期】 =====================
    # 分割、去空值
    date_series = df['变更日期'].str.split(',', expand=True).stack().str.strip()
    # 批量转日期，错误值设为NaT
    # 假设你的日期是 'YYYY-MM-DD' 格式
    date_parsed = pd.to_datetime(date_series, format='%Y-%m-%d', errors='coerce')
    # 恢复原索引结构，去除无效日期
    date_pivoted = date_parsed.unstack().dropna(axis=1, how='all')

    # 提取有效日期列并重命名
    date_cols = [f'date_{i}' for i in range(date_pivoted.shape[1])]
    df[date_cols] = date_pivoted

    # ===================== 2. 向量化解析【主体】 =====================
    # 分割主体，自动对齐列数
    subject_pivoted = df['主体'].str.split('-', expand=True).apply(lambda x: x.str.strip())
    subject_cols = [f'subject_{i}' for i in range(subject_pivoted.shape[1])]
    df[subject_cols] = subject_pivoted

    # 确保发生时间为datetime
    df['发生时间'] = pd.to_datetime(df['发生时间'])
    month = df['发生时间']

    # ===================== 3. 向量化匹配逻辑 =====================
    # 初始化结果列 = 原主体
    df['新主体'] = df['主体']

    # 场景1：1个日期 + 2个主体
    mask1 = (df[date_cols].notna().sum(axis=1) == 1) & (df[subject_cols].notna().sum(axis=1) == 2)
    if mask1.any() and len(date_cols)>=1 and len(subject_cols)>=2:
        date0 = df[date_cols[0]]
        cond = month < date0
        df.loc[mask1 & cond, '新主体'] = df.loc[mask1, subject_cols[0]]
        df.loc[mask1 & ~cond, '新主体'] = df.loc[mask1, subject_cols[1]]

    # 场景2：2个日期 + 3个主体
    mask2 = (df[date_cols].notna().sum(axis=1) == 2) & (df[subject_cols].notna().sum(axis=1) == 3)
    if mask2.any() and len(date_cols)>=2 and len(subject_cols)>=3:
        date0, date1 = df[date_cols[0]], df[date_cols[1]]
        # 原逻辑：idx = (month>=date0) + (month>=date1)
        idx = (month >= date0).astype(int) + (month >= date1).astype(int)
        # 按索引赋值
        df.loc[mask2 & (idx==0), '新主体'] = df.loc[mask2, subject_cols[0]]
        df.loc[mask2 & (idx==1), '新主体'] = df.loc[mask2, subject_cols[1]]
        df.loc[mask2 & (idx==2), '新主体'] = df.loc[mask2, subject_cols[2]]

    # 清理临时列
    drop_cols = date_cols + subject_cols
    df = df.drop(columns=drop_cols, errors='ignore')

    return df
# ===================== 主程序调用 =====================
fn_cz_grouped3 = vectorized_get_new_subject(fn_cz_grouped3)

#### 汇总

In [104]:
fn_cz_grouped4.rename(columns={'店铺名称（聚水潭）': '店铺名称', '合计': '金额'}, inplace=True)

In [105]:
fn_cz_grouped5 = fn_cz_grouped4.groupby(['店铺名称', '新主体', '身份', '店铺ID', '平台', '合伙人', '品牌', '类型', '来源文件'])[['收入金额', '支出金额', '金额']].sum().reset_index()

In [106]:
fn_cz_grouped5 = fn_cz_grouped5[['店铺名称', '新主体', '身份', '店铺ID', '平台', '合伙人', '品牌', '类型', '收入金额', '支出金额', '金额', '来源文件']]

In [107]:
fn_cz_grouped5

,店铺名称,新主体,身份,店铺ID,平台,合伙人,品牌,类型,收入金额,支出金额,金额,来源文件
0,云美人服饰店（白培）,深圳市汤琦贸易有限公司,小规模纳税人,12042758,拼多多,李培城,白珍珠,扣款-售后补偿消费者,0.00,-45.00,45.00,拼多多_货款明细
1,云美人服饰店（白培）,深圳市汤琦贸易有限公司,小规模纳税人,12042758,拼多多,李培城,白珍珠,扣款-小额打款,0.00,-91.64,91.64,拼多多_货款明细
2,云美人服饰店（白培）,深圳市汤琦贸易有限公司,小规模纳税人,12042758,拼多多,李培城,白珍珠,扣款-延迟发货,0.00,-78.00,78.00,拼多多_货款明细
3,云美人服饰店（白培）,深圳市汤琦贸易有限公司,小规模纳税人,12042758,拼多多,李培城,白珍珠,扣款-申诉补回,8.78,0.00,-8.78,拼多多_货款明细
4,云美人服饰店（白培）,深圳市汤琦贸易有限公司,小规模纳税人,12042758,拼多多,李培城,白珍珠,扣款-评价有礼,0.00,-198.00,198.00,拼多多_货款明细
...,...,...,...,...,...,...,...,...,...,...,...,...
1863,诺初服饰专营店(白鸿）,深圳市诺初贸易有限公司,小规模纳税人,11931837,拼多多,陈镇鸿,白珍珠,扣点-技术服务费,2.75,-38.73,35.98,拼多多_货款明细
1864,诺初服饰专营店(白鸿）,深圳市诺初贸易有限公司,小规模纳税人,11931837,拼多多,陈镇鸿,白珍珠,退款-优惠券退款,0.00,-25.95,25.95,拼多多_货款明细
1865,诺初服饰专营店(白鸿）,深圳市诺初贸易有限公司,小规模纳税人,11931837,拼多多,陈镇鸿,白珍珠,退款-订单退款,0.00,-638.57,638.57,拼多多_货款明细
1866,诺初服饰专营店(白鸿）,深圳市诺初贸易有限公司,小规模纳税人,11931837,拼多多,陈镇鸿,白珍珠,销售额- 订单收入,6335.57,0.00,6335.57,拼多多_货款明细


### 拼多多_推广账户

In [47]:
fn_tg_3 = fn_tg1[fn_tg1['类型'] != '不算'].copy()
fn_tg_3

,店铺,发生时间,资金类型,流水类型,店铺名称,交易金额,余额,交易摘要,类型,店铺名称（聚水潭）,店铺ID,平台,合伙人,品牌,状态,日期
0,11931578,2026-02-02,现金,支出,依酷内衣专营店,129.55,370.79,推广支出： 商品推广129.55元；,直通车-支出,白珍珠依酷专营店（白腾）,11931578,拼多多,许文腾,白珍珠,正常,None
1,11931578,2026-02-02,现金,收入,依酷内衣专营店,400.00,500.34,现金充值-账户实时余额低于设定值，自动充值生效,直通车-收入,白珍珠依酷专营店（白腾）,11931578,拼多多,许文腾,白珍珠,正常,None
3,11931578,2026-02-01,现金,支出,依酷内衣专营店,105.20,100.34,推广支出： 商品推广105.20元；,直通车-支出,白珍珠依酷专营店（白腾）,11931578,拼多多,许文腾,白珍珠,正常,None
5,11931589,2026-02-28,现金,支出,内秀内衣裤袜专营店,186.12,1968.65,推广支出： 商品推广186.12元；,直通车-支出,白珍珠内秀裤袜专营店（白宏）,11931589,拼多多,粟宏,白珍珠,正常,None
7,11931589,2026-02-27,现金,支出,内秀内衣裤袜专营店,170.69,2154.77,推广支出： 商品推广170.69元；,直通车-支出,白珍珠内秀裤袜专营店（白宏）,11931589,拼多多,粟宏,白珍珠,正常,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14662,18230774,2026-02-04,现金,支出,红蜻蜓柔薇专卖店,260.94,541.41,推广支出： 商品推广260.94元；,直通车-支出,红蜻蜓柔薇专卖店（红晓),18230774,拼多多,陈晓斌,红蜻蜓,正常,None
14663,18230774,2026-02-04,现金,收入,红蜻蜓柔薇专卖店,500.00,802.35,现金充值-账户实时余额低于设定值，自动充值生效,直通车-收入,红蜻蜓柔薇专卖店（红晓),18230774,拼多多,陈晓斌,红蜻蜓,正常,None
14664,18230774,2026-02-03,现金,支出,红蜻蜓柔薇专卖店,106.82,302.35,推广支出： 商品推广106.82元；,直通车-支出,红蜻蜓柔薇专卖店（红晓),18230774,拼多多,陈晓斌,红蜻蜓,正常,None
14665,18230774,2026-02-02,现金,支出,红蜻蜓柔薇专卖店,136.11,409.17,推广支出： 商品推广136.11元；,直通车-支出,红蜻蜓柔薇专卖店（红晓),18230774,拼多多,陈晓斌,红蜻蜓,正常,None


#### 主体

In [48]:
fn_tg_3['店铺ID'] = fn_tg_3['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
fn_tg_grouped3 = pd.merge(fn_tg_3, zt3, on = '店铺ID', how = 'left')

In [49]:
fn_tg_grouped3.rename(columns = {'时间': '发生时间'}, inplace = True)

In [50]:
# 主程序
fn_tg_grouped3['发生时间'] = pd.to_datetime(fn_tg_grouped3['发生时间'])
fn_tg_grouped3['新主体'] = fn_tg_grouped3.apply(get_new_subject, axis=1)

In [51]:
fn_tg_grouped4 = pd.merge(fn_tg_grouped3, zt2, left_on = '新主体', right_on = '主体', how = 'left')
fn_tg_grouped4['身份'] = np.where(fn_tg_grouped4['身份'].isna(), '小规模纳税人', fn_tg_grouped4['身份'])

In [52]:
fn_tg_grouped4['来源文件'] = '拼多多_推广账户'

In [53]:
fn_tg_grouped4 = fn_tg_grouped4[['店铺名称（聚水潭）', '新主体', '身份', '店铺ID', '平台', '合伙人', '品牌', '发生时间', '交易金额', '类型', '来源文件']].rename(columns = {'店铺名称（聚水潭）':'店铺名称', '交易金额':'金额'})

#### 汇总

In [54]:
fn_tg_grouped5 = fn_tg_grouped4.groupby(['店铺名称', '新主体', '身份', '店铺ID', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额']].sum().reset_index()

In [55]:
fn_tg_grouped5 = fn_tg_grouped5[['店铺名称', '新主体', '身份', '店铺ID', '平台', '合伙人', '品牌', '类型', '金额', '来源文件']]

### 拼多多_保证金

In [56]:
ff_bzj = pd.read_sql(f'select * from `拼多多_保证金`', conn)

In [57]:
ff_bzj_1 = ff_bzj[(ff_bzj['入账时间'] >= f'2026-{int(month)}-01')].copy()

In [58]:
# 一行调用，直接处理整列
ff_bzj_1['入账时间'] = convert_date_simple(ff_bzj_1['入账时间'])

In [59]:
ff_bzj_1.rename(columns={'入账时间': '发生时间', '收入金额（+元）':'收入金额', '支出金额（_元）':'支出金额'}, inplace=True)

In [62]:
import pandas as pd
import numpy as np

# 初始化类型列为未知
ff_bzj_1['类型'] = '未知'

# ===================== 一对一赋值，去掉 for 循环 =====================
# 条件1：提现-手动
cond1 = (ff_bzj_1['账户类型'].isin(['店铺保证金', '活动保证金'])) & (ff_bzj_1['账务类型'].str.contains('提现', na=False))
ff_bzj_1.loc[cond1, '类型'] = '提现-手动'

# 条件2：店铺保证金-转回
cond2 = (ff_bzj_1['账户类型'] == '店铺保证金') & (ff_bzj_1['账务类型'].str.contains('扣款', na=False)) & (ff_bzj_1['业务描述'].str.contains('转账-货款账户', na=False))
ff_bzj_1.loc[cond2, '类型'] = '店铺保证金-转回'

In [ ]:
import pandas as pd

# 定义需要转 float 的所有列名列表
float_cols = [
    '收入金额', '支出金额'
]

# 批量转换类型（安全写法，自动跳过不存在的列）
for col in float_cols:
    if col in ff_bzj_1.columns:
        ff_bzj_1[col] = pd.to_numeric(ff_bzj_1[col], errors='coerce')

In [ ]:
ff_bzj_1['收入金额'] = ff_bzj_1['收入金额'].fillna(0)
ff_bzj_1['支出金额'] = ff_bzj_1['支出金额'].fillna(0)

In [ ]:
import pandas as pd

# ===================== ff_1 专用 金额规则配置 =====================
# 支出类固定类型（取绝对值）
EXPENSE_TYPES = {
    '店铺保证金-转回', '提现-手动'
}

# 收入需要取负的类型
INCOME_NEG_TYPES = {
    # '扣点-技术服务费', '扣款-多多进宝', '扣款-运费补偿',
    # '扣款-小额打款', '保证金-店铺保证金'
}

# 销售额固定类型（收入取绝对值）
SALES_FIXED = {
    # '销售额-订单收入', '销售额-优惠券收入', '销售额-多单立减',
    # '销售额-平台补贴收入'
}

# ===================== 核心金额逻辑（和你给的格式完全一致） =====================
ap = ''  # 没有就留空，不影响运行

# 三大掩码（逻辑完全一样）
mask_abs = (ff_bzj_1['类型'].isin(SALES_FIXED)) & ff_bzj_1['收入金额'].ne(0)
mask_neg = (ff_bzj_1['类型'].isin(INCOME_NEG_TYPES) | (ff_bzj_1['类型'] == ap)) & ff_bzj_1['收入金额'].ne(0)
mask_type = (ff_bzj_1['类型'].isin(EXPENSE_TYPES) | (ff_bzj_1['类型'] == ap)) & ff_bzj_1['支出金额'].ne(0)

# 金额赋值
ff_bzj_1.loc[mask_neg, '收入金额'] *= -1
ff_bzj_1.loc[mask_abs, '收入金额'] = ff_bzj_1['收入金额'].abs()
ff_bzj_1.loc[mask_type, '支出金额'] = ff_bzj_1['支出金额'].abs()

In [ ]:
ff_bzj_1['金额'] = ff_bzj_1['收入金额'] + ff_bzj_1['支出金额']

In [63]:
ff_bzj_1.类型.value_counts()

类型
提现-手动    6
Name: count, dtype: int64

#### 未知类型

In [108]:
ff_bzj_wz = ff_bzj_1[(ff_bzj_1['类型'] == '未知') & (ff_bzj_1['金额'] != 0)]
ff_bzj_wz

,id,店铺,发生时间,账户类型,账务类型,收入金额,支出金额,备注,业务描述,create_time,类型,金额


#### 店铺表格

In [109]:
ff_bzj_1['店铺'] = ff_bzj_1['店铺'].astype(int)

In [110]:
ff_bzj_2 = pd.merge(ff_bzj_1, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [111]:
# 主执行逻辑（仅2行核心调用）
ff_bzj_2 = ff_bzj_2.copy()
for field in ['合伙人', '品牌']:
    ff_bzj_2 = optimize_field_by_date(ff_bzj_2, field)

#### 主体

In [112]:
ff_bzj_2['店铺ID'] = ff_bzj_2['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
ff_bzj_3 = pd.merge(ff_bzj_2, zt3, on = '店铺ID', how = 'left')

In [113]:
# 主程序
ff_bzj_3['发生时间'] = pd.to_datetime(ff_bzj_3['发生时间'])
ff_bzj_3['新主体'] = ff_bzj_3.apply(get_new_subject, axis=1)

In [114]:
ff_bzj_4 = pd.merge(ff_bzj_3, zt2, left_on = '新主体', right_on = '主体', how = 'left')
ff_bzj_4['身份'] = np.where(ff_bzj_4['身份'].isna(), '小规模纳税人', ff_bzj_4['身份'])

#### 汇总

In [115]:
ff_bzj_4['来源文件'] = "拼多多_保证金"

In [116]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
ff_bzj_5 = ff_bzj_4.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型',"来源文件"])[['金额']].sum().round(2).reset_index()

# 重命名列名
ff_bzj_5 = ff_bzj_5.rename(columns={'店铺名称（聚水潭）':'店铺名称'})

In [117]:
ff_bzj_5 = ff_bzj_5[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '金额',"来源文件"]].copy()

In [118]:
ff_bzj_5

,店铺名称,店铺ID,新主体,身份,平台,合伙人,品牌,类型,金额,来源文件
0,柔薇服饰专营店（白晓）,12242967,普宁市柔薇服饰有限公司,小规模纳税人,拼多多,陈晓斌,白珍珠,提现-手动,5000,拼多多_保证金
1,白珍珠蜜姿服装专营店（白斌）,14834119,深圳市蜜姿服饰有限公司,小规模纳税人,拼多多,陈晓斌,白珍珠,提现-手动,2000,拼多多_保证金
2,白珍珠蜜姿裤专营店（白斌),15776825,深圳市蜜姿服饰有限公司,小规模纳税人,拼多多,陈晓斌,白珍珠,提现-手动,6500,拼多多_保证金
3,红蜻蜓RED DRAGONFLY服饰配件旗舰店（红鸿）,15569706,普宁市棉坊屋服饰有限公司,小规模纳税人,拼多多,陈镇鸿,红蜻蜓,提现-手动,21000,拼多多_保证金


### 未知

In [119]:
pdd_wz = pd.concat([ff_2_wz, fn_tg_wz, ff_bzj_wz], ignore_index=True)
pdd_wz

,店铺,商户订单号,发生时间,收入金额,支出金额,账务类型,备注,业务描述,类型,合计,...,资金类型,流水类型,店铺名称,交易金额,余额,交易摘要,id,账户类型,create_time,金额


### 汇总

In [120]:
pdd_hz = pd.concat([fn_cz_grouped5, fn_tg_grouped5, ff_bzj_5], ignore_index=True)
pdd_hz = pdd_hz.fillna(0)

In [121]:
pdd_hz = pdd_hz.groupby(['店铺名称', '新主体', '身份', '店铺ID', '平台', '合伙人', '品牌', '类型', '来源文件'])[['收入金额', '支出金额','金额']].sum().reset_index()

In [122]:
# 定义需要保存的DataFrame和对应的文件路径
data_to_save = [
    (pdd_wz, fr'../../结果/未知/{int(month)}月拼多多未知类型.csv'),
    (pdd_hz, fr'../../结果/2026年{int(month)}月/拼多多/拼多多_货款明细.csv'),
]
# 分离DataFrame和文件路径列表
dataframes = [item[0] for item in data_to_save]
file_paths = [item[1] for item in data_to_save]
# 调用函数保存数据
try:
    save_dataframes(dataframes, file_paths)
except Exception as e:
    print(f"保存数据时出错: {e}")

跳过空数据: ../../结果/未知/2月拼多多未知类型.csv (无旧文件可删除)
已保存数据到: ../../结果/2026年2月/拼多多/拼多多_货款明细.csv


## 抖音

### 抖音资金

In [ ]:
dy = pd.read_sql(f'select * from `抖音资金`', conn)

In [ ]:
# 一行调用，直接处理整列
dy['动账时间'] = convert_date_simple(dy['动账时间'])

In [ ]:
dy_lx1 = dy[(dy['动账时间'].str[:7] == f'2026-{month}')]

In [ ]:
dy_lx1.rename(columns={'动账时间': '发生时间'}, inplace=True)

#### 统一修正

In [ ]:
import numpy as np
import pandas as pd

# 金额列
cols = [
    '平台服务费', '佣金', '服务商佣金', '渠道分成', '招商服务费',
    '动账金额', '站外推广费', '订单实付应结', '实际平台补贴_运费',
    '实际平台补贴', '其他平台补贴', '以旧换新抵扣', '政府补贴平台垫资',
    '实际抖音支付补贴',"实际达人补贴", '实际抖音月付营销补贴', '银行补贴',
    '订单退款', '免佣金额'
]

# 1. 转换类型
dy_lx1[cols] = dy_lx1[cols].astype(float)

# 2. 入账：全部转为正数（0 不变）
mask_rz = dy_lx1['动账方向'] == '入账'
dy_lx1.loc[mask_rz, cols] = dy_lx1.loc[mask_rz, cols].abs()

#### 区分类型 dy_lx

In [ ]:
import pandas as pd
import numpy as np

# ===================== 【条件构建器】 =====================
def get_mask(df, scene=None, bill_type=None, note=None, direction=None):
    mask = pd.Series(True, index=df.index)
    if direction is not None:
        mask &= df["动账方向"] == direction
    if scene is not None:
        mask &= df["动账场景"].isin(scene) if isinstance(scene, (list, tuple)) else df["动账场景"] == scene
    if bill_type is not None:
        mask &= df["计费类型"].isin(bill_type) if isinstance(bill_type, (list, tuple)) else df["计费类型"] == bill_type
    if note is not None:
        if callable(note):
            mask &= note(df)
        elif isinstance(note, list):
            note_mask = pd.Series(False, index=df.index)
            for kw in note:
                note_mask |= df["备注"].isna() if pd.isna(kw) else df["备注"].str.contains(str(kw), na=False)
            mask &= note_mask
        else:
            mask &= df["备注"].eq(note) | df["备注"].str.contains(note, na=False)
    return mask

# ===================== 配置 =====================
BT = ["小店自卖", "精选联盟", "巨量千川"]
SALE_COLUMNS = [
    "订单实付应结","实际平台补贴_运费","实际平台补贴","其他平台补贴","以旧换新抵扣",
    "政府补贴平台垫资","实际达人补贴","实际抖音支付补贴","实际抖音月付营销补贴","银行补贴"
]
COST_COLS = ["渠道分成","招商服务费","站外推广费","其他分成","服务商佣金"]

# ===================== 工具函数 =====================
def set_type_amount(df, mask, type_prefix, bt, amount_col, negative=False):
    """批量设置 类型 + 金额，简化重复代码"""
    for t in bt if isinstance(bt, list) else [bt]:
        m = mask & (df["计费类型"] == t)
        df.loc[m, "类型"] = f"{type_prefix}-{t}"
        val = df.loc[m, amount_col]
        df.loc[m, "金额"] = -val if negative else val

def add_new_row(new_rows, row, type_name, amount):
    """追加新行统一写法"""
    nr = row.copy()
    nr["类型"] = type_name
    nr["金额"] = float(amount)
    new_rows.append(nr)

# ===================== 全量数据初始化 =====================
dy_cl = dy_lx1.copy()
dy_cl[["类型", "金额"]] = ["未知", 0.0]
new_rows = []

# ------------------- 1. 销售额：追加新行 -------------------
mask_sale = get_mask(dy_cl, "货款结算入账", direction="入账", note="订单结算|运费单结算") & (dy_cl["类型"] == "未知")
for idx, row in dy_cl[mask_sale].iterrows():
    bt = row["计费类型"]
    for col in SALE_COLUMNS:
        val = row[col]
        if pd.notna(val) and val != 0:
            add_new_row(new_rows, row, f"销售额-{bt}-{col}", val)

# ===================== 退款类：原地打标（按你给的类型列表重构） =====================
# 1. 未结算退款合集（生成：退款-{x}-未结算退款）
# 条件：货款结算入账 + 订单结算 + 入账 + 未知
mask_refund_uns1 = get_mask(dy_cl, "货款结算入账", BT, "订单结算", "入账") & (dy_cl["类型"] == "未知")
# 条件：退款-订单退款触发-分账 + 精选联盟 + 退款失败分账 + 入账 + 未知
mask_refund_uns2 = get_mask(dy_cl, ["退款-订单退款触发-分账"], "精选联盟", "退款失败分账", "入账") & (dy_cl["类型"] == "未知")
# 条件：退款-退转付扣减商家货款 + 精选联盟 + 出账 + 未知
mask_refund_uns3 = get_mask(dy_cl, "退款-退转付扣减商家货款", "精选联盟", None, "出账") & (dy_cl["类型"] == "未知")

# 统一标记为 未结算退款
dy_cl.loc[mask_refund_uns1, "类型"] = dy_cl.loc[mask_refund_uns1, "计费类型"].apply(lambda x: f"退款-{x}-未结算退款")
dy_cl.loc[mask_refund_uns1, "金额"] = dy_cl.loc[mask_refund_uns1, "订单退款"]

dy_cl.loc[mask_refund_uns2, "类型"] = "退款-精选联盟-未结算退款"
dy_cl.loc[mask_refund_uns2, "金额"] = -dy_cl.loc[mask_refund_uns2, "动账金额"]

dy_cl.loc[mask_refund_uns3, "类型"] = "退款-精选联盟-未结算退款"
dy_cl.loc[mask_refund_uns3, "金额"] = dy_cl.loc[mask_refund_uns3, "动账金额"]

# 2. 结算后退款：修正类型生成方式
# 注意：你的类型列表里没有“结算后退款”，如果不需要可以直接删掉这部分
# 但如果你需要保留，就用下面的写法（和你的类型列表兼容）
mask_refund_settle1 = get_mask(dy_cl, ["退款-结算后退款-退用户", "退款-极速退二阶段商家资金回补"], BT, None, "出账") & (dy_cl["类型"] == "未知")
mask_refund_settle2 = get_mask(dy_cl, ["退款-订单退款触发-分账"], BT, "极速退款分账", "入账") & (dy_cl["类型"] == "未知")

# 直接赋值，不用set_type_amount，确保类型正确
# 如果你不需要“结算后退款”类型，直接删掉这两行，把金额逻辑合并到未结算退款里
dy_cl.loc[mask_refund_settle1, "类型"] = dy_cl.loc[mask_refund_settle1, "计费类型"].apply(lambda x: f"退款-{x}-未结算退款")
dy_cl.loc[mask_refund_settle1, "金额"] = -dy_cl.loc[mask_refund_settle1, "订单退款"]

dy_cl.loc[mask_refund_settle2, "类型"] = dy_cl.loc[mask_refund_settle2, "计费类型"].apply(lambda x: f"退款-{x}-未结算退款")
dy_cl.loc[mask_refund_settle2, "金额"] = -dy_cl.loc[mask_refund_settle2, "订单实付应结"]

# 3. 平台补贴扣回（生成：退款-{x}-退平台补贴）
mask_refund_sub1 = get_mask(dy_cl, "退款-订单退款触发-退补贴", BT, "平台补贴扣回", "出账") & (dy_cl["类型"] == "未知")
mask_refund_sub2 = get_mask(dy_cl, "平台返现/返券活动追缴用户退回平台", BT, None, "出账") & (dy_cl["类型"] == "未知")
mask_refund_sub3 = get_mask(dy_cl, ["退款-订单退款触发-分账", "退款-订单退款触发-退分账"], BT,
                          lambda d: d["备注"].str.contains("平台返现|签到领现金追缴", na=False), "入账") & (dy_cl["类型"] == "未知")

mask_sub_all = mask_refund_sub1 | mask_refund_sub2
# 分开赋值，确保类型正确
dy_cl.loc[mask_sub_all, "类型"] = dy_cl.loc[mask_sub_all, "计费类型"].apply(lambda x: f"退款-{x}-退平台补贴")
dy_cl.loc[mask_sub_all, "金额"] = dy_cl.loc[mask_sub_all, "动账金额"]

dy_cl.loc[mask_refund_sub3, "类型"] = dy_cl.loc[mask_refund_sub3, "计费类型"].apply(lambda x: f"退款-{x}-退平台补贴")
dy_cl.loc[mask_refund_sub3, "金额"] = -dy_cl.loc[mask_refund_sub3, "动账金额"]

# ------------------- 3. 扣点类：追加新行 -------------------
mask_cost = get_mask(dy_cl, "货款结算入账", direction="入账", note="订单结算|运费单结算")
mask_cost_order = get_mask(dy_cl, "货款结算入账", direction="入账", note="订单结算")

# 平台服务费
for idx, row in dy_cl[mask_cost].iterrows():
    bt = row["计费类型"]
    if bt in BT and pd.notna(row["平台服务费"]):
        add_new_row(new_rows, row, f"扣点-{bt}-平台服务费", row["平台服务费"])

# 佣金 + 分成
for idx, row in dy_cl[mask_cost_order].iterrows():
    bt = row["计费类型"]
    if bt not in BT:
        continue
    add_new_row(new_rows, row, f"扣点-{bt}-佣金", row.get("佣金", 0.0))
    for col in COST_COLS:
        if pd.notna(row[col]):
            add_new_row(new_rows, row, f"扣点-{bt}-{col}", row[col])

# 退款场景扣点返还
mask_refund_fee = get_mask(dy_cl, ["退款-订单退款触发-分账", "退款-订单退款触发-退分账"], note="服务费返还", direction="入账")
for idx, row in dy_cl[mask_refund_fee].iterrows():
    bt = row["计费类型"]
    scene = row["动账场景"]
    if bt not in BT:
        continue
    if scene == "退款-订单退款触发-退分账" and pd.notna(row["平台服务费"]):
        add_new_row(new_rows, row, f"扣点-{bt}-平台服务费", -row["平台服务费"])
    add_new_row(new_rows, row, f"扣点-{bt}-佣金", -row.get("佣金", 0.0))
    for col in ["招商服务费", "站外推广费"]:
        if pd.notna(row[col]):
            add_new_row(new_rows, row, f"扣点-{bt}-{col}", -row[col])

# 极速退款分账扣点（精选联盟）
mask_exp_fee = get_mask(dy_cl, ["退款-订单退款触发-分账", "退款-订单退款触发-退分账"], "精选联盟", "极速退款分账", "入账")
for idx, row in dy_cl[mask_exp_fee].iterrows():
    add_new_row(new_rows, row, "扣点-精选联盟-佣金", -row.get("佣金", 0.0))
    for col in ["招商服务费", "站外推广费"]:
        if pd.notna(row[col]):
            add_new_row(new_rows, row, f"扣点-精选联盟-{col}", -row[col])

# ------------------- 4. 其他杂项：统一处理 -------------------
def set_other(df, mask, type_name, amount_col, negative=False):
    df.loc[mask, "类型"] = type_name
    df.loc[mask, "金额"] = -df.loc[mask, amount_col] if negative else df.loc[mask, amount_col]

# 巨量千川充值
qc_in = dy_cl["动账方向"] == "入账"
qc_out = dy_cl["动账方向"] == "出账"
qc_note = lambda d: d["备注"].str.contains("未消耗充值款退回货款|划扣电商货款充值巨量千川", na=False)
mask_qc = qc_note(dy_cl) & (dy_cl["类型"] == "未知")
set_other(dy_cl, mask_qc & qc_in, "巨量千川-充值", "动账金额", negative=True)
set_other(dy_cl, mask_qc & qc_out, "巨量千川-充值", "动账金额")

# 物流费
log1 = get_mask(dy_cl, "上门取件运费", direction="出账")
log2 = get_mask(dy_cl, ["", "偏远地区物流服务"], note="拦截费|偏远地区配送费", direction="出账")
log3 = get_mask(dy_cl, "偏远地区物流服务", note="偏远地区配送费", direction="入账")
mask_log = (log1 | log2 | log3) & (dy_cl["类型"] == "未知")
set_other(dy_cl, mask_log, "扣点-物流费", "动账金额")
dy_cl.loc[log3, "金额"] *= -1

# 各类赔付/扣款/贴息/保证金
other_config = [
    (get_mask(dy_cl, "偏远地区物流服务", direction="入账") & dy_cl["备注"].str.contains("偏远地区货物类赔付", na=False), "扣款-货物赔付", True),
    (get_mask(dy_cl, "平台赔付", direction="入账"), "扣款-平台赔付", True),
    # 消费者赔付 这里统一先标记类型，金额后面单独精准处理
    (get_mask(dy_cl, "消费者赔付"), "扣款-消费者赔付", False),
    (get_mask(dy_cl, "小额打款", direction="出账"), "扣款-小额打款", False),
    (get_mask(dy_cl, "公益捐款", direction="出账"), "扣款-公益捐款", False),
    (get_mask(dy_cl, "欠票扣款-商家开票", direction="出账"), "扣点-欠票扣款", False),
    (get_mask(dy_cl, "抖音月付与商家联合贴息活动", direction="出账", note="抖音月付联合贴息费用划扣"), "扣点-月付贴息", False),
    (get_mask(dy_cl, "提现", direction="出账"), "提现", False),
    (get_mask(dy_cl, "评价有礼活动资金回退", direction="入账") & dy_cl["备注"].str.contains("评价有礼活动资金回退"), "保证金-评价有礼保证金-资金回退", True),
    (get_mask(dy_cl, "评价有礼保证金扣款", direction="出账") & dy_cl["备注"].str.contains("扣除货款充值评价有礼保证金"), "保证金-评价有礼保证金-扣款", False),
    (get_mask(dy_cl, direction="出账", note="货款充值保证金"), "保证金-货款充值保证金", False),
    (get_mask(dy_cl, "权益保险", direction="出账") & dy_cl["备注"].str.contains("保费扣除"), "扣点-权益保险", False),
]

for mask, t, neg in other_config:
    mask &= dy_cl["类型"] == "未知"
    set_other(dy_cl, mask, t, "动账金额", neg)

# ===================== 【修复：消费者赔付 精准金额处理】 =====================
# 入账 = 负数（赔付给平台/用户，商家扣钱）
mask3 = get_mask(dy_cl, "消费者赔付", direction="入账") & (dy_cl["类型"] == "扣款-消费者赔付")
# 出账 = 正数（商家赔付支出）
mask4 = get_mask(dy_cl, "消费者赔付", direction="出账") & (dy_cl["类型"] == "扣款-消费者赔付")

# 强制赋值，确保正负正确
dy_cl.loc[mask3, "金额"] = -dy_cl.loc[mask3, "动账金额"].astype(float)
dy_cl.loc[mask4, "金额"] = dy_cl.loc[mask4, "动账金额"].astype(float)

# ------------------- 合并新行 =====================
if new_rows:
    dy_cl = pd.concat([dy_cl, pd.DataFrame(new_rows)], ignore_index=True)

# ===================== 清理无用行 =====================
mask_remove = (
    ((dy_cl["动账场景"] == "货款结算入账") & dy_cl["备注"].isin(["订单结算", "运费单结算"])) |
    (dy_cl["动账场景"].isin(["退款-订单退款触发-分账", "退款-订单退款触发-退分账"]))
)
dy_cl = dy_cl[~(mask_remove & (dy_cl["类型"] == "未知"))].reset_index(drop=True)
dy_cl['金额'] = dy_cl['金额'].astype(float)

In [ ]:
dy_cl[(dy_cl['店铺'] == '14271036') & (dy_cl['类型'] == '扣款-消费者赔付')].金额.sum()

In [ ]:
dy_cl['类型'].value_counts()

In [ ]:
dy_cl[(dy_cl['类型'] == "未知")][['店铺', '动账方向','动账场景','计费类型', '备注', '类型', '金额']]

#### 未知类型

In [ ]:
dy_cl_wz = dy_cl[(dy_cl['类型'] == "未知") & (dy_cl['金额'] != 0)]

# 精准规则：特殊不截 / 两类截7 / 其余截8
dy_cl_wz[['动账方向','动账场景','计费类型','备注']].assign(
    备注=lambda x: x['备注'].astype(str).apply(
        lambda s:
            s  # 包含这3个 → 完整保留
            if any(kw in s for kw in ['服务费返还', '运费单结算', '极速退款分账'])
            else s[:7]  # 包含这2个 → 截前7
            if any(kw in s for kw in ['货款充值保证金', '偏远地区配送费'])
            else s[:12]  # 其他所有 → 截前8
    )
).drop_duplicates()

In [ ]:
dy_cl_wz_1 = dy_cl_wz[['店铺', '发生时间', '动账方向','动账场景','计费类型', '备注', '类型', '金额']].copy()
dy_cl_wz_1

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
dy_cl = dy_cl.fillna(0).infer_objects()

#### 匹配店铺表格

In [ ]:
dy_cl['店铺'] = dy_cl['店铺'].astype(int)
dp_bg['店铺ID'] = dp_bg['店铺ID'].astype(int)

In [ ]:
dy_cl_1 = pd.merge(dy_cl, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
dy_cl_1 = dy_cl_1.copy()
for field in ['合伙人', '品牌']:
    dy_cl_1 = optimize_field_by_date(dy_cl_1, field)

In [ ]:
dy_cl_1 = dy_cl_1[['店铺名称（聚水潭）', '店铺ID', '平台', '合伙人', '品牌', '发生时间', '订单号', '动账方向', '动账金额', '动账账户', '动账场景', '计费类型', '订单类型', '订单实付应结',
       '运费实付', '实际平台补贴_运费', '实际平台补贴', '其他平台补贴', '以旧换新抵扣', '政府补贴平台垫资', '实际达人补贴',
       '实际抖音支付补贴', '实际抖音月付营销补贴', '银行补贴', '订单退款', '平台服务费', '佣金', '服务商佣金',
       '渠道分成', '招商服务费', '站外推广费', '其他分成', '是否免佣', '免佣金额', '备注', '类型', '金额',
       '状态']].copy()

#### 主体

In [ ]:
dy_cl_1['店铺ID'] = dy_cl_1['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
dy_cl_2 = pd.merge(dy_cl_1, zt3, on = '店铺ID', how = 'left')

In [ ]:
dy_cl_2

In [ ]:
# 主程序
dy_cl_2['发生时间'] = pd.to_datetime(dy_cl_2['发生时间'])
dy_cl_2['新主体'] = dy_cl_2.apply(get_new_subject, axis=1)

In [ ]:
dy_cl_3 = pd.merge(dy_cl_2, zt2, left_on = '新主体', right_on = '主体', how = 'left')
dy_cl_3['身份'] = np.where(dy_cl_3['身份'].isna(), '小规模纳税人', dy_cl_3['身份'])

In [ ]:
dy_cl_4 = dy_cl_3[~(dy_cl_3.发生时间 == '')]

In [ ]:
dy_cl_4['金额'] = dy_cl_4['金额'].astype(float)

In [ ]:
dy_cl_4['金额'] = round(dy_cl_4['金额'],2)

In [ ]:
dy_cl_4['来源文件'] = '抖音_资金流水明细'

#### 汇总

In [ ]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
dy_cl_5 = dy_cl_4.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额']].sum().round(2).reset_index()

# 重命名列名
dy_cl_5 = dy_cl_5.rename(columns={'店铺名称（聚水潭）':'店铺名称', '调整金额':'金额'})

In [ ]:
dy_cl_5 = dy_cl_5[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '金额', '来源文件']].copy()

In [ ]:
dy_cl_5

### 抖音_日结报表

In [ ]:
dy_tg = pd.read_sql(f'select * from 抖音推广账户', conn)

In [ ]:
dy_tg = dy_tg[dy_tg['日期'].str[:7] == f'2026-{month}']

In [ ]:
dy_tg['日期'] = convert_date_simple(dy_tg['日期'])

In [ ]:
dy_tg.rename(columns = {'日期':'发生时间'}, inplace = True)

In [ ]:
import pandas as pd

# 定义需要转 float 的所有列名列表
float_cols = [
    '总存入元', '余额总消耗元', '非赠款消耗元', '赠款消耗元'
]

# 批量转换类型（安全写法，自动跳过不存在的列）
for col in float_cols:
    if col in dy_tg.columns:
        dy_tg[col] = pd.to_numeric(dy_tg[col], errors='coerce')

#### 区分类型

In [ ]:
import pandas as pd
import numpy as np

# 1. 初始化（完全对齐你给的新版格式）
dy_tg_cl = dy_tg.copy()
dy_tg_cl["类型"] = "未知"
dy_tg_cl["金额"] = 1
new_rows = []

# 统一简写
df = dy_tg_cl

# 2. 遍历每一行，按规则生成所有类型（逐条规则对应拆分）
for idx, row in df.iterrows():
    r = row.to_dict()

    # ------------------- 规则1：巨量千川-总存入 -------------------
    if pd.notnull(r["总存入元"]):
        new_rows.append({**r, "类型": "巨量千川-总存入", "金额": r["总存入元"]})

    # ------------------- 规则2：巨量千川-余额总消耗 -------------------
    if pd.notnull(r["余额总消耗元"]):
        new_rows.append({**r, "类型": "巨量千川-余额总消耗", "金额": abs(r["余额总消耗元"])})

    # ------------------- 规则3：巨量千川-非赠款消耗 -------------------
    if pd.notnull(r["非赠款消耗元"]):
        new_rows.append({**r, "类型": "巨量千川-非赠款消耗", "金额": r["非赠款消耗元"]})

    # ------------------- 规则4：巨量千川-赠款消耗 -------------------
    if pd.notnull(r["赠款消耗元"]):
        new_rows.append({**r, "类型": "巨量千川-赠款消耗", "金额": r["赠款消耗元"]})

# ===================== 合并结果 =====================
dy_tg_sc = pd.DataFrame(new_rows)

#### 未知类型

In [ ]:
dy_tg_sc_wz = dy_tg_sc[(dy_tg_sc['类型'] == '未知') & (dy_tg_sc['金额'] != 0)]
dy_tg_sc_wz

#### 店铺表格

In [ ]:
dy_tg_sc['店铺'] = dy_tg_sc['店铺'].astype(int)

In [ ]:
dy_tg_1 = pd.merge(dy_tg_sc, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
dy_tg_1 = dy_tg_1.copy()
for field in ['合伙人', '品牌']:
    dy_tg_1 = optimize_field_by_date(dy_tg_1, field)

#### 主体

In [ ]:
dy_tg_1['店铺ID'] = dy_tg_1['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
dy_tg_2 = pd.merge(dy_tg_1, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
dy_tg_2['发生时间'] = pd.to_datetime(dy_tg_2['发生时间'])
dy_tg_2['新主体'] = dy_tg_2.apply(get_new_subject, axis=1)

In [ ]:
dy_tg_3 = pd.merge(dy_tg_2, zt2, left_on = '新主体', right_on = '主体', how = 'left')
dy_tg_3['身份'] = np.where(dy_tg_3['身份'].isna(), '小规模纳税人', dy_tg_3['身份'])

#### 汇总

In [ ]:
# ===================== 只保留有效数据，清理0金额 =====================
dy_tg_4 = dy_tg_3[~((dy_tg_3['类型']=="未知") & (dy_tg_3['金额']==0))].copy().reset_index(drop=True)

In [ ]:
dy_tg_4['来源文件'] = '抖音_日结报表'

In [ ]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
dy_tg_5 = dy_tg_4.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额']].sum().round(2).reset_index()

# 重命名列名
dy_tg_5 = dy_tg_5.rename(columns={'店铺名称（聚水潭）':'店铺名称', '调整金额':'金额'})

In [ ]:
dy_tg_5 = dy_tg_5[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '金额', '来源文件']].copy()

In [ ]:
dy_tg_5

### 抖音_收支明细

In [ ]:
dy_sz = pd.read_sql(f'select * from `抖音收支明细`', conn)

In [ ]:
# 一行调用，直接处理整列
dy_sz['交易时间'] = convert_date_simple(dy_sz['交易时间'])

In [ ]:
dy_sz.rename(columns = {'交易时间':'发生时间'}, inplace = True)

In [ ]:
import pandas as pd

# 定义需要转 float 的所有列名列表
float_cols = [
    '交易金额'
]

# 批量转换类型（安全写法，自动跳过不存在的列）
for col in float_cols:
    if col in dy_sz.columns:
        dy_sz[col] = pd.to_numeric(dy_sz[col], errors='coerce')

In [ ]:
import pandas as pd
import numpy as np

# 初始化类型列为未知
dy_sz['类型'] = '未知'
dy_sz['金额'] = dy_sz['交易金额']

commission_conditions = [
    (dy_sz['交易类型'].str.contains('充值', na=False)) & (dy_sz['操作类型'].str.contains('账户余额充值', na=False)),
]
for cond in commission_conditions:
    dy_sz.loc[cond, '类型'] = '巨量千川-账户余额充值'
    dy_sz.loc[cond, '金额'] = abs(dy_sz['交易金额'])

service_expense_conditions = [
    (dy_sz['交易类型'].str.contains('充值', na=False)) & (dy_sz['操作类型'].str.contains(r'不可退返佣充值', na=False)),
]
for cond in service_expense_conditions:
    dy_sz.loc[cond, '类型'] = '巨量千川-不可退返佣充值'
    dy_sz.loc[cond, '金额'] = abs(dy_sz['交易金额'])

not_count_conditions = [
    (dy_sz['交易类型'].str.contains('扣款', na=False)) & (dy_sz['操作类型'].str.contains(r'不可退返佣到期清空', na=False)),
]
for cond in not_count_conditions:
    dy_sz.loc[cond, '类型'] = '巨量千川-不可退返佣到期清空'
    dy_sz.loc[cond, '金额'] = abs(dy_sz['交易金额'])

not_count_conditions2 = [
    (dy_sz['交易类型'].str.contains('充值', na=False)) & (dy_sz['操作类型'].str.contains(r'平台发放赠款', na=False)),
]
for cond in not_count_conditions2:
    dy_sz.loc[cond, '类型'] = '巨量千川-平台赠款'
    dy_sz.loc[cond, '金额'] = abs(dy_sz['交易金额'])

not_count_conditions3 = [
    (dy_sz['交易类型'].str.contains('退款', na=False)) & (dy_sz['操作类型'].str.contains(r'账户余额退款', na=False)) & (dy_sz['资金详情信息'].str.contains(r'现金/类目返佣', na=False)),
]
for cond in not_count_conditions3:
    dy_sz.loc[cond, '类型'] = '巨量千川-账户余额退款'
    dy_sz.loc[cond, '金额'] = abs(dy_sz['交易金额'])

In [ ]:
dy_sz.类型.value_counts()

#### 未知类型

In [ ]:
dy_sz_wz = dy_sz[(dy_sz['类型'] == '未知') & (dy_sz['金额'] != 0)]
dy_sz_wz

#### 店铺表格

In [ ]:
dy_sz['店铺'] = dy_sz['店铺'].astype(int)

In [ ]:
dy_sz_1 = pd.merge(dy_sz, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
dy_sz_1 = dy_sz_1.copy()
for field in ['合伙人', '品牌']:
    dy_sz_1 = optimize_field_by_date(dy_sz_1, field)

#### 主体

In [ ]:
dy_sz_1['店铺ID'] = dy_sz_1['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
dy_sz_2 = pd.merge(dy_sz_1, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
dy_sz_2['发生时间'] = pd.to_datetime(dy_sz_2['发生时间'])
dy_sz_2['新主体'] = dy_sz_2.apply(get_new_subject, axis=1)

In [ ]:
dy_sz_3 = pd.merge(dy_sz_2, zt2, left_on = '新主体', right_on = '主体', how = 'left')
dy_sz_3['身份'] = np.where(dy_sz_3['身份'].isna(), '小规模纳税人', dy_sz_3['身份'])

#### 汇总

In [ ]:
dy_sz_3['来源文件'] = "抖音_收支明细"

In [ ]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
dy_sz_4 = dy_sz_3.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型',"来源文件"])[['金额']].sum().round(2).reset_index()

# 重命名列名
dy_sz_4 = dy_sz_4.rename(columns={'店铺名称（聚水潭）':'店铺名称', '调整金额':'金额'})

In [ ]:
dy_sz_4 = dy_sz_4[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '金额',"来源文件"]].copy()

In [ ]:
dy_sz_4

### 抖音管家账户

In [ ]:
dy_gj = pd.read_sql(f'select * from 抖音管家账户', conn)

In [ ]:
dy_gj['店铺'] = dy_gj['店铺'].str.split('_').str[0]

In [ ]:
# 一行调用，直接处理整列
dy_gj['动账时间'] = convert_date_simple(dy_gj['动账时间'])

In [ ]:
dy_gj.rename(columns = {'动账时间':'发生时间'}, inplace = True)

In [ ]:
dy_gj['动账金额_元_'] = dy_gj['动账金额_元_'].astype(float)

In [ ]:
dy_gj_1 = dy_gj.copy()

In [ ]:
# 初始化
dy_gj_1['类型'] = '未知'
## 销售额 退款
dy_gj_1.loc[(dy_gj_1['账户方向'] == '支出') & (dy_gj_1['动账场景'] == '小额打款'), '类型'] = '扣款-小额打款'

In [ ]:
## 支出金额
dy_gj_1.loc[
    dy_gj_1['类型'].isin([
    '扣款-小额打款'
    ]), '动账金额_元_'] = abs(dy_gj_1['动账金额_元_'])

In [ ]:
dy_gj_1.rename(columns = {'动账金额_元_':'金额'}, inplace = True)

#### 未知类型

In [ ]:
dy_gj_wz = dy_gj_1[(dy_gj_1['类型'] == '未知')]
dy_gj_wz

#### 店铺表格

In [ ]:
dy_gj_1['店铺'] = dy_gj_1['店铺'].astype(float)

In [ ]:
dy_gj_2 = pd.merge(dy_gj_1, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
dy_gj_2 = dy_gj_2.copy()
for field in ['合伙人', '品牌']:
    dy_gj_2 = optimize_field_by_date(dy_gj_2, field)

In [ ]:
dy_gj_2

#### 主体

In [ ]:
dy_gj_2['店铺ID'] = dy_gj_2['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
dy_gj_3 = pd.merge(dy_gj_2, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
dy_gj_3['发生时间'] = pd.to_datetime(dy_gj_3['发生时间'])
dy_gj_3['新主体'] = dy_gj_3.apply(get_new_subject, axis=1)

In [ ]:
dy_gj_4 = pd.merge(dy_gj_3, zt2, left_on = '新主体', right_on = '主体', how = 'left')
dy_gj_4['身份'] = np.where(dy_gj_4['身份'].isna(), '小规模纳税人', dy_gj_4['身份'])

#### 汇总

In [ ]:
dy_gj_4['来源文件'] = "抖音_管家账户"

In [ ]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
dy_gj_5 = dy_gj_4.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型',"来源文件"])[['金额']].sum().round(2).reset_index()

# 重命名列名
dy_gj_5 = dy_gj_5.rename(columns={'店铺名称（聚水潭）':'店铺名称', '调整金额':'金额'})

In [ ]:
dy_gj_5 = dy_gj_5[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '金额',"来源文件"]].copy()

In [ ]:
dy_gj_5

### 处理后

In [ ]:
dy_cl_wz_1['来源文件'] = '抖音_资金流水明细'
dy_tg_sc_wz['来源文件'] = '抖音_日结报表'
dy_sz_wz['来源文件'] = '抖音_收支明细'
dy_gj_wz['来源文件'] = '抖音_管家账户'

In [ ]:
dy_wz = pd.concat([dy_cl_wz_1, dy_tg_sc_wz, dy_sz_wz, dy_gj_wz], ignore_index=True)
dy_wz

In [ ]:
dy_hz = pd.concat([dy_cl_5, dy_tg_5, dy_sz_4, dy_gj_5], ignore_index=True)
dy_hz

In [ ]:
# 定义需要保存的DataFrame和对应的文件路径
data_to_save = [
    (dy_wz, fr'../../结果/未知/抖音未知类型.csv'),
    (dy_hz, fr'../../结果/2026年{int(month)}月/抖音/抖音2.0(待处理).csv'),
]
# 分离DataFrame和文件路径列表
dataframes = [item[0] for item in data_to_save]
file_paths = [item[1] for item in data_to_save]
# 调用函数保存数据
try:
    save_dataframes(dataframes, file_paths)
except Exception as e:
    print(f"保存数据时出错: {e}")

## 淘系

### 淘系_支付宝

In [ ]:
tx = pd.read_sql(f'select * from [支付宝账单]', conn2)

#### 店铺表格

In [ ]:
tx['店铺'] = tx['店铺'].str.split('_').str[0]

In [ ]:
# 一行调用，直接处理整列
tx['入账时间'] = convert_date_simple(tx['入账时间'])

In [ ]:
tx.rename(columns = {'收入（+元）': '收入金额', '支出（_元）': '支出金额', '入账时间':'发生时间'}, inplace = True)

In [ ]:
tx['收入金额'] = tx['收入金额'].replace(' ', 0).replace('', 0).replace('\n', 0).replace('\t', 0).fillna(0).astype(float)
tx['支出金额'] = tx['支出金额'].replace(' ', 0).replace('', 0).replace('\n', 0).replace('\t', 0).fillna(0).astype(float)

In [ ]:
tx['店铺'] = tx['店铺'].astype(int)
dp_bg['店铺ID'] = dp_bg['店铺ID'].astype(int)

In [ ]:
tx_1 = pd.merge(tx, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
tx_1 = tx_1.copy()
for field in ['合伙人', '品牌']:
    tx_1 = optimize_field_by_date(tx_1, field)

In [ ]:
tx_1 = tx_1[(tx_1['平台'].isin(['天猫', '淘宝'])) & (tx_1['发生时间'].str[:7] == f'2026-{month}')]

In [ ]:
tx_1['来源文件'] = '淘系_支付宝'

In [ ]:
tx_1

#### 区分类型

In [ ]:
ft_lx = tx_1.copy()

In [ ]:
import pandas as pd
import numpy as np

# 初始化
ft_lx['类型'] = '未知'

# 别名（缩短长列名，不改动数据）
tx = ft_lx['账务类型']
desc = ft_lx['业务描述']
note = ft_lx['备注']
shop = ft_lx['店铺']
goods = ft_lx['商品名称']
order_id = ft_lx['商户订单号'].astype(str)
other_name = ft_lx['对方名称']

# 统一正则/包含预计算（向量化，只算一次）
note_has = lambda s: note.str.contains(s, na=False)
desc_has = lambda s: desc.str.contains(s, na=False)
tx_in = lambda lst: tx.isin(lst)

# ===================== 规则（一行一条，纯向量化掩码赋值） =====================
# 销售额-退款
ft_lx.loc[(tx == '在线支付') & (desc == '0010001|交易收款-交易收款'), '类型'] = '销售额-交易收款'
ft_lx.loc[(tx == '退款（交易退款）') & (desc == '0020001|交易退款-余额退款'), '类型'] = '退款-交易退款'

# 扣点
ft_lx.loc[(tx == '转账') & (desc == '0030003|软件服务费-类目软件服务费（原天猫佣金）'), '类型'] = '扣点-软件服务费-类目软件服务费'
ft_lx.loc[(tx == '转账') & (desc_has('0030018|软件服务费-天猫返点积分')), '类型'] = '扣点-软件服务费-天猫返点积分'
ft_lx.loc[(tx == '转账') & (desc == '0170125T|服务费-消费者体验提升计划服务费'), '类型'] = '扣点-服务费'
ft_lx.loc[(tx_in(['转账','分账'])) & (note_has('品牌新享-天猫营销托管软件服务费')), '类型'] = '扣点-营销托管服务费'
ft_lx.loc[(tx == '分账') & (desc == '0030130|软件服务费-基础软件服务费'), '类型'] = '扣点-基础软件服务费'
ft_lx.loc[(tx == '分账') & (note_has('品牌新享-首单拉新计划')), '类型'] = '扣点-拉新计划'
ft_lx.loc[(tx == '分账') & (note_has('先用后付技术服务费')), '类型'] = '扣点-先用后付服务费'
ft_lx.loc[(tx == '分账') & (note_has('品牌新享新品孵化软件服务费')), '类型'] = '扣点-新品孵化服务费'
ft_lx.loc[(tx == '分账') & (desc_has('软件服务费-热浪引擎第三方服务商服务费')), '类型'] = '扣点-软件服务费-热浪引擎'
ft_lx.loc[(tx == '分账') & (note_has('光合平台软件服务费')), '类型'] = '扣点-软件服务费-光合平台'
ft_lx.loc[(tx == '分账') & (desc_has('软件服务费-淘金币软件服务费')), '类型'] = '扣点-软件服务费-淘金币'

# 保证金
ft_lx.loc[(tx == '保证金') & (desc_has(r'保证金-淘宝-出账缴存$|保证金-淘宝-缴存（退款锁定）$')) & (note_has('淘宝消费者保证金-充值|淘宝消费者保证金-交易售后')), '类型'] = '资金保证金-淘宝保证金-充值'
ft_lx.loc[(tx == '保证金') & (desc_has(r'008002800010\|保证金-淘宝-扣除转移')), '类型'] = '其他收入-淘宝保证金-扣除转移'
ft_lx.loc[(tx == '保证金') & (desc_has('保证金-天猫-出账缴存')), '类型'] = '资金保证金-天猫保证金-充值'
ft_lx.loc[(tx == '保证金') & (desc_has('保证金-天猫-扣除转移')), '类型'] = '其他收入-天猫保证金-扣除转移'
ft_lx.loc[(tx == '保证金') & (desc.isin(['008002800015|保证金-淘宝-额度补齐缴存','008002800006|保证金-淘宝-缴存','008002800007|保证金-淘宝-解冻'])), '类型'] = '资金保证金-淘宝保证金-充值'

# 扣款
ft_lx.loc[(tx == '转账') & (desc == '记账本转账') & (note_has('支付宝转账小额打款')), '类型'] = '扣款-小额打款'
ft_lx.loc[(tx_in(['分账','扣款'])) & (desc.isin(['0530294T|技术&服务费-限时红包代商家垫付扣回','限时红包代商家垫付扣回'])), '类型'] = '扣款-商家垫付扣回'
ft_lx.loc[(tx == '分账') & (desc_has('公益性捐赠支出-公益宝贝')), '类型'] = '扣款-公益捐赠'
ft_lx.loc[(tx == '分账') & (desc_has('营销支出-淘宝客佣金')), '类型'] = '扣款-佣金'
ft_lx.loc[(tx == '转账') & (note_has('淘宝联盟佣金代扣')), '类型'] = '扣款-佣金'
ft_lx.loc[(tx_in(['其他','其它'])) & (desc == '0020002|交易退款-保证金退款'), '类型'] = '其他支出-保证金退款'
ft_lx.loc[(tx_in(['其他','其它'])) & (note_has('商家权益红包-预算追加-卖家延迟发货赔付红包')), '类型'] = '其他支出-延迟发货'
ft_lx.loc[(tx_in(['其他','其它'])) & (note_has('商家权益红包-预算追加-淘宝虚假发货赔付红包')), '类型'] = '其他支出-虚假发货'
ft_lx.loc[(tx_in(['其他','其它'])) & (note_has('商家权益红包-预算追加-淘宝物流轨迹异常红包')), '类型'] = '其他支出-物流异常'
ft_lx.loc[(tx_in(['其他','其它'])) & (note_has('商家权益红包-预算追加-淘宝缺货赔付红包-赔付红包')), '类型'] = '其他支出-缺货'
ft_lx.loc[(tx == '转账') & (desc.str.contains('0060011|营销支出-淘宝客佣金', na=False)), '类型'] = '扣款-佣金'
ft_lx.loc[(tx_in(['其他','其它'])) & (note_has('淘宝联盟推广佣金返还')), '类型'] = '扣款-佣金'
ft_lx.loc[(tx == '转账') & (note_has('淘宝联盟推广佣金返还')), '类型'] = '扣款-佣金'

# 其他杂费
ft_lx.loc[(tx == '转账') & (note_has('扣款用途：万相台无界版自动充值')), '类型'] = '万相台-充值'
ft_lx.loc[(tx == '提现'), '类型'] = '提现'
ft_lx.loc[(tx == '分账') & (desc == '0030162T|软件服务费-淘金币软件服务费'), '类型'] = '扣点-软件服务费-淘金币'
ft_lx.loc[(tx == '分账') & (note_has('品牌新享天猫超级老客加速软件服务费')), '类型'] = '扣点-软件服务费-老客加速'
ft_lx.loc[(tx == '转账') & (note_has('商家集运中转操作费')), '类型'] = '扣点-物流费-集运中转'
ft_lx.loc[(tx == '转账') & (note_has('商家集运物流服务费')), '类型'] = '扣点-物流费-集运物流'
ft_lx.loc[(tx_in(['其他','其它'])) & (note_has('保险承保-天猫海外退货险保费收取')), '类型'] = '扣点-退货保费'
ft_lx.loc[(tx == '分账') & (desc.str.contains('0010003|交易收款-店铺主体变更', na=False)), '类型'] = '销售额-交易收款'
ft_lx.loc[(tx == '转账') & (note_has('往来款')), '类型'] = '应付账款-' +  other_name
ft_lx.loc[(tx == '转账') & (desc.isin([np.nan,' '])) & (note.isin([np.nan,' '])), '类型'] = '扣款-其他'
ft_lx.loc[(tx == '在线支付') & (goods.str.contains('店铺过户服务费',na=False)), '类型'] = '扣点-过户费'
ft_lx.loc[(tx == '在线支付') & (goods.str.contains('万相台无界版扫码充值',na=False)), '类型'] = '万相台-充值'
ft_lx.loc[(tx == '分账') & (note_has('淘特营销推广服务费')), '类型'] = '扣点-软件服务费-淘特营销'
ft_lx.loc[(tx == '分账') & (desc_has('0530288T|技术&服务费-大服饰跨境服务增值费')), '类型'] = '扣点-软件服务费-跨境增值费'
ft_lx.loc[(tx == '分账') & (note_has('百亿补贴软件服务费T62')), '类型'] = '扣点-软件服务费-百亿补贴'
ft_lx.loc[(tx == '在线支付') & (order_id.str[:5] == 'T200P'), '类型'] = '销售额-交易收款'
ft_lx.loc[(tx == '分账') & (note_has('天猫APP专享折扣服务费')), '类型'] = '扣点-软件服务费-折扣服务费'
ft_lx.loc[(tx_in(['其他','其它'])) & (note_has('余利宝-基金申购')), '类型'] = '余利宝-充值'
ft_lx.loc[(tx == '分账') & (desc_has('服务费-大服饰全球包邮跨境服务基础费_信息技术服务')), '类型'] = '扣点-软件服务费-跨境基础费'
ft_lx.loc[(tx == '分账') & (note_has('淘宝新客礼金技术服务费')), '类型'] = '扣点-软件服务费-新客服务费'
ft_lx.loc[(tx == '在线支付') & (goods.str.contains('淘宝联盟扫码充值还款',na=False)), '类型'] = '阿里妈妈-充值'

# 个人店铺（16865238）
ft_lx.loc[(tx == '理财申购') & (shop == '16865238'), '类型'] = '扣款-理财申购'
ft_lx.loc[(tx == '在线支付') & (shop == '16865238'), '类型'] = '扣款-福利费'
ft_lx.loc[(tx == '在线支付') & (desc == '0010001|交易收款-交易收款') & (shop == '16865238'), '类型'] = '销售额-交易收款'
ft_lx.loc[(tx == '退款（交易退款）') & (shop == '16865238'), '类型'] = '扣款-福利费'
ft_lx.loc[(tx == '转账') & (shop == '16865238'), '类型'] = '其他收入'
ft_lx.loc[(tx_in(['其他','其它'])) & (note_has('保险承保')) & (shop == '16865238'), '类型'] = '扣款-保险费'
ft_lx.loc[(tx_in(['其他','其它'])) & (note_has('每日自动提现')) & (shop == '16865238'), '类型'] = '其他收入'
ft_lx.loc[(tx == '在线支付（购汇支付）') & (shop == '16865238'), '类型'] = '扣款-购汇'

In [ ]:
ft_lx_1 = ft_lx.copy()

In [ ]:
import pandas as pd

# 固定类型
EXPENSE_TYPES = {
    '扣点-软件服务费-类目软件服务费','扣点-软件服务费-天猫返点积分','扣点-服务费',
    '扣款-小额打款','扣点-营销托管服务费','万相台-充值',
    '扣点-基础软件服务费','扣款-商家垫付扣回','扣点-拉新计划',
    '扣点-先用后付服务费','扣点-新品孵化服务费','扣款-公益捐赠',
    '扣款-佣金','扣点-软件服务费-热浪引擎','扣点-软件服务费-光合平台',
    '资金保证金-店铺保证金-充值','退款-交易退款','提现','扣点-延迟发货',
    '扣点-虚假发货','扣点-物流异常','扣点-软件服务费-淘金币','扣款-理财申购',
    '扣款-福利费','扣款-保险费','扣款-购汇','资金保证金-淘宝保证金-充值',
    '扣款-其他','扣点-过户费','扣点-软件服务费-老客加速','扣点-物流费-集运中转',
    '扣点-物流费-集运物流','资金保证金-天猫保证金-充值','扣点-退货保费',
    '扣点-软件服务费-淘特营销','扣款-缺货赔付','扣点-软件服务费-跨境增值费',
    '扣点-软件服务费-百亿补贴','扣点-软件服务费-折扣服务费','余利宝-充值',
    '扣点-软件服务费-跨境基础费','扣点-软件服务费-新客服务费','阿里妈妈-充值'
}

INCOME_NEG_TYPES = {
    '扣点-软件服务费-类目软件服务费','扣点-软件服务费-天猫返点积分','扣点-营销托管服务费',
    '扣款-商家垫付扣回','扣点-新品孵化服务费','扣点-软件服务费-淘金币',
    '扣款-福利费','资金保证金-淘宝保证金-充值','扣点-物流费-集运物流',
    '资金保证金-天猫保证金-扣除转移','扣款-佣金','资金保证金-淘宝保证金-扣除转移', '扣点-拉新计划', '扣点-软件服务费-热浪引擎'
}

SALES_FIXED = {'销售额-交易收款', '其他收入'}

# 核心逻辑（极简）
ap = '应付账款-' + ft_lx_1['对方名称'].astype(str)
mask_abs = (ft_lx_1['类型'].isin(SALES_FIXED)) & ft_lx_1['收入金额'].ne(0)
mask_neg = (ft_lx_1['类型'].isin(INCOME_NEG_TYPES) | (ft_lx_1['类型'] == ap)) & ft_lx_1['收入金额'].ne(0)
mask_type = (ft_lx_1['类型'].isin(EXPENSE_TYPES) | (ft_lx_1['类型'] == ap)) & ft_lx_1['支出金额'].ne(0)

# 赋值
ft_lx_1.loc[mask_neg, '收入金额'] *= -1
ft_lx_1.loc[mask_abs, '收入金额'] = ft_lx_1['收入金额'].abs()
ft_lx_1.loc[mask_type, '支出金额'] = ft_lx_1['支出金额'].abs()

In [ ]:
ft_lx_1['金额'] = round(ft_lx_1['收入金额'] + ft_lx_1['支出金额'],3)

In [ ]:
## 未知
ft_lx_1.loc[
    ft_lx_1['类型'].isin([
        '未知'
    ]), '金额'] = abs(ft_lx_1['金额'])

#### 未知类型

In [ ]:
ft_lx_2 = ft_lx_1[['店铺', '银行账号', '商户订单号', '支付宝流水号', '发生时间', '账务类型',  '备注', '业务描述', '类型', '金额']].copy()

In [ ]:
 # 筛选类型为未知
ft_lx_wz = ft_lx_2[(ft_lx_2['类型'] == '未知') & (ft_lx_2['发生时间'] != '')].copy()

# 定义处理函数（支持 if + elif + else 无限扩展）
def process_remark(x):
    s = str(x)
    if '店铺过户' in s:
        return s[:4]
    elif '保证金退款' in s:
        return s[:5]   # 条件2：自定义
    elif '公益宝贝捐赠' in s:
        return s[:6]   # 条件2：自定义
    elif ('淘特营销推广服务费' in s
          or '光合平台软件服务费' in s
          or '淘宝内容推广服务费' in s):
        return s[:9]   # 条件2：自定义
    elif '淘金币软件服务费' in s or '淘宝客佣金代扣款' in s:
        return s[:8]        # 条件1：取前8字
    elif '淘宝联盟推广佣金返还' in s:
        return s[:10]   # 条件2：自定义
    elif ('淘宝新客礼金技术服务费' in s
          or '淘宝天猫跨境服务增值费' in s
          or '淘宝天猫跨境服务基础费' in s):
        return s[:11]        # 条件3：取前5字
    elif '天猫APP专享折扣服务费' in s:
        return s[:12]
    elif '保险承保-天猫海外退货险保费收取' in s:
        return s[:16]
    elif ('百亿补贴软件服务费T62（全渠道）' in s
          or '品牌新享天猫超级老客加速软件服务费' in s):
        return s[:17]
    elif '代扣款' in s:
        return s[:18]   # 条件2：自定义
    elif '商家权益红包-预算追加-淘宝缺货赔付红包-赔付红包' in s:
        return s[:25]   # 条件2：自定义
    elif ('商家权益红包-预算追加-卖家延迟发货赔付红包-赔付红包' in s
          or '商家权益红包-预算追加-淘宝虚假发货赔付红包-赔付红包' in s
          or '商家权益红包-预算追加-淘宝物流轨迹异常红包-赔付红包' in s):
        return s[:27]   # 条件2：自定义
    else:
        return x            # 其他：保留原样
ft_lx_wz_1 = ft_lx_wz[['店铺', '银行账号', '商户订单号', '支付宝流水号', '账务类型', '备注', '业务描述', '金额']].assign(
    备注=lambda df: df['备注'].apply(process_remark)
).drop_duplicates()

In [ ]:
ft_lx_wz_1['来源文件'] = '淘系_支付宝'

In [ ]:
ft_lx_wz_1

In [ ]:
ft_lx_wz_1.业务描述.value_counts()

In [ ]:
ft_lx_wz_1.备注.value_counts()

In [ ]:
# ft_lx.loc[(ft_lx['支付宝流水号'].str.contains('1570759955689984210|1570759514245884210|1570779063289004300|1570771739018784210', na=False)), '类型'] = '不算'

#### 主体

In [ ]:
ft_lx_1['店铺ID'] = ft_lx_1['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
ft_lx_2_2 = pd.merge(ft_lx_1, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
ft_lx_2_2['发生时间'] = pd.to_datetime(ft_lx_2_2['发生时间'])
ft_lx_2_2['新主体'] = ft_lx_2_2.apply(get_new_subject, axis=1)

In [ ]:
ft_lx_2_3 = pd.merge(ft_lx_2_2, zt2, left_on = '新主体', right_on = '主体', how = 'left')
ft_lx_2_3['身份'] = np.where(ft_lx_2_3['身份'].isna(), '小规模纳税人', ft_lx_2_3['身份'])

In [ ]:
ft_lx_3 = ft_lx_2_3[~(ft_lx_2_3.发生时间 == '')]

#### 汇总

In [ ]:
ft_lx_4 = ft_lx_3.groupby(['店铺名称（聚水潭）', '店铺ID', '银行账号', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额']].sum().reset_index().rename(columns={'店铺名称（聚水潭）':'店铺名称'})

In [ ]:
ft_lx_4 = ft_lx_4[['平台', '合伙人', '品牌', '店铺ID', '身份', '店铺名称', '新主体', '银行账号', '类型', '金额', '来源文件']]

In [ ]:
ft_lx_4

### 淘系_聚合账户

In [ ]:
tx_jh = pd.read_sql(f'select * from `淘系_聚合账户`', conn)

In [ ]:
# 一行调用，直接处理整列
tx_jh['入账时间'] = convert_date_simple(tx_jh['入账时间'])

In [ ]:
tx_jh.rename(columns = {'收入金额_元': '收入金额', '支出金额': '支出金额', '入账时间':'发生时间'}, inplace = True)

#### 区分类型

In [ ]:
tx_jh_1 = tx_jh.copy()

In [ ]:
# 初始化
tx_jh_1['类型'] = '未知'
## 销售额 退款
tx_jh_1.loc[(tx_jh_1['入账类型'] == '交易收款'), '类型'] = '销售额-交易收款'
tx_jh_1.loc[(tx_jh_1['入账类型'] == '交易退款(售后)'), '类型'] = '退款-交易退款'
## 扣点
tx_jh_1.loc[(tx_jh_1['入账类型'] == '扣款') & (tx_jh_1['业务描述'] == '0030130|软件服务费-基础软件服务费'), '类型'] = '扣点-基础软件服务费'
tx_jh_1.loc[(tx_jh_1['入账类型'].isin(['扣款', '扣款退回'])) & (tx_jh_1['备注'].str.contains('品牌新享-首单拉新计划',na=False)), '类型'] = '扣点-拉新计划'
tx_jh_1.loc[(tx_jh_1['入账类型'].isin(['扣款', '扣款退回'])) & (tx_jh_1['备注'].str.contains('品牌新享-天猫营销托管软件服务费',na=False)), '类型'] = '扣点-营销托管服务费'
tx_jh_1.loc[(tx_jh_1['入账类型'].isin(['扣款', '扣款退回'])) & (tx_jh_1['业务描述'].str.contains('限时红包代商家垫付扣回',na=False)), '类型'] = '扣点-商家垫付扣回'
tx_jh_1.loc[(tx_jh_1['入账类型'] == '扣款') & (tx_jh_1['业务描述'].str.contains('公益性捐赠支出-公益宝贝',na=False)), '类型'] = '扣点-公益捐赠'
tx_jh_1.loc[(tx_jh_1['入账类型'].isin(['扣款', '扣款退回'])) & (tx_jh_1['业务描述'].str.contains('软件服务费-淘金币软件服务费',na=False)), '类型'] = '扣点-软件服务费-淘金币'
tx_jh_1.loc[(tx_jh_1['入账类型'] == '扣款') & (tx_jh_1['备注'].str.contains('淘特营销推广服务费',na=False)), '类型'] = '扣点-软件服务费-淘特营销'
tx_jh_1.loc[(tx_jh_1['入账类型'] == '扣款') & (tx_jh_1['备注'].str.contains('光合平台软件服务费',na=False)), '类型'] = '扣点-软件服务费-光合平台'
tx_jh_1.loc[(tx_jh_1['入账类型'] == '扣款') & (tx_jh_1['备注'].str.contains('先用后付技术服务费',na=False)), '类型'] = '扣点-先用后付服务费'
tx_jh_1.loc[(tx_jh_1['入账类型'] == '扣款') & (tx_jh_1['备注'].str.contains('品牌新享天猫超级老客加速软件服务费',na=False)), '类型'] = '扣点-软件服务费-老客加速'
tx_jh_1.loc[(tx_jh_1['入账类型'] == '扣款') & (tx_jh_1['业务描述'].str.contains('软件服务费-热浪引擎第三方服务商服务费',na=False)), '类型'] = '扣点-软件服务费-热浪引擎'
tx_jh_1.loc[(tx_jh_1['入账类型'] == '扣款') & (tx_jh_1['业务描述'].str.contains('服务费-大服饰全球包邮跨境服务基础费_信息技术服务',na=False)), '类型'] = '扣点-软件服务费-跨境基础费'
tx_jh_1.loc[(tx_jh_1['入账类型'] == '扣款') & (tx_jh_1['业务描述'].str.contains('技术&服务费-大服饰跨境服务增值费',na=False)), '类型'] = '扣点-软件服务费-跨境增值费'
tx_jh_1.loc[(tx_jh_1['入账类型'] == '扣款') & (tx_jh_1['备注'].str.contains('品牌新享新品孵化软件服务费',na=False)), '类型'] = '扣点-新品孵化服务费'
## 扣款
tx_jh_1.loc[(tx_jh_1['入账类型'] == '保证金扣款') & (tx_jh_1['备注'].str.contains('保证金管控资金使用',na=False)), '类型'] = '扣款-售后赔付'
## 提现
tx_jh_1.loc[(tx_jh_1['入账类型'] == '提现'), '类型'] = '提现'
## 转账
tx_jh_1.loc[(tx_jh_1['入账类型'] == '转账') & (tx_jh_1['业务描述'].str.contains('店铺过户资金调拨',na=False)), '类型'] = '其他货币资金-店铺过户'

In [ ]:
tx_jh_1['收入金额'] = tx_jh_1['收入金额'].str.replace(',', '').astype(float)
tx_jh_1['支出金额'] = tx_jh_1['支出金额'].str.replace(',', '').astype(float)

In [ ]:
## 支出金额
tx_jh_1.loc[
    tx_jh_1['类型'].isin([
    '扣点-基础软件服务费', '扣点-商家垫付扣回', '扣点-拉新计划',
    '扣点-营销托管服务费', '退款-交易退款', '提现', '扣点-公益捐赠',
    '扣点-软件服务费-淘金币', '扣点-软件服务费-淘特营销',  '扣点-软件服务费-光合平台',
    '扣点-先用后付服务费', '扣款-售后赔付', '扣点-软件服务费-老客加速',
    '扣点-软件服务费-热浪引擎', '扣点-软件服务费-跨境基础费', '扣点-软件服务费-跨境增值费',
    '扣点-新品孵化服务费', '其他支出-保证金退款', '其他支出-延迟发货', '其他支出-虚假发货', '其他支出-物流异常'
    ]), '支出金额'] = abs(tx_jh_1['支出金额'])

## 销售额
tx_jh_1.loc[(tx_jh_1['类型'].isin(['销售额-交易收款', '其他货币资金-店铺过户'])) & (tx_jh_1['收入金额'] != 0), '收入金额'] = abs(tx_jh_1['收入金额'])

## 收入金额
tx_jh_1.loc[
   (tx_jh_1['类型'].isin([
    '扣点-商家垫付扣回', '扣点-营销托管服务费', '扣点-拉新计划', '扣点-软件服务费-淘金币', '其他收入-淘宝保证金-扣除转移'
   ])) &
   (tx_jh_1['收入金额'] != 0), '收入金额'] = -tx_jh_1['收入金额']

In [ ]:
tx_jh_1['金额'] = round(tx_jh_1['收入金额'] + tx_jh_1['支出金额'],3)

In [ ]:
## 未知
tx_jh_1.loc[
    tx_jh_1['类型'].isin([
        '未知'
    ]), '金额'] = abs(tx_jh_1['金额'])

#### 未知类型

In [ ]:
tx_jh_2 = tx_jh_1[['店铺', '发生时间', '支付流水号', '淘宝订单编号', '入账类型', '业务描述', '备注', '类型', '金额']].copy()

In [ ]:
 # 筛选类型为未知
tx_jh_wz = tx_jh_2[(tx_jh_2['类型'] == '未知') & (tx_jh_2['发生时间'] != '')].copy()

# 定义处理函数（支持 if + elif + else 无限扩展）
def process_remark(x):
    s = str(x)
    if ('淘金币软件服务费' in s
            or '淘宝客佣金代扣款' in s
            or '店铺过户资金调拨' in s):
        return s[:8]        # 条件1：取前8字
    elif ('淘特营销推广服务费' in s
          or '光合平台软件服务费' in s
          or '淘宝内容推广服务费' in s
          or '先用后付技术服务费' in s):
        return s[:9]   # 条件2：自定义
    elif '公益宝贝捐赠' in s:
        return s[:6]   # 条件2：自定义
    elif '保证金退款' in s:
        return s[:5]   # 条件2：自定义
    elif '代扣款' in s:
        return s[:18]   # 条件2：自定义
    elif ('商家权益红包-预算追加-卖家延迟发货赔付红包-赔付红包' in s
          or '商家权益红包-预算追加-淘宝虚假发货赔付红包-赔付红包' in s
          or '商家权益红包-预算追加-淘宝物流轨迹异常红包-赔付红包' in s):
        return s[:27]   # 条件2：自定义
    elif '商家权益红包-预算追加-淘宝缺货赔付红包-赔付红包' in s:
        return s[:25]   # 条件2：自定义
    elif '淘宝联盟推广佣金返还' in s:
        return s[:10]   # 条件2：自定义
    elif ('淘宝新客礼金技术服务费' in s
          or '淘宝天猫跨境服务增值费' in s
          or '淘宝天猫跨境服务基础费' in s
          or '限时红包代商家垫付扣回' in s):
        return s[:11]        # 条件3：取前5字
    elif '天猫APP专享折扣服务费' in s:
        return s[:12]
    elif '品牌新享新品孵化软件服务费' in s:
        return s[:13]
    elif ('百亿补贴软件服务费T62（全渠道）' in s
          or '品牌新享天猫超级老客加速软件服务费' in s
          ):
        return s[:17]
    elif '保险承保-天猫海外退货险保费收取' in s:
        return s[:16]
    elif '保证金管控资金使用' in s:
        return s[-9:]
    elif '店铺过户' in s:
        return s[:4]
    else:
        return x            # 其他：保留原样

tx_jh_wz_1 = tx_jh_wz[['店铺', '支付流水号', '淘宝订单编号', '入账类型', '备注', '业务描述', '金额']].assign(
    备注=lambda df: df['备注'].apply(process_remark)
).drop_duplicates()

In [ ]:
tx_jh_wz_1['来源文件'] = '淘系_聚合账户'

In [ ]:
tx_jh_wz_1

#### 店铺表格

In [ ]:
tx_jh_2['店铺'] = tx_jh_2['店铺'].astype(int)
dp_bg['店铺ID'] = dp_bg['店铺ID'].astype(int)

In [ ]:
tx_jh_2_1 = pd.merge(tx_jh_2, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
tx_jh_2_1 = tx_jh_2_1.copy()
for field in ['合伙人', '品牌']:
    tx_jh_2_1 = optimize_field_by_date(tx_jh_2_1, field)

In [ ]:
tx_jh_2_1['来源文件'] = '淘系_聚合账户'

#### 主体

In [ ]:
tx_jh_2_1['店铺ID'] = tx_jh_2_1['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
tx_jh_2_2 = pd.merge(tx_jh_2_1, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
tx_jh_2_2['发生时间'] = pd.to_datetime(tx_jh_2_2['发生时间'])
tx_jh_2_2['新主体'] = tx_jh_2_2.apply(get_new_subject, axis=1)

In [ ]:
tx_jh_2_3 = pd.merge(tx_jh_2_2, zt2, left_on = '新主体', right_on = '主体', how = 'left')
tx_jh_2_3['身份'] = np.where(tx_jh_2_3['身份'].isna(), '小规模纳税人', tx_jh_2_3['身份'])

In [ ]:
tx_jh_3 = tx_jh_2_3[~(tx_jh_2_3.发生时间 == '')]

In [ ]:
tx_jh_3

#### 汇总

In [ ]:
tx_jh_4 = tx_jh_3.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额']].sum().reset_index().rename(columns={'店铺名称（聚水潭）':'店铺名称'})

In [ ]:
tx_jh_4 = tx_jh_4[['平台', '合伙人', '品牌', '店铺ID', '身份', '店铺名称', '新主体', '类型', '金额', '来源文件']]

In [ ]:
tx_jh_4

### 淘系_保证金

In [ ]:
tx_bzj = pd.read_sql(f'select * from `淘系_保证金`', conn)

In [ ]:
# 一行调用，直接处理整列
tx_bzj['完成时间'] = convert_date_simple(tx_bzj['完成时间'])

In [ ]:
tx_bzj.rename(columns = {'收支金额_元': '收支金额', '完成时间':'发生时间'}, inplace = True)

#### 区分类型

In [ ]:
tx_bzj_1 = tx_bzj.copy()

In [ ]:
tx_bzj_1['收支金额'] = tx_bzj_1['收支金额'].str.replace(',', '').astype(float)

In [ ]:
# 初始化
tx_bzj_1['类型'] = '未知'
## 保证金-淘宝保证金-充值
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '充值') &
    (tx_bzj_1['业务描述'].str.contains(
        '008002800014\\|保证金-淘宝-出账缴存|008002800001\\|保证金-淘宝-缴存（退款锁定）', na = False
    )), '类型'] = '保证金-淘宝保证金-充值'
## 保证金-扣款-违规违约金
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '出账') &
    (tx_bzj_1['业务描述'].str.contains('0070001|其他支出-违规违约金（保证金扣款）', na = False)) &
    (tx_bzj_1['原因'].str.contains('违约金罚扣/描述或品质不符', na = False)), '类型'] = '扣款-违规违约金'
## 保证金-运费赔付
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '出账') &
    (tx_bzj_1['业务描述'].str.contains('0020003\\|交易退款-保证金退款|0070002\\|其他支出-交易赔付（保证金扣款）', na = False)) &
    (tx_bzj_1['原因'].str.contains('交易赔付/争议赔付/额外赔付/运费|交易赔付/争议处理/运费争议', na = False)), '类型'] = '扣款-运费赔付'
## 保证金-欠费赔付
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '出账') &
    (tx_bzj_1['业务描述'].str.contains('0020002|交易退款-保证金退款', na = False)) &
    (tx_bzj_1['原因'].str.contains('欠费划扣/平台垫资欠费/运费垫资欠费', na = False)), '类型'] = '扣款-欠费赔付'
## 保证金-扣除转移
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '出账') &
    (tx_bzj_1['业务描述'].str.contains('0020002|交易退款-保证金退款', na = False)) &
    (tx_bzj_1['原因'].str.contains('交易售后$', na = False)), '类型'] = '退款-交易退款'
## 保证金-扣除转移
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '出账') &
    (tx_bzj_1['业务描述'].str.contains('0070003|其他支出-交易赔付（保证金扣款）', na = False)) &
    (tx_bzj_1['原因'].str.contains('交易赔付/违背承诺/违背发货承诺/虚假发货', na = False)), '类型'] = '扣款-虚假发货'
## 保证金-扣除转移
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '出账') &
    (tx_bzj_1['业务描述'].str.contains('0070004|其他支出-交易赔付（保证金扣款）', na = False)) &
    (tx_bzj_1['原因'].str.contains('交易赔付/违背承诺/违背发货承诺/物流轨迹异常', na = False)), '类型'] = '扣款-物流异常'
## 保证金-扣除转移
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '出账') &
    (tx_bzj_1['业务描述'].str.contains('0070005|其他支出-交易赔付（保证金扣款）', na = False)) &
    (tx_bzj_1['原因'].str.contains('交易赔付$', na = False)), '类型'] = '扣款-交易赔付'
## 保证金-扣除转移
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '出账') &
    (tx_bzj_1['业务描述'].str.contains('008002800024|保证金-淘宝-延迟发货赔付（红包冻结）', na = False)) &
    (tx_bzj_1['原因'].str.contains('交易赔付/违背承诺/违背发货承诺/延迟发货', na = False)), '类型'] = '扣款-延迟发货'
## 保证金-天猫保证金-充值
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '充值') &
    (tx_bzj_1['业务描述'].str.contains('008002800014\\|保证金-天猫-出账缴存', na = False)), '类型'] = '保证金-天猫保证金-充值'
## 保证金-延迟发货
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '出账') &
    (tx_bzj_1['业务描述'] == '0070002|其他支出-交易赔付（保证金扣款）') &
    (tx_bzj_1['原因'].str.contains('交易赔付/违背承诺/违背发货承诺/延迟发货', na = False)), '类型'] = '扣款-延迟发货'
## 保证金-物流异常
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '出账') &
    (tx_bzj_1['业务描述'] == '0070002|其他支出-交易赔付（保证金扣款）') &
    (tx_bzj_1['原因'].str.contains('交易赔付/违背承诺/违背发货承诺/物流轨迹异常|交易赔付/违背承诺/违背发货承诺/物流轨迹超时', na = False)), '类型'] = '扣款-物流异常'
## 保证金-缺货
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '出账') &
    (tx_bzj_1['业务描述'] == '0070002|其他支出-交易赔付（保证金扣款）')  &
    (tx_bzj_1['原因'].str.contains('交易赔付/违背承诺/违背发货承诺/缺货', na = False)), '类型'] = '扣款-缺货'
## 保证金-淘宝保证金-充值
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '充值') &
    (tx_bzj_1['业务描述'].str.contains(r'008002800015\|保证金-淘宝-额度补齐缴存|008002800006\|保证金-淘宝-缴存', na = False)), '类型'] = '保证金-淘宝保证金-充值'
## 扣款-运费赔付
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '出账') &
    (tx_bzj_1['业务描述'] == '0020002|交易退款-保证金退款')  &
    (tx_bzj_1['原因'].str.contains('交易赔付/争议赔付/额外赔付/运费', na = False)), '类型'] = '扣款-运费赔付'
## 扣款-欠费划扣
tx_bzj_1.loc[
    (tx_bzj_1['操作类型'] == '出账') &
    (tx_bzj_1['业务描述'] == '0070003|其他支出-天猫积分发票违约金（保证金扣款）')  &
    (tx_bzj_1['原因'].str.contains('欠费划扣', na = False)), '类型'] = '扣款-欠费划扣'

In [ ]:
## 收支金额
tx_bzj_1.loc[
    tx_bzj_1['类型'].isin([
    '保证金-淘宝保证金-充值', '扣款-违规违约金', '退款-交易退款',
    '扣款-运费赔付', '扣款-欠费赔付', '扣款-运费赔付', '扣款-欠费划扣',
    '扣款-虚假发货', '扣款-物流异常', '扣款-交易赔付',
    '保证金-天猫保证金-充值', '扣款-延迟发货', '扣款-缺货'
    ]), '收支金额'] = abs(tx_bzj_1['收支金额'])

In [ ]:
tx_bzj_1['金额'] = round(tx_bzj_1['收支金额'],3)

In [ ]:
## 未知
tx_bzj_1.loc[
    tx_bzj_1['类型'].isin([
        '未知'
    ]), '金额'] = abs(tx_bzj_1['金额'])

In [ ]:
tx_bzj_1[tx_bzj_1['类型'].str.contains('天猫', na = False)]

#### 未知类型

In [ ]:
tx_bzj_2 = tx_bzj_1[['店铺', '发生时间', '订单编号', '操作类型', '原因', '业务描述', '类型', '金额']].copy()

In [ ]:
 # 筛选类型为未知
tx_bzj_wz = tx_bzj_2[(tx_bzj_2['类型'] == '未知') & (tx_bzj_2['发生时间'] != '')].copy()

tx_bzj_wz_1 = tx_bzj_wz[['店铺', '订单编号', '操作类型', '原因', '业务描述', '金额']].drop_duplicates()

In [ ]:
tx_bzj_wz_1['来源文件'] = '淘系_保证金'

In [ ]:
tx_bzj_wz_1

#### 店铺表格

In [ ]:
tx_bzj_2['店铺'] = tx_bzj_2['店铺'].astype(int)
dp_bg['店铺ID'] = dp_bg['店铺ID'].astype(int)

In [ ]:
tx_bzj_2_1 = pd.merge(tx_bzj_2, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
tx_bzj_2_1 = tx_bzj_2_1.copy()
for field in ['合伙人', '品牌']:
    tx_bzj_2_1 = optimize_field_by_date(tx_bzj_2_1, field)

In [ ]:
tx_bzj_2_1['来源文件'] = '淘系_保证金'

#### 主体

In [ ]:
tx_bzj_2_1['店铺ID'] = tx_bzj_2_1['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
tx_bzj_2_2 = pd.merge(tx_bzj_2_1, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
tx_bzj_2_2['发生时间'] = pd.to_datetime(tx_bzj_2_2['发生时间'])
tx_bzj_2_2['新主体'] = tx_bzj_2_2.apply(get_new_subject, axis=1)

In [ ]:
tx_bzj_2_3 = pd.merge(tx_bzj_2_2, zt2, left_on = '新主体', right_on = '主体', how = 'left')
tx_bzj_2_3['身份'] = np.where(tx_bzj_2_3['身份'].isna(), '小规模纳税人', tx_bzj_2_3['身份'])

In [ ]:
tx_bzj_3 = tx_bzj_2_3[~(tx_bzj_2_3.发生时间 == '')]

In [ ]:
tx_bzj_3

#### 汇总

In [ ]:
tx_bzj_4 = tx_bzj_3.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额']].sum().reset_index().rename(columns={'店铺名称（聚水潭）':'店铺名称'})

In [ ]:
tx_bzj_4 = tx_bzj_4[['平台', '合伙人', '品牌', '店铺ID', '身份', '店铺名称', '新主体', '类型', '金额', '来源文件']]

In [ ]:
tx_bzj_4

### 淘系_推广

In [ ]:
tx_tg = pd.read_sql(f'select * from `淘系_推广`', conn)

In [ ]:
# 一行调用，直接处理整列
tx_tg['记账时间'] = convert_date_simple(tx_tg['记账时间'])

In [ ]:
tx_tg.rename(columns = {'操作金额元': '操作金额', '记账时间':'发生时间'}, inplace = True)

#### 区分类型

In [ ]:
tx_tg_1 = tx_tg.copy()

In [ ]:
tx_tg_1['操作金额'] = tx_tg_1['操作金额'].str.replace(',', '').astype(float)

In [ ]:
# 初始化
tx_tg_1['类型'] = '未知'
## 万相台-支付宝充值-自动
tx_tg_1.loc[
    (tx_tg_1['收支类型'] == '收入') &
    (tx_tg_1['备注'].str.contains('支付宝自动充值', na = False)), '类型'] = '万相台-支付宝充值-自动'
## 万相台-货品全站扣款
tx_tg_1.loc[
    (tx_tg_1['收支类型'] == '支出') &
    (tx_tg_1['备注'].str.contains('货品全站推消耗扣款', na = False)), '类型'] = '万相台-货品全站扣款'
## 万相台-现金扣款
tx_tg_1.loc[
    (tx_tg_1['收支类型'] == '支出') &
    (tx_tg_1['备注'].str.contains('现金消耗扣款', na = False)), '类型'] = '万相台-现金扣款'
## 万相台-支付宝充值-手动
tx_tg_1.loc[
    (tx_tg_1['收支类型'] == '收入') &
    (tx_tg_1['备注'].str.contains('支付宝在线充值', na = False)), '类型'] = '万相台-支付宝充值-手动'

In [ ]:
## 收支金额
tx_tg_1.loc[
    tx_tg_1['类型'].isin([
    '万相台-支付宝充值-自动', '万相台-货品全站扣款', '万相台-现金扣款','万相台-支付宝充值-手动',
    ]), '操作金额'] = abs(tx_tg_1['操作金额'])

In [ ]:
tx_tg_1['金额'] = round(tx_tg_1['操作金额'],3)

In [ ]:
## 未知
tx_tg_1.loc[
    tx_tg_1['类型'].isin([
        '未知'
    ]), '金额'] = abs(tx_tg_1['金额'])

In [ ]:
tx_tg_1

#### 未知类型

In [ ]:
 # 筛选类型为未知
tx_tg_wz = tx_tg_1[(tx_tg_1['类型'] == '未知') & (tx_tg_1['金额'] != 0)].copy()

tx_tg_wz = tx_tg_wz[['店铺', '收支类型', '交易类型', '备注', '金额']].drop_duplicates()

In [ ]:
tx_tg_wz['来源文件'] = '淘系_推广账户'

In [ ]:
tx_tg_wz

#### 店铺表格

In [ ]:
tx_tg_1['店铺'] = tx_tg_1['店铺'].astype(int)
dp_bg['店铺ID'] = dp_bg['店铺ID'].astype(int)

In [ ]:
tx_tg_2 = pd.merge(tx_tg_1, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
tx_tg_2 = tx_tg_2.copy()
for field in ['合伙人', '品牌']:
    tx_tg_2 = optimize_field_by_date(tx_tg_2, field)

In [ ]:
tx_tg_2['来源文件'] = '淘系_推广账户'

#### 主体

In [ ]:
tx_tg_2['店铺ID'] = tx_tg_2['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
tx_tg_2_1 = pd.merge(tx_tg_2, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
tx_tg_2_1['发生时间'] = pd.to_datetime(tx_tg_2_1['发生时间'])
tx_tg_2_1['新主体'] = tx_tg_2_1.apply(get_new_subject, axis=1)

In [ ]:
tx_tg_2_2 = pd.merge(tx_tg_2_1, zt2, left_on = '新主体', right_on = '主体', how = 'left')
tx_tg_2_2['身份'] = np.where(tx_tg_2_2['身份'].isna(), '小规模纳税人', tx_tg_2_2['身份'])

In [ ]:
tx_tg_2_2

#### 汇总

In [ ]:
tx_tg_3 = tx_tg_2_2.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额']].sum().reset_index().rename(columns={'店铺名称（聚水潭）':'店铺名称'})

In [ ]:
tx_tg_4 = tx_tg_3[['平台', '合伙人', '品牌', '店铺ID', '身份', '店铺名称', '新主体', '类型', '金额', '来源文件']].copy()

In [ ]:
tx_tg_4

### 未知保存

In [ ]:
tx_wz = pd.concat([ft_lx_wz_1, tx_jh_wz_1, tx_bzj_wz_1, tx_tg_wz], ignore_index=True)
tx_wz

### 汇总

In [ ]:
ft_hz = pd.concat([ft_lx_4, tx_jh_4, tx_bzj_4, tx_tg_4], ignore_index=True)

In [ ]:
ft_hz

### 处理后

In [ ]:
# 定义需要保存的DataFrame和对应的文件路径
data_to_save = [
    (tx_wz, fr'../../结果/未知/淘系未知类型.csv'),
    (ft_hz, fr'../../结果/2026年{int(month)}月/淘系/淘系(待处理).csv'),
]
# 分离DataFrame和文件路径列表
dataframes = [item[0] for item in data_to_save]
file_paths = [item[1] for item in data_to_save]
# 调用函数保存数据
try:
    save_dataframes(dataframes, file_paths)
except Exception as e:
    print(f"保存数据时出错: {e}")

## 淘工厂

### 淘工厂_支付宝

In [ ]:
tgc = pd.read_sql(f'select * from [支付宝账单]', conn2)

In [ ]:
tgc = tgc.replace(' ', '').replace('', np.nan).fillna(0)

In [ ]:
# 一行调用，直接处理整列
tgc['入账时间'] = convert_date_simple(tgc['入账时间'])

In [ ]:
tgc.rename(columns={'收入（+元）': '收入金额', '支出（_元）': '支出金额', '入账时间': '发生时间'}, inplace=True)

#### 店铺表格

In [ ]:
tgc['店铺'] = tgc['店铺'].astype(int)
dp_bg['店铺ID'] = dp_bg['店铺ID'].astype(int)

In [ ]:
tgc_1 = pd.merge(tgc, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
tgc_1 = tgc_1.copy()
for field in ['合伙人', '品牌']:
    tgc_1 = optimize_field_by_date(tgc_1, field)

In [ ]:
tgc_2 = tgc_1[(tgc_1['平台'] == '淘工厂') & (tgc_1['发生时间'].str[:7] == f'2026-{month}')]

In [ ]:
tgc_2['收入金额'] = tgc_2['收入金额'].astype(float)
tgc_2['支出金额'] = tgc_2['支出金额'].astype(float)

In [ ]:
import pandas as pd
import numpy as np

# ===================== 初始化 =====================
tgc_2['类型'] = '未知'

# 别名简化（缩短长列名，不改动数据）
tx = tgc_2['账务类型']
desc = tgc_2['业务描述']
note = tgc_2['备注']
goods = tgc_2['商品名称']

# 统一预计算函数（向量化，只算一次）
note_has = lambda s: note.str.contains(s, na=False)
desc_has = lambda s: desc.str.contains(s, na=False)
tx_has = lambda s: tx.str.contains(s, na=False)

# ===================== 规则：一行一条，纯向量化掩码赋值 =====================
# 销售额
tgc_2.loc[tx_has('转账') & desc_has('货款-交易货款'), '类型'] = '销售额-商品货款'
tgc_2.loc[tx_has('转账') & note_has('交易补贴'), '类型'] = '销售额-交易补贴'

# 退款
tgc_2.loc[tx_has('在线支付') & goods.str.contains('订单交易货款分账退回', na=False), '类型'] = '退款-商品货款'

# 扣点
tgc_2.loc[tx_has('转账') & note_has('逆向退款-支付宝定向推广费用'), '类型'] = '扣点-营销费用'
tgc_2.loc[tx_has('转账') & note_has('直营&联营&营促销'), '类型'] = '扣点-技术服务费'
tgc_2.loc[tx_has('在线支付') & goods.str.contains('直营&联营&营促销', na=False), '类型'] = '扣点-技术服务费'
tgc_2.loc[tx_has('在线支付') & goods.str.contains('退货包运费代扣', na=False), '类型'] = '扣点-退货包运费'
tgc_2.loc[tx_has('在线支付') & goods.str.contains('正向扣佣-支付宝定向推广费用', na=False), '类型'] = '扣点-营销费用'
tgc_2.loc[tx_has('在线支付') & goods.str.contains('技术服务费-售后客服服务费', na=False), '类型'] = '扣点-客服服务费'
tgc_2.loc[tx_has('在线支付') & goods.str.contains('技术服务费-售前客服服务费', na=False), '类型'] = '扣点-客服服务费'
tgc_2.loc[tx_has('在线支付') & goods.str.contains('技术服务费-退款客服服务费', na=False), '类型'] = '扣点-客服服务费'
tgc_2.loc[tx_has('在线支付') & goods.str.contains('正向扣佣-投流推广', na=False), '类型'] = '扣点-投流推广'
tgc_2.loc[tx_has('在线支付') & goods.str.contains('正向扣款-先用后付技术服务费', na=False), '类型'] = '扣点-先用后付'

# 扣款
tgc_2.loc[tx_has('转账') & note_has('逆向退款-直播推广服务费'), '类型'] = '扣款-佣金'
tgc_2.loc[tx_has('在线支付') & goods.str.contains('赔付追缴', na=False), '类型'] = '扣款-赔付追缴'
tgc_2.loc[tx_has('在线支付') & goods.str.contains('商家处罚', na=False), '类型'] = '扣款-处罚'
tgc_2.loc[tx_has('在线支付') & goods.str.contains('正向扣款-直播推广服务费', na=False), '类型'] = '扣款-佣金'
tgc_2.loc[tx_has('在线支付') & goods.str.contains('正向扣佣-精选淘客', na=False), '类型'] = '扣款-佣金'

# 其他
tgc_2.loc[tx_has('在线支付') & goods.str.contains('账户充值', na=False), '类型'] = '推广充值'
tgc_2.loc[tx_has('提现'), '类型'] = '提现'

In [ ]:
tgc_2['金额'] = abs(tgc_2['支出金额']) - tgc_2['收入金额']

In [ ]:
# ===================== 金额修正（完全匹配原逻辑） =====================
# 销售额-商品货款
cond1 = tgc_2['类型'] == '销售额-商品货款'
tgc_2.loc[cond1 & (tgc_2['金额'] != 0), '金额'] = tgc_2['金额'].abs()

# 销售额-交易补贴
cond2 = tgc_2['类型'] == '销售额-交易补贴'
tgc_2.loc[cond2 & (tgc_2['金额'] != 0), '金额'] = tgc_2['金额'].abs()

# 退款-商品货款
cond3 = tgc_2['类型'] == '退款-商品货款'
tgc_2.loc[cond3 & (tgc_2['金额'] != 0), '金额'] = tgc_2['金额'].abs()

# 扣点类（负收入）
cond4 = tgc_2['类型'].isin([
    '扣点-营销费用', '扣点-技术服务费', '扣点-退货包运费',
    '扣点-客服服务费', '扣点-投流推广', '扣点-先用后付'
])
tgc_2.loc[cond4 & (tgc_2['金额'] != 0), '金额'] = tgc_2['金额']

# 扣款类
cond5 = tgc_2['类型'].isin(['扣款-佣金', '扣款-赔付追缴', '扣款-处罚'])
tgc_2.loc[cond5 & (tgc_2['金额'] != 0), '金额'] = tgc_2['金额'].abs()

# 推广充值
cond6 = tgc_2['类型'] == '推广充值'
tgc_2.loc[cond6 & (tgc_2['金额'] != 0), '金额'] = tgc_2['金额'].abs()

# 提现
cond7 = tgc_2['类型'] == '提现'
tgc_2.loc[cond7 & (tgc_2['金额'] != 0), '金额'] = tgc_2['金额'].abs()

In [ ]:
tgc_2['类型'].value_counts()

#### 未知类型

In [ ]:
tgc_wz_1 = tgc_2[tgc_2['类型'] == '未知']
tgc_wz_1

#### 主体

In [ ]:
tgc_2['店铺ID'] = tgc_2['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
tgc_2_1 = pd.merge(tgc_2, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
tgc_2_1['发生时间'] = pd.to_datetime(tgc_2_1['发生时间'])
tgc_2_1['新主体'] = tgc_2_1.apply(get_new_subject, axis=1)

In [ ]:
tgc_3 = pd.merge(tgc_2_1, zt2, left_on = '新主体', right_on = '主体', how = 'left')
tgc_3['身份'] = np.where(tgc_3['身份'].isna(), '小规模纳税人', tgc_3['身份'])

In [ ]:
tgc_3['来源文件'] = "淘工厂_支付宝"

In [ ]:
tgc_3

#### 处理后

In [ ]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
tgc_4 = tgc_3.groupby(['店铺名称（聚水潭）', '店铺ID', '银行账号', '新主体', '身份', '平台', '合伙人', '品牌', '类型',"来源文件"])[['金额']].sum().round(2).reset_index()

# 重命名列名
tgc_4 = tgc_4.rename(columns={'店铺名称（聚水潭）':'店铺名称', '银行账号':'支付宝账号'})

In [ ]:
tgc_4 = tgc_4[['店铺名称', '店铺ID', '支付宝账号', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '金额',"来源文件"]].copy()

In [ ]:
tgc_4

### 淘工厂_微信

In [ ]:
tgc_wx = pd.read_sql(f'select * from 淘工厂_货款账户', conn2)

In [ ]:
tgc_wx = tgc_wx.replace(' ', '').replace('', np.nan).fillna(0)

In [ ]:
# 一行调用，直接处理整列
tgc_wx['动账时间'] = convert_date_simple(tgc_wx['动账时间'])

In [ ]:
tgc_wx.rename(columns={'动账时间': '发生时间'}, inplace=True)

#### 店铺表格

In [ ]:
tgc_wx['店铺'] = tgc_wx['店铺'].astype(int)
dp_bg['店铺ID'] = dp_bg['店铺ID'].astype(int)

In [ ]:
tgc_wx_1 = pd.merge(tgc_wx, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
tgc_wx_1 = tgc_wx_1.copy()
for field in ['合伙人', '品牌']:
    tgc_wx_1 = optimize_field_by_date(tgc_wx_1, field)

In [ ]:
tgc_wx_2 = tgc_wx_1[(tgc_wx_1['发生时间'].str[:7] == f'2026-{month}')]

In [ ]:
tgc_wx_2['金额'] = tgc_wx_2['金额'].astype(float)

In [ ]:
import pandas as pd
import numpy as np

# 初始化类型列为未知
tgc_wx_2['类型'] = '未知'
tgc_wx_2['金额'] = tgc_wx_2['金额']

# ===================== 统一规则列表：[条件, 赋值结果] =====================
rule_list = [
    # 销售额
    (tgc_wx_2['资金方向'].str.contains('收入', na=False) & tgc_wx_2['备注说明'].str.contains('订单交易货款分账', na=False)), '销售额-商品货款', abs(tgc_wx_2['金额']),
    (tgc_wx_2['资金方向'].str.contains('收入', na=False) & tgc_wx_2['备注说明'].str.contains('交易补贴', na=False)), '销售额-交易补贴', abs(tgc_wx_2['金额']),
    # 退款
    (tgc_wx_2['资金方向'].str.contains('支出', na=False) & tgc_wx_2['备注说明'].str.contains('订单交易货款分账退回', na=False)), '退款-商品货款', abs(tgc_wx_2['金额']),
    # 扣点
    (tgc_wx_2['资金方向'].str.contains('收入', na=False) & tgc_wx_2['备注说明'].str.contains('直营&联营&营促销', na=False)), '扣点-技术服务费', -tgc_wx_2['金额'],
    (tgc_wx_2['资金方向'].str.contains('支出', na=False) & tgc_wx_2['备注说明'].str.contains('正向扣佣-投流推广', na=False)), '扣点-投流推广', abs(tgc_wx_2['金额']),
    (tgc_wx_2['资金方向'].str.contains('支出', na=False) & tgc_wx_2['备注说明'].str.contains('直营&联营&营促销', na=False)), '扣点-技术服务费', abs(tgc_wx_2['金额']),
    # 扣款
    (tgc_wx_2['资金方向'].str.contains('支出', na=False) & tgc_wx_2['备注说明'].str.contains('正向扣款-直播推广服务费', na=False)), '扣款-佣金', abs(tgc_wx_2['金额']),
    # 其他
    (tgc_wx_2['资金方向'].str.contains('支出', na=False) & tgc_wx_2['备注说明'].str.contains('供应商提现', na=False)), '提现', abs(tgc_wx_2['金额']),
]

# ===================== 批量应用所有规则 =====================
# 两两一组：条件、结果
for i in range(0, len(rule_list), 3):
    cond = rule_list[i]
    value = rule_list[i+1]
    money = rule_list[i+2]
    tgc_wx_2.loc[cond, '类型'] = value
    tgc_wx_2.loc[cond & tgc_wx_2['金额'] != 0, '金额'] = money

In [ ]:
tgc_wx_2['类型'].value_counts()

#### 未知类型

In [ ]:
tgc_wx_wz = tgc_wx_2[tgc_wx_2['类型'] == '未知']
tgc_wx_wz

#### 主体

In [ ]:
tgc_wx_2['店铺ID'] = tgc_wx_2['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
tgc_wx_2_1 = pd.merge(tgc_wx_2, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
tgc_wx_2_1['发生时间'] = pd.to_datetime(tgc_wx_2_1['发生时间'])
tgc_wx_2_1['新主体'] = tgc_wx_2_1.apply(get_new_subject, axis=1)

In [ ]:
tgc_wx_3 = pd.merge(tgc_wx_2_1, zt2, left_on = '新主体', right_on = '主体', how = 'left')
tgc_wx_3['身份'] = np.where(tgc_wx_3['身份'].isna(), '小规模纳税人', tgc_wx_3['身份'])

In [ ]:
tgc_wx_3

#### 处理后

In [ ]:
tgc_wx_3['来源文件'] = "淘工厂_微信"

In [ ]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
tgc_wx_4 = tgc_wx_3.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型',"来源文件"])[['金额']].sum().round(2).reset_index()

# 重命名列名
tgc_wx_4 = tgc_wx_4.rename(columns={'店铺名称（聚水潭）':'店铺名称'})

In [ ]:
tgc_wx_4 = tgc_wx_4[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '金额',"来源文件"]].copy()

In [ ]:
tgc_wx_4

### 淘工厂_推广

In [ ]:
tgc_tg = pd.read_sql(f'select * from `淘工厂_推广`', conn)

In [ ]:
tgc_tg = tgc_tg.replace(' ', '').replace('', np.nan).fillna(0)

In [ ]:
# 一行调用，直接处理整列
tgc_tg['结算时间'] = convert_date_simple(tgc_tg['结算时间'])

In [ ]:
tgc_tg.rename(columns={'结算时间': '发生时间'}, inplace=True)

In [ ]:
import pandas as pd
import numpy as np

# 初始化类型列为未知
tgc_tg['类型'] = '未知'

# ===================== 统一规则列表：[条件, 赋值结果] =====================
rule_list = [
    # 销售额
    (tgc_tg['收支类型'].str.contains('收入', na=False) & (tgc_tg['交易类型'].str.contains('【协议扣款】- 自动充值', na=False))), '万相台-自动充值',
    (tgc_tg['收支类型'].str.contains('支出', na=False) & (tgc_tg['交易类型'].str.contains('充值加码消耗', na=False))), '扣款-消耗',

]

# ===================== 批量应用所有规则 =====================
# 两两一组：条件、结果
for i in range(0, len(rule_list), 2):
    cond = rule_list[i]
    value = rule_list[i+1]
    tgc_tg.loc[cond, '类型'] = value

In [ ]:
tgc_tg['金额'] = tgc_tg['金额'].astype(float)

In [ ]:
## 收支金额
tgc_tg.loc[
    tgc_tg['类型'].isin([
    '万相台-自动充值', '扣款-消耗'
    ]), '金额'] = abs(tgc_tg['金额'])

In [ ]:
## 收支金额
tgc_tg.loc[
    tgc_tg['类型'].isin([
    '未知'
    ]), '金额'] = abs(tgc_tg['金额'])

In [ ]:
tgc_tg['类型'].value_counts()

#### 未知类型

In [ ]:
tgc_tg_wz = tgc_tg[tgc_tg['类型'] == '未知']
tgc_tg_wz

#### 店铺表格

In [ ]:
tgc_tg['店铺'] = tgc_tg['店铺'].astype(int)

In [ ]:
tgc_tg_1 = pd.merge(tgc_tg, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
tgc_tg_1 = tgc_tg_1.copy()
for field in ['合伙人', '品牌']:
    tgc_tg_1 = optimize_field_by_date(tgc_tg_1, field)

#### 主体

In [ ]:
tgc_tg_1['店铺ID'] = tgc_tg_1['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
tgc_tg_2 = pd.merge(tgc_tg_1, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
tgc_tg_2['发生时间'] = pd.to_datetime(tgc_tg_2['发生时间'])
tgc_tg_2['新主体'] = tgc_tg_2.apply(get_new_subject, axis=1)

In [ ]:
tgc_tg_3 = pd.merge(tgc_tg_2, zt2, left_on = '新主体', right_on = '主体', how = 'left')
tgc_tg_3['身份'] = np.where(tgc_tg_3['身份'].isna(), '小规模纳税人', tgc_tg_3['身份'])

In [ ]:
tgc_tg_3

#### 汇总

In [ ]:
tgc_tg_3['来源文件'] = "淘工厂_推广账户"

In [ ]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
tgc_tg_4 = tgc_tg_3.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型',"来源文件"])[['金额']].sum().round(2).reset_index()

# 重命名列名
tgc_tg_4 = tgc_tg_4.rename(columns={'店铺名称（聚水潭）':'店铺名称'})

In [ ]:
tgc_tg_4 = tgc_tg_4[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '金额',"来源文件"]].copy()

In [ ]:
tgc_tg_4

### 处理后

#### 未知

In [ ]:
tgc_wz = pd.concat([tgc_wz_1, tgc_wx_wz, tgc_tg_wz], ignore_index = True)
tgc_wz

#### 汇总

In [ ]:
tgc_hz = pd.concat([tgc_4, tgc_wx_4, tgc_tg_4], ignore_index = True)

In [ ]:
tgc_hz

In [ ]:
pd.set_option('future.no_silent_downcasting', True)  # 启用未来行为
tgc_hz = tgc_hz.fillna(0).infer_objects()

In [ ]:
tgc_hz_1 = tgc_hz.groupby(['店铺名称', '店铺ID', '支付宝账号', '新主体', '身份', '平台', '合伙人', '品牌', '类型','来源文件'])['金额'].sum().reset_index()
tgc_hz_1

In [ ]:
# 定义需要保存的DataFrame和对应的文件路径
data_to_save = [
    (tgc_wz, fr'../../结果/未知/淘工厂未知类型.csv'),
    (tgc_hz, fr'../../结果/2026年{int(month)}月/淘工厂/淘工厂账单2.0(待处理).csv'),
]
# 分离DataFrame和文件路径列表
dataframes = [item[0] for item in data_to_save]
file_paths = [item[1] for item in data_to_save]
# 调用函数保存数据
try:
    save_dataframes(dataframes, file_paths)
except Exception as e:
    print(f"保存数据时出错: {e}")

## 得物

### 得物_销售订单

In [ ]:
dw = pd.read_sql(f'select * from `得物_销售订单`', conn)

In [ ]:
# 一行调用，直接处理整列
dw['业务时间'] = convert_date_simple(dw['业务时间'])

In [ ]:
dw.rename(columns = {'业务时间':'发生时间'}, inplace = True)

In [ ]:
dw_1 = dw.copy()

In [ ]:
import pandas as pd

# 定义需要转 float 的所有列名列表
float_cols = [
    '商品金额', '联合营销费', '其中:基础服务费金额', '其中:履约服务费金额', '消费者邮费补贴金额',
]

# 批量转换类型（安全写法，自动跳过不存在的列）
for col in float_cols:
    if col in dw_1.columns:
        dw_1[col] = pd.to_numeric(dw_1[col], errors='coerce')

In [ ]:
import pandas as pd
import numpy as np

# 1. 初始化（完全对齐你给的新版格式）
dw_1_cl = dw_1.copy()
dw_1_cl["类型"] = "未知"
dw_1_cl["金额"] = 0.0
new_rows = []

# 统一简写
df = dw_1_cl

# 2. 遍历每一行，按规则生成所有类型（逐条规则对应拆分）
for idx, row in df.iterrows():
    r = row.to_dict()

    # ------------------- 规则1：销售额-商品金额 -------------------
    if pd.notnull(r["商品金额"]):
        new_rows.append({**r, "类型": "销售额-商品金额", "金额": abs(r["商品金额"])})

    # ------------------- 规则2：扣点-联合营销费 -------------------
    if pd.notnull(r["联合营销费"]):
        new_rows.append({**r, "类型": "扣点-联合营销费", "金额": abs(r["联合营销费"])})

    # ------------------- 规则3：扣点-技术服务费（基础服务费） -------------------
    if pd.notnull(r["其中:基础服务费金额"]):
        new_rows.append({**r, "类型": "扣点-技术服务费", "金额": -r["其中:基础服务费金额"]})

    # ------------------- 规则4：扣点-技术服务费（履约服务费） -------------------
    if pd.notnull(r["其中:履约服务费金额"]):
        new_rows.append({**r, "类型": "扣点-技术服务费", "金额": -r["其中:履约服务费金额"]})

    # ------------------- 规则5：扣点-物流费（消费者邮费补贴） -------------------
    if pd.notnull(r["消费者邮费补贴金额"]):
        new_rows.append({**r, "类型": "扣点-物流费", "金额": abs(r["消费者邮费补贴金额"])})

# ===================== 合并结果 =====================
dw_1_sc = pd.DataFrame(new_rows)

In [ ]:
dw_1_sc[dw_1_sc['类型'].isin(['扣点-技术服务费'])].金额.sum()

#### 未知类型

In [ ]:
dw_1_wz = dw_1_sc[dw_1_sc['类型'] == '未知']
dw_1_wz

#### 店铺表格

In [ ]:
dw_1_sc['店铺'] = dw_1_sc['店铺'].astype(int)

In [ ]:
dw_2 = pd.merge(dw_1_sc, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
dw_2 = dw_2.copy()
for field in ['合伙人', '品牌']:
    dw_2 = optimize_field_by_date(dw_2, field)

#### 主体

In [ ]:
dw_2['店铺ID'] = dw_2['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
dw_3 = pd.merge(dw_2, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
dw_3['发生时间'] = pd.to_datetime(dw_3['发生时间'])
dw_3['新主体'] = dw_3.apply(get_new_subject, axis=1)

In [ ]:
dw_4 = pd.merge(dw_3, zt2, left_on = '新主体', right_on = '主体', how = 'left')
dw_4['身份'] = np.where(dw_4['身份'].isna(), '小规模纳税人', dw_4['身份'])

#### 处理后

In [ ]:
dw_4['来源文件'] = "得物_资金账单"

In [ ]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
dw_5 = dw_4.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型',"来源文件"])[['金额']].sum().round(2).reset_index()

# 重命名列名
dw_5 = dw_5.rename(columns={'店铺名称（聚水潭）':'店铺名称', '调整金额':'金额'})

In [ ]:
dw_5 = dw_5[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '金额',"来源文件"]].copy()

In [ ]:
dw_5

### 得物_退货退款订单

In [ ]:
dw_th = pd.read_sql(f'select * from `得物_退货退款订单`', conn)

In [ ]:
# 一行调用，直接处理整列
dw_th['退货完成时间'] = convert_date_simple(dw_th['退货完成时间'])

In [ ]:
dw_th.rename(columns = {'退货完成时间':'发生时间'}, inplace = True)

In [ ]:
dw_th_1 = dw_th.copy()

In [ ]:
import pandas as pd

# 定义需要转 float 的所有列名列表
float_cols = [
    '商品金额', '其中:基础服务费金额', '其中:履约服务费金额',
]

# 批量转换类型（安全写法，自动跳过不存在的列）
for col in float_cols:
    if col in dw_th_1.columns:
        dw_th_1[col] = pd.to_numeric(dw_th_1[col], errors='coerce')

In [ ]:
import pandas as pd
import numpy as np

# 1. 初始化（完全对齐你给的新版格式）
dw_th_1_cl = dw_th_1.copy()
dw_th_1_cl["类型"] = "未知"
dw_th_1_cl["金额"] = 0.0
new_rows = []

# 统一简写
df = dw_th_1_cl

# 2. 遍历每一行，按规则生成所有类型（逐条规则对应拆分）
for idx, row in df.iterrows():
    r = row.to_dict()

    # ------------------- 规则1：销售额-商品金额 -------------------
    if pd.notnull(r["商品金额"]):
        new_rows.append({**r, "类型": "退款-结算退款", "金额": abs(r["商品金额"])})

    # ------------------- 规则3：扣点-技术服务费（基础服务费） -------------------
    if pd.notnull(r["其中:基础服务费金额"]):
        new_rows.append({**r, "类型": "扣点-技术服务费", "金额": -r["其中:基础服务费金额"]})

    # ------------------- 规则4：扣点-技术服务费（履约服务费） -------------------
    if pd.notnull(r["其中:履约服务费金额"]):
        new_rows.append({**r, "类型": "扣点-技术服务费", "金额": -r["其中:履约服务费金额"]})

# ===================== 合并结果 =====================
dw_th_1_sc = pd.DataFrame(new_rows)

In [ ]:
dw_th_1_sc

#### 未知类型

In [ ]:
dw_th_1_wz = dw_th_1_sc[dw_th_1_sc['类型'] == '未知']
dw_th_1_wz

#### 店铺表格

In [ ]:
dw_th_1_sc['店铺'] = dw_th_1_sc['店铺'].astype(int)

In [ ]:
dw_th_2 = pd.merge(dw_th_1_sc, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
dw_th_2 = dw_th_2.copy()
for field in ['合伙人', '品牌']:
    dw_th_2 = optimize_field_by_date(dw_th_2, field)

#### 主体

In [ ]:
dw_th_2['店铺ID'] = dw_th_2['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
dw_th_3 = pd.merge(dw_th_2, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
dw_th_3['发生时间'] = pd.to_datetime(dw_th_3['发生时间'])
dw_th_3['新主体'] = dw_th_3.apply(get_new_subject, axis=1)

In [ ]:
dw_th_4 = pd.merge(dw_th_3, zt2, left_on = '新主体', right_on = '主体', how = 'left')
dw_th_4['身份'] = np.where(dw_th_4['身份'].isna(), '小规模纳税人', dw_th_4['身份'])

In [ ]:
dw_th_4

#### 处理后

In [ ]:
dw_th_4['来源文件'] = "得物_退货退款订单"

In [ ]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
dw_th_5 = dw_th_4.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型',"来源文件"])[['金额']].sum().round(2).reset_index()

# 重命名列名
dw_th_5 = dw_th_5.rename(columns={'店铺名称（聚水潭）':'店铺名称', '调整金额':'金额'})

In [ ]:
dw_th_5 = dw_th_5[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '金额',"来源文件"]].copy()

In [ ]:
dw_th_5

### 得物_扣减其他费用明细

In [ ]:
dw_kj = pd.read_sql(f'select * from `得物_扣减其他费用明细`', conn)

In [ ]:
# 一行调用，直接处理整列
dw_kj['费用生成时间'] = convert_date_simple(dw_kj['费用生成时间'])

In [ ]:
dw_kj.rename(columns = {'费用生成时间':'发生时间'}, inplace = True)

In [ ]:
dw_kj_1 = dw_kj.copy()

In [ ]:
import pandas as pd

# 定义需要转 float 的所有列名列表
float_cols = [
    '本次偿还金额',
]

# 批量转换类型（安全写法，自动跳过不存在的列）
for col in float_cols:
    if col in dw_kj_1.columns:
        dw_kj_1[col] = pd.to_numeric(dw_kj_1[col], errors='coerce')

In [ ]:
import pandas as pd
import numpy as np

# 1. 初始化（完全对齐你给的新版格式）
dw_kj_1_cl = dw_kj_1.copy()
dw_kj_1_cl["类型"] = "未知"
dw_kj_1_cl["金额"] = 0.0
new_rows = []

# 统一简写
df = dw_kj_1_cl

# 2. 遍历每一行，按规则生成所有类型（逐条规则对应拆分）
for idx, row in df.iterrows():
    r = row.to_dict()

    # ------------------- 规则1：销售额-商品金额 -------------------
    if pd.notnull(r["本次偿还金额"]):
        new_rows.append({**r, "类型": "扣款-售后赔付款", "金额": abs(r["本次偿还金额"])})

# ===================== 合并结果 =====================
dw_kj_1_cl = pd.DataFrame(new_rows)

In [ ]:
dw_kj_1_cl

#### 未知类型

In [ ]:
dw_kj_1_wz = dw_kj_1_cl[dw_kj_1_cl['类型'] == '未知']
dw_kj_1_wz

#### 店铺表格

In [ ]:
dw_kj_1_cl['店铺'] = dw_kj_1_cl['店铺'].astype(int)

In [ ]:
dw_kj_2 = pd.merge(dw_kj_1_cl, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
dw_kj_2 = dw_kj_2.copy()
for field in ['合伙人', '品牌']:
    dw_kj_2 = optimize_field_by_date(dw_kj_2, field)

#### 主体

In [ ]:
dw_kj_2['店铺ID'] = dw_kj_2['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
dw_kj_3 = pd.merge(dw_kj_2, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
dw_kj_3['发生时间'] = pd.to_datetime(dw_kj_3['发生时间'])
dw_kj_3['新主体'] = dw_kj_3.apply(get_new_subject, axis=1)

In [ ]:
dw_kj_4 = pd.merge(dw_kj_3, zt2, left_on = '新主体', right_on = '主体', how = 'left')
dw_kj_4['身份'] = np.where(dw_kj_4['身份'].isna(), '小规模纳税人', dw_kj_4['身份'])

In [ ]:
dw_kj_4

#### 处理后

In [ ]:
dw_kj_4['来源文件'] = "得物_扣减其他费用明细"

In [ ]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
dw_kj_5 = dw_kj_4.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型',"来源文件"])[['金额']].sum().round(2).reset_index()

# 重命名列名
dw_kj_5 = dw_kj_5.rename(columns={'店铺名称（聚水潭）':'店铺名称', '调整金额':'金额'})

In [ ]:
dw_kj_5 = dw_kj_5[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '金额',"来源文件"]].copy()

In [ ]:
dw_kj_5

### 得物_账单总览

In [ ]:
dw_zl = pd.read_sql(f'select * from [得物_账单总览]', conn2)

In [ ]:
dw_zl_1 = dw_zl.copy()

In [ ]:
import pandas as pd

# 定义需要转 float 的所有列名列表
float_cols = [
    '应结金额',
]

# 批量转换类型（安全写法，自动跳过不存在的列）
for col in float_cols:
    if col in dw_zl_1.columns:
        dw_zl_1[col] = pd.to_numeric(dw_zl_1[col], errors='coerce')

In [ ]:
import pandas as pd
import numpy as np

# 1. 初始化（完全对齐你给的新版格式）
dw_zl_1_cl = dw_zl_1.copy()
dw_zl_1_cl["类型"] = "未知"
dw_zl_1_cl["金额"] = 0.0
new_rows = []

# 统一简写
df = dw_zl_1_cl

# 2. 遍历每一行，按规则生成所有类型（逐条规则对应拆分）
for idx, row in df.iterrows():
    r = row.to_dict()

    # ------------------- 规则1：应结金额 -------------------
    if pd.notnull(r["应结金额"]):
        new_rows.append({**r, "类型": "应结金额", "金额": abs(r["应结金额"])})

# ===================== 合并结果 =====================
dw_zl_1_sc = pd.DataFrame(new_rows)

In [ ]:
dw_zl_1_sc

#### 未知类型

In [ ]:
dw_zl_1_wz = dw_zl_1_sc[dw_zl_1_sc['类型'] == '未知']
dw_zl_1_wz

#### 店铺表格

In [ ]:
dw_zl_1_sc['店铺'] = dw_zl_1_sc['店铺'].astype(int)

In [ ]:
dw_zl_2 = pd.merge(dw_zl_1_sc, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

#### 主体

In [ ]:
dw_zl_2['店铺ID'] = dw_zl_2['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
dw_zl_3 = pd.merge(dw_zl_2, zt3, on = '店铺ID', how = 'left')

In [ ]:
dw_zl_4 = pd.merge(dw_zl_3, zt2, left_on = '主体', right_on = '主体', how = 'left')
dw_zl_4['身份'] = np.where(dw_zl_4['身份'].isna(), '小规模纳税人', dw_zl_4['身份'])

#### 处理后

In [ ]:
dw_zl_4['来源文件'] = "得物_账单总览"

In [ ]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
dw_zl_5 = dw_zl_4.groupby(['店铺名称（聚水潭）', '店铺ID', '主体', '身份', '平台', '合伙人', '品牌', '类型',"来源文件"])[['金额']].sum().round(2).reset_index()

# 重命名列名
dw_zl_5 = dw_zl_5.rename(columns={'店铺名称（聚水潭）':'店铺名称', '调整金额':'金额', '主体': '新主体'})

In [ ]:
dw_zl_5 = dw_zl_5[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '金额',"来源文件"]].copy()

In [ ]:
dw_zl_5

### 处理后

In [ ]:
dw_wz = pd.concat([dw_1_wz, dw_th_1_wz, dw_kj_1_wz, dw_zl_1_wz], ignore_index=True)

In [ ]:
dw_wz

### 汇总

In [ ]:
dw_hb = pd.concat([dw_5, dw_th_5, dw_kj_5, dw_zl_5], ignore_index = True)

In [ ]:
dw_hb_1 = dw_hb.groupby(['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])['金额'].sum().reset_index()
dw_hb_1

In [ ]:
# 定义需要保存的DataFrame和对应的文件路径
data_to_save = [

    (dw_wz, fr'../../结果/未知/得物未知类型.csv'),
    (dw_hb_1, fr'../../结果/2026年{int(month)}月/得物/得物2.0(待处理).csv'),
]
# 分离DataFrame和文件路径列表
dataframes = [item[0] for item in data_to_save]
file_paths = [item[1] for item in data_to_save]
# 调用函数保存数据
try:
    save_dataframes(dataframes, file_paths)
except Exception as e:
    print(f"保存数据时出错: {e}")

## 快手

### 快手结算账单

In [ ]:
ks_js = pd.read_sql(f'select * from [快手_结算账单(测试)]', conn2)

In [ ]:
ks = ks_js.copy()

In [ ]:
# 一行调用，直接处理整列
ks['实际结算时间'] = convert_date_simple(ks['实际结算时间'])

In [ ]:
ks.rename(columns = {'实际结算时间':'发生时间'}, inplace = True)

In [ ]:
import pandas as pd

# 定义需要转 float 的所有列名列表
float_cols = [
    '订单实付元', '政府补贴', '支付营销补贴', '平台补贴', '商家补贴元',
    '达人补贴', '合计收入元', '订单退款元', '支付营销回退（元）',
    '技术服务费元', '预售增收技术服务费（元）', '达人佣金元',
    '团长佣金元', '快赚客佣金元', '服务商佣金元', '其他收费',
    '合计支出元', '实际结算金额元'
]

# 批量转换类型（安全写法，自动跳过不存在的列）
for col in float_cols:
    if col in ks.columns:
        ks[col] = pd.to_numeric(ks[col], errors='coerce')

In [ ]:
import pandas as pd
import numpy as np

# 初始化类型列为未知
ks_cl = ks.copy()
ks_cl["类型"] = "未知"
ks_cl["金额"] = 0.0
new_rows = []

# 统一简写
df = ks_cl.copy()

# 2. 遍历每一行，按规则生成所有类型（逐条规则对应拆分）
for idx, row in df.iterrows():
    r = row.to_dict()

    # ------------------- 规则1：销售额-商品金额 -------------------
    if pd.notnull(r["订单实付元"]):
        new_rows.append({**r, "类型": "销售额-订单实付", "金额": abs(r["订单实付元"])})
    if pd.notnull(r["支付营销补贴"]):
        new_rows.append({**r, "类型": "销售额-营销补贴", "金额": abs(r["支付营销补贴"])})
    if pd.notnull(r["政府补贴"]):
        new_rows.append({**r, "类型": "销售额-政府补贴", "金额": abs(r["政府补贴"])})
    if pd.notnull(r["平台补贴"]):
        new_rows.append({**r, "类型": "销售额-平台补贴", "金额": abs(r["平台补贴"])})
    if pd.notnull(r["商家补贴元"]):
        new_rows.append({**r, "类型": "销售额-商家补贴", "金额": abs(r["商家补贴元"])})
    if pd.notnull(r["达人补贴"]):
        new_rows.append({**r, "类型": "销售额-达人补贴", "金额": abs(r["达人补贴"])})

    # ------------------- 规则2：退款-结算后退款 -------------------
    if pd.notnull(r["订单退款元"]):
        new_rows.append({**r, "类型": "退款-结算后退款", "金额": abs(r["订单退款元"])})

    # ------------------- 规则3：扣点-技术服务费 -------------------
    if pd.notnull(r["技术服务费元"]):
        new_rows.append({**r, "类型": "扣点-技术服务费", "金额": abs(r["技术服务费元"])})

    # ------------------- 规则4：扣点-技术服务费（履约服务费） -------------------
    if pd.notnull(r["达人佣金元"]):
        new_rows.append({**r, "类型": "扣款-达人佣金", "金额": abs(r["达人佣金元"])})

    # ------------------- 规则5：扣点-物流费（消费者邮费补贴） -------------------
    if pd.notnull(r["团长佣金元"]):
        new_rows.append({**r, "类型": "扣款-团长佣金", "金额": abs(r["团长佣金元"])})

    if pd.notnull(r["快赚客佣金元"]):
        new_rows.append({**r, "类型": "扣款-快赚客佣金", "金额": abs(r["快赚客佣金元"])})

    if pd.notnull(r["其他收费"]):
        new_rows.append({**r, "类型": "扣点-分销信息服务费", "金额": abs(r["其他收费"])})

    # ------------------- 【关键修正】结算账单-合计支出判断 -------------------
    # 把所有可能为NaN的字段用fillna(0)处理，避免加法结果为NaN
    sum_expense = (
        abs(r["订单退款元"]) if pd.notnull(r["订单退款元"]) else 0
        + abs(r["技术服务费元"]) if pd.notnull(r["技术服务费元"]) else 0
        + abs(r["达人佣金元"]) if pd.notnull(r["达人佣金元"]) else 0
        + abs(r["团长佣金元"]) if pd.notnull(r["团长佣金元"]) else 0
        + abs(r["快赚客佣金元"]) if pd.notnull(r["快赚客佣金元"]) else 0
        + abs(r["其他收费"]) if pd.notnull(r["其他收费"]) else 0
    )

    # 或者更简洁的写法（推荐）
    sum_expense = (
        np.nan_to_num(abs(r["订单退款元"]))
        + np.nan_to_num(abs(r["技术服务费元"]))
        + np.nan_to_num(abs(r["达人佣金元"]))
        + np.nan_to_num(abs(r["团长佣金元"]))
        + np.nan_to_num(abs(r["快赚客佣金元"]))
        + np.nan_to_num(abs(r["其他收费"]))
    )

    # 再和合计支出元比较（同样处理NaN）
    total_expense = np.nan_to_num(r["合计支出元"])
    if abs(total_expense - sum_expense) < 1e-6:  # 浮点数精度问题，用差值判断
        new_rows.append({**r, "类型": "结算账单-合计支出", "金额": total_expense})
    else:
        new_rows.append({**r, "类型": "异常", "金额": total_expense})  # 异常行也保留原始金额，方便排查

    # ------------------- 实际结算金额 -------------------
    if pd.notnull(r["实际结算金额元"]):
        new_rows.append({**r, "类型": "结算账单-货款结算", "金额": abs(r["实际结算金额元"])})

# ===================== 合并结果 =====================
ks_1 = pd.DataFrame(new_rows)

In [ ]:
ks_1[(ks_1['类型'] == '结算账单-合计支出') & (ks_1['店铺'] == '16683668')].金额.sum()

#### 店铺表格

In [ ]:
ks_1['店铺'] = ks_1['店铺'].astype(int)

In [ ]:
# 6. 合并 + 长表转换
ks_1_1 = pd.merge(ks_1, dp_bg, left_on='店铺', right_on='店铺ID', how='left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
ks_1_1 = ks_1_1.copy()
for field in ['合伙人', '品牌']:
    ks_1_1 = optimize_field_by_date(ks_1_1, field)

#### 主体

In [ ]:
ks_1_1['店铺ID'] = ks_1_1['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
ks_1_2 = pd.merge(ks_1_1, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
ks_1_2['发生时间'] = pd.to_datetime(ks_1_2['发生时间'])
ks_1_2['新主体'] = ks_1_2.apply(get_new_subject, axis=1)

In [ ]:
ks_1_3 = pd.merge(ks_1_2, zt2, left_on = '新主体', right_on = '主体', how = 'left')
ks_1_3['身份'] = np.where(ks_1_3['身份'].isna(), '小规模纳税人', ks_1_3['身份'])

In [ ]:
ks_1_3 = ks_1_3[~(ks_1_3.发生时间 == '')]

In [ ]:
ks_2_wz = ks_1_3[ks_1_3['类型'] == '未知']
ks_2_wz

In [ ]:
# ===================== 只保留有效数据，清理0金额 =====================
ks_2 = ks_1_3[~((ks_1_3['类型']=="未知") & (ks_1_3['金额']==0))].copy().reset_index(drop=True)

In [ ]:
ks_2['来源文件'] = '快手_结算账单'

In [ ]:
ks_3 = ks_2.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])['金额'].sum().reset_index().rename(columns = {'店铺名称（聚水潭）':'店铺名称'})

In [ ]:
ks_3

### 快手资金账单

In [ ]:
ks_zj = pd.read_sql(f'select * from [快手_资金账单(测试)]', conn2)
ks_zj['入账时间'] = convert_date_simple(ks_zj['入账时间'])

In [ ]:
ks_zj.rename(columns = {'入账时间':'发生时间'}, inplace = True)

In [ ]:
ks_zj['金额'] = ks_zj['发生额（元）'].astype(float)

#### 区分类型

In [ ]:
import pandas as pd
import numpy as np

# ==================== 数据初始化 ====================
ks_zj_cl = ks_zj.copy()
ks_zj_cl["类型"] = "未知"
ks_zj_cl["规则金额"] = 0.0  # 规则计算金额，不污染原数据

# 别名简化（向量化，只取一次）
df = ks_zj_cl
direction = df['账务方向']
busi_type = df['业务类型']
desc = df['描述']
amount = df['金额']

# 统一预计算 lambda（只算一次，性能更好）
desc_has = lambda s: desc.str.contains(s, na=False)
dir_is = lambda s: direction == s
type_is = lambda s: busi_type == s
type_in = lambda lst: busi_type.isin(lst)

# ==================== 规则：一行一条，纯向量化 ====================
# 1. 资金账单-交易结算
df.loc[(dir_is('收')) & (type_is('货款结算')) & desc_has('快手小店交易结算'), '类型'] = "资金账单-交易结算"

# 2. 资金账单-退款补结算
df.loc[(dir_is('收')) & (type_is('货款结算')) & desc_has('退款补结算'), '类型'] = "资金账单-退款补结算"

# 3. 扣款-保证金-集运扣款
cond1 = (dir_is('收')) & type_in(['扣减前临时解冻','资金解冻','资金扣减回退']) & desc_has('集运扣款|集运扣款解冻')
cond2 = (dir_is('支')) & (type_is('资金冻结')) & desc_has('集运扣款冻结')
df.loc[cond1 | cond2, '类型'] = "扣款-保证金-集运扣款"

# 4. 扣款-保证金-小店退款
cond1 = (dir_is('收')) & (type_is('扣减前临时解冻')) & desc_has('快手小店退款')
cond2 = (dir_is('支')) & (type_is('资金冻结')) & desc_has('退款-快手电商')
df.loc[cond1 | cond2, '类型'] = "扣款-保证金-小店退款"

# 5. 扣款-达人佣金
df.loc[(dir_is('收')) & (type_is('佣金/技术服务费返还')) & desc_has('分账追回-超售后期-MCN机构'), '类型'] = "扣款-达人佣金"

# 6. 扣款-团长佣金
df.loc[(dir_is('收')) & (type_is('佣金/技术服务费返还')) & desc_has('分账追回-超售后期-cps团长三方分佣户'), '类型'] = "扣款-团长佣金"

# 7. 扣款-小额打款
df.loc[(dir_is('支')) & (type_is('资金扣减')) & desc_has('小额打款'), '类型'] = "扣款-小额打款"

# 8. 扣款-晚发立赔
df.loc[(dir_is('支')) & (type_is('资金扣减')) & desc_has('晚发立赔-售后商责/延迟发货等违约金扣款'), '类型'] = "扣款-晚发立赔"

# 9. 扣款-晚揽立赔
df.loc[(dir_is('支')) & (type_is('资金扣减')) & desc_has('晚揽立赔平台追回'), '类型'] = "扣款-晚揽立赔"

# 10. 扣款-商责退运费
df.loc[(dir_is('支')) & (type_is('资金扣减')) & desc_has('商责退运费-垫资追回|快手商品赔付-现金补偿'), '类型'] = "扣款-商责退运费"

# 11. 扣点-技术服务费
df.loc[(dir_is('收')) & (type_is('佣金/技术服务费返还')) & desc_has('分账追回-超售后期-平台服务费'), '类型'] = "扣点-技术服务费"

# 12. 扣点-分销信息服务费
df.loc[(dir_is('收')) & (type_is('佣金/技术服务费返还')) & desc_has('分账追回-超售后期-泛商城服务费'), '类型'] = "扣点-分销信息服务费"

# 13. 扣点-物流费
df.loc[(dir_is('支')) & (type_is('资金扣减')) & desc_has('集运扣款'), '类型'] = "扣点-物流费"

# 14. 扣点-运费险
df.loc[(dir_is('支')) & (type_is('资金转账')) & desc_has('安心钱包转到退货补运费'), '类型'] = "扣点-运费险"

# 15. 退款-快手小店退款
df.loc[(dir_is('支')) & (type_is('资金扣减')) & desc_has('快手小店退款'), '类型'] = "退款-快手小店退款"

# 16. 退款-结算后退款
df.loc[(dir_is('支')) & (type_is('结算后退款')) & desc_has('退现金金额'), '类型'] = "退款-结算后退款"

# 17. 提现
df.loc[(dir_is('支')) & (type_is('余额提现')), '类型'] = "提现"

# 18. 磁力金牛-充值
df.loc[(dir_is('支')) & (type_is('资金扣减')) & desc_has('磁力金牛免密支付'), '类型'] = "磁力金牛-充值"


In [ ]:
# ==================== 统一批量计算规则金额（模板风格） ====================
# 定义类型分组
ABS_TYPES = {
    "资金账单-交易结算", "资金账单-退款补结算",
    "扣款-小额打款", "扣款-晚发立赔", "扣款-晚揽立赔", "扣款-商责退运费",
    "扣点-物流费", "扣点-运费险",
    "退款-快手小店退款", "退款-结算后退款", "提现", "磁力金牛-充值"
}

NEG_TYPES = {
    "扣款-保证金-集运扣款", "扣款-保证金-小店退款",
    "扣款-达人佣金", "扣款-团长佣金",
    "扣点-技术服务费", "扣点-分销信息服务费"
}

# 批量赋值（性能远优于逐条）
df.loc[df['类型'].isin(ABS_TYPES), '规则金额'] = df.loc[df['类型'].isin(ABS_TYPES), '金额'].abs()
df.loc[df['类型'].isin(NEG_TYPES), '规则金额'] = -df.loc[df['类型'].isin(NEG_TYPES), '金额']

# 最终覆盖金额，输出结果
df['金额'] = df['规则金额']
ks_zj_sc = df.copy()

In [ ]:
ks_zj_sc

#### 店铺表格

In [ ]:
ks_zj_sc['店铺'] = ks_zj_sc['店铺'].astype(int)

In [ ]:
ks_zj_1 = pd.merge(ks_zj_sc, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
ks_zj_1 = ks_zj_1.copy()
for field in ['合伙人', '品牌']:
    ks_zj_1 = optimize_field_by_date(ks_zj_1, field)

#### 主体

In [ ]:
ks_zj_1['店铺ID'] = ks_zj_1['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
ks_zj_2 = pd.merge(ks_zj_1, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
ks_zj_2['发生时间'] = pd.to_datetime(ks_zj_2['发生时间'])
ks_zj_2['新主体'] = ks_zj_2.apply(get_new_subject, axis=1)

In [ ]:
ks_zj_3 = pd.merge(ks_zj_2, zt2, left_on = '新主体', right_on = '主体', how = 'left')
ks_zj_3['身份'] = np.where(ks_zj_3['身份'].isna(), '小规模纳税人', ks_zj_3['身份'])

In [ ]:
ks_zj_4 = ks_zj_3[~(ks_zj_3.发生时间 == '')]

In [ ]:
ks_zj_2_wz = ks_zj_4[ks_zj_4['类型'] == '未知']
ks_zj_2_wz

In [ ]:
# ===================== 只保留有效数据，清理0金额 =====================
ks_zj_5 = ks_zj_4[~((ks_zj_4['类型']=="未知") & (ks_zj_4['金额']==0))].copy().reset_index(drop=True)

In [ ]:
ks_zj_5['来源文件'] = '快手_资金账单'

In [ ]:
ks_zj_6 = ks_zj_5.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])['金额'].sum().reset_index().rename(columns = {'店铺名称（聚水潭）':'店铺名称'})

In [ ]:
# ks_zj[ks_zj['店铺'] == 17300574]

In [ ]:
ks_zj_6

### 处理后

In [ ]:
ks_hb = pd.concat([ks_3, ks_zj_6], ignore_index = True)

In [ ]:
ks_zj_2_wz['来源文件'] = '快手_资金账单'
ks_zj_2_wz.drop(['主体_x', '主体_y'], axis=1, inplace=True)
ks_2_wz['来源文件'] = '快手_结算账单'
ks_2_wz.drop(['主体_x', '主体_y'], axis=1, inplace=True)

In [ ]:
ks_hb_wz = pd.concat([ks_2_wz, ks_zj_2_wz], ignore_index = True)
ks_hb_wz

In [ ]:
ks_hb_1 = ks_hb.groupby(['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])['金额'].sum().reset_index()
ks_hb_1

In [ ]:
# 定义需要保存的DataFrame和对应的文件路径
data_to_save = [

    (ks_hb_wz, fr'../../结果/未知/快手未知类型.csv'),
    (ks_hb_1, fr'../../结果/2026年{int(month)}月/快手/快手2.0(待处理).csv'),
]
# 分离DataFrame和文件路径列表
dataframes = [item[0] for item in data_to_save]
file_paths = [item[1] for item in data_to_save]
# 调用函数保存数据
try:
    save_dataframes(dataframes, file_paths)
except Exception as e:
    print(f"保存数据时出错: {e}")

## 小红书

### 小红书结算账单

In [ ]:
xhs_jszd = pd.read_sql(f'select * from `小红书结算账单`', conn)

In [ ]:
# 一行调用，直接处理整列
xhs_jszd['结算时间'] = convert_date_simple(xhs_jszd['结算时间'])

#### 账单处理

In [ ]:
# 白名单：保留为 object/非数值 的列
keep_as_object_cols = [
    '店铺', '订单号', '售后单号', '下单时间', '结算时间',
    '交易类型', '结算账户', '跨境税代缴', '备注'
]

# 自动识别所有非白名单列，转为数值类型
# 1. 生成要转换的列列表（不在白名单中的列）
convert_to_numeric_cols = [col for col in xhs_jszd.columns if col not in keep_as_object_cols]

# 2. 定义一个可复用的清洗转换函数
def clean_and_convert_to_numeric(series: pd.Series) -> pd.Series:
    # 先处理空字符串、空格、制表符、换行符，统一替换为 NaN
    series = series.astype(str)  # 先统一转为字符串，避免对非字符串类型报错
    series = series.replace(['', ' ', '\t', '\n', ''], pd.NA)
    # 转为数值，无法转换的会变成 NaN（比如带非数字字符的脏数据）
    return pd.to_numeric(series, errors='coerce')

# 3. 批量应用到所有目标列
xhs_jszd[convert_to_numeric_cols] = xhs_jszd[convert_to_numeric_cols].apply(clean_and_convert_to_numeric)

#### 店铺表格

In [ ]:
xhs_jszd.rename(columns = {'结算时间':'发生时间'}, inplace = True)

In [ ]:
xhs_jszd['店铺'] = xhs_jszd['店铺'].apply(float)
dp_bg_pt=dp_bg.copy()
dp_bg_pt['店铺名称'] = dp_bg_pt['店铺ID'].apply(float)
xhs_jszd_dp = pd.merge(xhs_jszd, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
xhs_jszd_dp = xhs_jszd_dp.copy()
for field in ['合伙人', '品牌']:
    xhs_jszd_dp = optimize_field_by_date(xhs_jszd_dp, field)

#### 账单转置

In [ ]:
# ====================== 固定不转置的列（白名单）======================
id_cols = [
    '店铺名称（聚水潭）', '店铺ID', '平台', '合伙人', '品牌', '状态', '日期',
    '店铺', '订单号', '售后单号', '下单时间', '发生时间',
    '交易类型', '结算账户', '跨境税代缴', '备注'
]
# ====================== 自动获取：除固定列外，全部转置 ======================
try:
    # 自动获取所有需要转置的列（不在白名单里的列）
    value_cols = [col for col in xhs_jszd_dp.columns if col not in id_cols]
    # 执行 melt 转置（自动适配所有新增列）
    xhs_jszd_1 = xhs_jszd_dp.melt(
        id_vars=id_cols,
        value_vars=value_cols,  # 自动转置所有非固定列
        var_name='类型1',
        value_name='金额'
    )
    # 规整索引
    xhs_jszd_2 = xhs_jszd_1.reset_index(drop=True)
    print("✅ 转置完成，共生成", len(xhs_jszd_2), "行数据")

except KeyError as e:
    print(f"❌ 缺少固定列：{e}，请检查列名")
except Exception as e:
    print(f"❌ 转置失败：{str(e)}")

#### 销售额-商品实付

In [ ]:
xhs_jszd_xse_spsf= ((xhs_jszd_2['交易类型'].str.contains('结算入账', na=False))&
             (
             (xhs_jszd_2['类型1'].str.contains('商品实付_实退', na=False)))
             )
xhs_jszd_2.loc[xhs_jszd_xse_spsf, '类型'] = '销售额-商品实付'
xhs_jszd_2.loc[xhs_jszd_xse_spsf, '金额'] = abs(xhs_jszd_2['金额'])

#### 退款-结算后退款

In [ ]:
xhs_jszd_tk_jshtk= ((xhs_jszd_2['交易类型'].str.contains('退款', na=False))&
             (
             (xhs_jszd_2['类型1'].str.contains('商品实付_实退', na=False)))
             )
xhs_jszd_2.loc[xhs_jszd_tk_jshtk, '类型'] = '退款-结算后退款'
xhs_jszd_2.loc[xhs_jszd_tk_jshtk, '金额'] = abs(xhs_jszd_2['金额'])

#### 扣款-佣金

In [ ]:
xhs_jszd_kd_yj= (xhs_jszd_2['交易类型'].str.contains('结算入账|退款', na=False)) & (xhs_jszd_2['类型1'].str.contains('佣金', na=False))
xhs_jszd_2.loc[xhs_jszd_kd_yj, '类型'] = '扣款-佣金'
xhs_jszd_2.loc[xhs_jszd_kd_yj, '金额'] = -xhs_jszd_2['金额']

#### 结算账单-货款结算

In [ ]:
xhs_jszd_HKJS= (xhs_jszd_2['类型1'].str.contains('动账金额', na=False))
xhs_jszd_2.loc[xhs_jszd_HKJS, '类型'] = '结算账单-货款结算'

#### 不计数

In [ ]:
xhs_jszd_bqs = (xhs_jszd_2['类型1'].str.contains('计佣基数', na=False))
xhs_jszd_2.loc[xhs_jszd_bqs, '类型'] = '不取数'
xhs_jszd_2['类型'] = np.where((xhs_jszd_2['类型'].isnull()),'未知',xhs_jszd_2['类型'])
xhs_jszd_nobqs = xhs_jszd_2[xhs_jszd_2['类型'] !='不取数']
xhs_jszd_nobqs

#### 未知

In [ ]:
xhs_jszd_wz = xhs_jszd_nobqs[(xhs_jszd_nobqs['类型']=='未知') & (xhs_jszd_nobqs['金额'] != 0)]
xhs_jszd_wz

In [ ]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
xhs_jszd_nobqs['来源文件'] = '小红书_结算账单'
xhs_jszd_3 = xhs_jszd_nobqs.groupby(['店铺名称（聚水潭）', '店铺ID', '发生时间', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额']].sum().round(2).reset_index()
# 重命名列名
xhs_jszd_3 = xhs_jszd_3.rename(columns={'店铺名称（聚水潭）':'店铺名称', '调整金额':'金额'})

In [ ]:
xhs_jszd_3

### 小红书资金账单

In [ ]:
xhs_zjzd = pd.read_sql(f'select * from `小红书资金账单`', conn)

In [ ]:
# 一行调用，直接处理整列
xhs_zjzd['创建时间'] = convert_date_simple(xhs_zjzd['创建时间'])

In [ ]:
xhs_zjzd.rename(columns = {'创建时间':'发生时间'}, inplace = True)

In [ ]:
xhs_zjzd['收入（元）'] = xhs_zjzd['收入（元）'].replace('', 0).replace(' ', 0).replace('\t', 0).replace('\n', 0).astype(float)
xhs_zjzd['支出（元）'] = xhs_zjzd['支出（元）'].replace('', 0).replace(' ', 0).replace('\t', 0).replace('\n', 0).astype(float)

##### 资金账单-交易结算

In [ ]:
xhs_zjzd

In [ ]:
xhs_zjzd_jyjs = ((xhs_zjzd['交易类型描述'].str.contains('结算入账', na=False))&
             (
             (xhs_zjzd['备注'].str.contains('结算入账-订单号', na=False)))
             )
xhs_zjzd.loc[xhs_zjzd_jyjs, '类型'] = '资金账单-交易结算'
xhs_zjzd.loc[xhs_zjzd_jyjs, '金额'] = abs(xhs_zjzd['收入（元）'])

##### 扣点-运费宝

In [ ]:
xhs_zjzd_kd_yfb = ((xhs_zjzd['交易类型描述'].str.contains('结算入账', na=False))&
             (
             (xhs_zjzd['备注'].str.contains('运费宝', na=False)))
             )
xhs_zjzd.loc[xhs_zjzd_kd_yfb, '类型'] = '扣点-运费宝'
xhs_zjzd.loc[xhs_zjzd_kd_yfb, '金额'] = abs(xhs_zjzd['支出（元）'])

#### 未知

In [ ]:
xhs_zjzd['类型'] = np.where((xhs_zjzd['类型'].isnull()),'未知',xhs_zjzd['类型'])

In [ ]:
xhs_zjzd['店铺'] = xhs_zjzd['店铺'].astype(int)

#### 店铺表格

In [ ]:
xhs_zjzd_1 = pd.merge(xhs_zjzd, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
# 主执行逻辑（仅2行核心调用）
xhs_zjzd_1 = xhs_zjzd_1.copy()
for field in ['合伙人', '品牌']:
    xhs_zjzd_1 = optimize_field_by_date(xhs_zjzd_1, field)

#### 未知类型

In [ ]:
xhs_zjzd_wz = xhs_zjzd_1[xhs_zjzd_1['类型'] == '未知']
xhs_zjzd_wz

#### 资金账单核算

In [ ]:
xhs_jszd_dp_hs=xhs_jszd_dp[['店铺ID','订单号','动账金额']]

In [ ]:
xhs_zjzd_hs = pd.merge(xhs_zjzd_1, xhs_jszd_dp_hs, left_on =[ '店铺ID','业务单号'], right_on =[ '店铺ID','订单号'], how = 'left')

In [ ]:
xhs_zjzd_hs['核算'] = np.where((xhs_zjzd_hs['金额']==xhs_zjzd_hs['动账金额']),'等于结算账单','不等于结算账单')

In [ ]:
xhs_zjzd_hs['来源文件'] = '小红书_资金账单'

In [ ]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
xhs_zjzd_1 = xhs_zjzd_hs.groupby(['店铺名称（聚水潭）', '店铺ID', '发生时间', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额']].sum().round(2).reset_index()
# 重命名列名
xhs_zjzd_2 = xhs_zjzd_1.rename(columns={'店铺名称（聚水潭）':'店铺名称'})

### 小红书汇总

In [ ]:
xhs = pd.concat([xhs_jszd_3,xhs_zjzd_2], axis=0, ignore_index=True)

In [ ]:
xhs_zjzd_wz['来源文件'] = '小红书_资金账单'
xhs_zjzd_wz

In [ ]:
xhs_jszd_wz['来源文件'] = '小红书_结算账单'
xhs_jszd_wz

In [ ]:
xhs_wz = pd.concat([xhs_jszd_wz, xhs_zjzd_wz], ignore_index=True)

### 主体

In [ ]:
xhs['店铺ID'] = xhs['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
xhs_1 = pd.merge(xhs, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
xhs_1['发生时间'] = pd.to_datetime(xhs_1['发生时间'])
xhs_1['新主体'] = xhs_1.apply(get_new_subject, axis=1)

In [ ]:
xhs_1 = pd.merge(xhs_1, zt2, left_on = '新主体', right_on = '主体', how = 'left')
xhs_1['身份'] = np.where(xhs_1['身份'].isna(), '小规模纳税人', xhs_1['身份'])

In [ ]:
xhs_2 = xhs_1[~(xhs_1.发生时间 == '')]

In [ ]:
xhs_2_1 = xhs_2[~((xhs_2.类型 == '未知') & (xhs_2.金额 == 0))]

In [ ]:
xhs_3 = xhs_2_1.groupby(['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额']].sum().round(2).reset_index()

In [ ]:
xhs_3

### 小红书数据保存

In [ ]:
# 定义需要保存的DataFrame和对应的文件路径
data_to_save = [

    (xhs_wz, fr'../../结果/未知/小红书未知类型.csv'),
    (xhs_3, fr'../../结果/2026年{int(month)}月/小红书/小红书2.0(待处理).csv'),
]
# 分离DataFrame和文件路径列表
dataframes = [item[0] for item in data_to_save]
file_paths = [item[1] for item in data_to_save]
# 调用函数保存数据
try:
    save_dataframes(dataframes, file_paths)
except Exception as e:
    print(f"保存数据时出错: {e}")

## 微信小店

In [ ]:
wx_xd = pd.read_sql(f'select * from `微信小店资金帐单`', conn)

In [ ]:
# 一行调用，直接处理整列
wx_xd['记账时间'] = convert_date_simple(wx_xd['记账时间'])

In [ ]:
wx_xd.rename(columns = {'记账时间':'发生时间'}, inplace = True)

In [ ]:
import pandas as pd

# 定义需要转 float 的所有列名列表
float_cols = [
    '收支金额'
]

# 批量转换类型（安全写法，自动跳过不存在的列）
for col in float_cols:
    if col in wx_xd.columns:
        wx_xd[col] = pd.to_numeric(wx_xd[col], errors='coerce')

In [ ]:
import pandas as pd
import numpy as np

# 初始化
wx_xd['类型'] = '未知'
wx_xd['金额'] = 0.0
wx_xd_1 = wx_xd.copy
# 别名简写
df = wx_xd_1.copy()
in_out = df['收支类型']
move_type = df['动帐类型']
money = df['收支金额']

# 统一预计算 lambda
in_has = lambda s: in_out.str.contains(s, na=False)
move_has = lambda s: move_type.str.contains(s, na=False)

# ===================== 规则（一行一条，纯向量化） =====================
# 销售额-订单支付
df.loc[in_has('收入') & move_has('订单支付'), '类型'] = '销售额-订单支付'

# 销售额-先用后付
df.loc[in_has('收入') & move_has('订单交易｜先用后付'), '类型'] = '销售额-先用后付'

# 扣点-运费险
df.loc[in_has('支出') & move_has('运费险'), '类型'] = '扣点-运费险'

# 扣点-技术服务费
df.loc[in_has('支出') & move_has('技术服务费'), '类型'] = '扣点-技术服务费'

# 扣点-达人佣金
df.loc[in_has('支出') & move_has('达人佣金'), '类型'] = '扣点-达人佣金'

# 退款-订单退款
df.loc[in_has('支出') & move_has('订单退款'), '类型'] = '退款-订单退款'

# 退款-平台垫退款
df.loc[in_has('支出') & move_has('回补极速退款垫资'), '类型'] = '退款-平台垫退款'


In [ ]:
# ===================== 统一金额赋值 =====================
# 所有规则匹配到的行，金额都取绝对值（和你原逻辑完全一样）
df.loc[df['类型'] != '未知', '金额'] = money.abs()
wx_xd = df.copy()

In [ ]:
wx_xd['收支金额'] = wx_xd['收支金额'].astype(float)

In [ ]:
wx_xd

In [ ]:
wx_xd.类型.value_counts()

### 未知类型

In [ ]:
wx_xd_wz = wx_xd[wx_xd['类型'] == '未知']
wx_xd_wz

### 店铺表格

In [ ]:
wx_xd['店铺'] = wx_xd['店铺'].astype(int)

In [ ]:
wx_xd_1 = pd.merge(wx_xd, dp_bg, left_on = '店铺', right_on = '店铺ID', how = 'left')

In [ ]:
wx_xd_1.rename(columns = {'记账时间':'发生时间'}, inplace = True)

In [ ]:
# 主执行逻辑（仅2行核心调用）
wx_xd_1 = wx_xd_1.copy()
for field in ['合伙人', '品牌']:
    wx_xd_1 = optimize_field_by_date(wx_xd_1, field)

### 主体

In [ ]:
wx_xd_1['店铺ID'] = wx_xd_1['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
wx_xd_2 = pd.merge(wx_xd_1, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
wx_xd_2['发生时间'] = pd.to_datetime(wx_xd_2['发生时间'])
wx_xd_2['新主体'] = wx_xd_2.apply(get_new_subject, axis=1)

In [ ]:
wx_xd_3 = pd.merge(wx_xd_2, zt2, left_on = '新主体', right_on = '主体', how = 'left')
wx_xd_3['身份'] = np.where(wx_xd_3['身份'].isna(), '小规模纳税人', wx_xd_3['身份'])

In [ ]:
wx_xd_4 = wx_xd_3[~(wx_xd_3.发生时间 == '')]

### 汇总

In [ ]:
wx_xd_4['来源文件'] = "微信小店_资金账单"

In [ ]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
wx_xd_5 = wx_xd_4.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型',"来源文件"])[['金额']].sum().round(2).reset_index()

# 重命名列名
wx_xd_5 = wx_xd_5.rename(columns={'店铺名称（聚水潭）':'店铺名称', '调整金额':'金额'})

In [ ]:
wx_xd_5 = wx_xd_5[['店铺名称', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '金额',"来源文件"]].copy()

In [ ]:
wx_xd_5

### 处理后

In [ ]:
# 定义需要保存的DataFrame和对应的文件路径
data_to_save = [

    (wx_xd_wz, fr'../../结果/未知/微信小店账单未知类型.csv'),
    (wx_xd_5, fr'../../结果/2026年{int(month)}月/微信小店/微信小店2.0(待处理).csv'),
]
# 分离DataFrame和文件路径列表
dataframes = [item[0] for item in data_to_save]
file_paths = [item[1] for item in data_to_save]
# 调用函数保存数据
try:
    save_dataframes(dataframes, file_paths)
except Exception as e:
    print(f"保存数据时出错: {e}")

## 阿里

In [ ]:
al = pd.read_sql(f'select * from [支付宝账单]', conn2)

In [ ]:
al['店铺'] = al['店铺'].str.split('_').str[0]

### 匹配店铺表格

In [ ]:
# 一行调用，直接处理整列
al['入账时间'] = convert_date_simple(al['入账时间'])

In [ ]:
dp_bg['店铺ID'] = dp_bg['店铺ID'].astype(str)
al['店铺'] = al['店铺'].astype(str)

In [ ]:
al_1 = pd.merge(al, dp_bg, left_on = '店铺', right_on='店铺ID', how = 'left').rename(columns={'入账时间':'发生时间'})

In [ ]:
# 主执行逻辑（仅2行核心调用）
al_1 = al_1.copy()
for field in ['合伙人', '品牌']:
    al_1 = optimize_field_by_date(al_1, field)

In [ ]:
 al_1 = al_1[(al_1['平台'] == '阿里') & (al_1['发生时间'].str[:7] == f'2026-{month}')]

In [ ]:
al_1['来源文件']  = '阿里支付宝'

### 阿里类型

In [ ]:
al_lx = al_1.copy()

In [ ]:
al_lx.rename(columns = {'收入（+元）':'收入金额', '支出（_元）':'支出金额'}, inplace = True)

In [ ]:
# 处理收入列：先替换空字符串，再替换 NaN，最后转 float
al_lx['收入金额'] = al_lx['收入金额'].replace(' ', '').replace('', 0).fillna(0).astype(float)
# 处理支出列
al_lx['支出金额'] = al_lx['支出金额'].replace(' ', '').replace('', 0).fillna(0).astype(float)

In [ ]:
import pandas as pd
import numpy as np

# 初始化
al_lx['类型'] = '未知'

# 列别名简化
df = al_lx
tx = df['账务类型']
desc = df['业务描述']
note = df['备注']
income = df['收入金额']
outcome = df['支出金额']

# 统一预计算
tx_is = lambda s: tx == s
tx_in = lambda lst: tx.isin(lst)
desc_has = lambda s: desc.str.contains(s, na=False)
note_has = lambda s: note.str.contains(s, na=False)

# ===================== 规则（一行一条，优先级从上到下） =====================
# 在线支付类
df.loc[tx_is('在线支付') & desc_has('交易收款-交易收款'), '类型'] = '销售额-交易收款'
df.loc[tx_is('在线支付') & (df['类型'] == '未知'), '类型'] = '销售额-其他'

# 退款类
df.loc[tx_is('退款'), '类型'] = '退款-售后退款'

# 提现类
df.loc[tx_is('提现'), '类型'] = '提现'

# 保证金类
df.loc[tx_is('保证金') & note_has('买家保障自缴保证金充值'), '类型'] = '保证金-买家保证金'
df.loc[tx_is('保证金') & note_has('产生交易理赔') & (income > 0) & (outcome == 0), '类型'] = '保证金-买家保证金'
df.loc[tx_is('保证金') & note_has('产生交易理赔') & (outcome > 0) & (income == 0), '类型'] = '扣款-理赔'

# 转账扣款类
df.loc[tx_is('转账') & note_has('分销客佣金代扣款'), '类型'] = '扣款-佣金'
df.loc[tx_is('转账') & note_has('1688大分销业务平台抽佣'), '类型'] = '扣款-佣金'
df.loc[tx_is('转账') & note_has('1688供应链服务-跨境渠道'), '类型'] = '扣款-佣金'
df.loc[tx_is('转账') & (df['类型'] == '未知'), '类型'] = '转账'

# 其他类型（按备注细分）
df.loc[tx_is('其它') & note_has('1688增值服务-先采后付担保付款'), '类型'] = '销售额-担保付款'
df.loc[tx_is('其它') & note_has('先采后付买家还款'), '类型'] = '销售额-买家还款'
df.loc[tx_is('其它') & note_has('1688增值买家保障服务费'), '类型'] = '扣款-保障服务费'
df.loc[tx_is('其它') & note_has('先采后付服务费'), '类型'] = '扣点-服务费'

# 收费类型
df.loc[tx_is('收费'), '类型'] = '扣款-手续费'

In [ ]:
al_lx_wz = al_lx[al_lx['类型'] == "未知"]
al_lx_wz

### 计算金额

In [ ]:
al_lx['合计'] = al_lx['支出金额'] - al_lx['收入金额']

In [ ]:
import numpy as np

# 销售类：取 收入金额
sale_types = [
    "销售额-交易收款",
    "销售额-其他",
    "销售额-担保付款",
    "销售额-买家还款"
]

# 合计类（扣款/退款/提现/保证金）：取 支出金额
expense_types = [
    "退款-售后退款",
    "提现",
    "保证金-买家保证金",
    "扣款-理赔",
    "扣款-佣金",
    "扣点-服务费",
    "扣款-保障服务费",
    "转账",
    "扣款-手续费"
]

# 新增「金额」列
al_lx['金额'] = np.select(
    condlist=[
        al_lx['类型'].isin(sale_types),
        al_lx['类型'].isin(expense_types)
    ],
    choicelist=[
        al_lx['收入金额'],
        al_lx['合计']
    ],
    default=0  # 未匹配类型默认 0，也可改为 np.nan
)

### 主体

In [ ]:
al_lx['店铺ID'] = al_lx['店铺ID'].astype(str)
zt3['店铺ID'] = zt3['店铺ID'].astype(str)
al_lx_3 = pd.merge(al_lx, zt3, on = '店铺ID', how = 'left')

In [ ]:
# 主程序
al_lx_3['发生时间'] = pd.to_datetime(al_lx_3['发生时间'])
al_lx_3['新主体'] = al_lx_3.apply(get_new_subject, axis=1)

In [ ]:
al_lx_4 = pd.merge(al_lx_3, zt2, left_on = '新主体', right_on = '主体', how = 'left')
al_lx_4['身份'] = np.where(al_lx_4['身份'].isna(), '小规模纳税人', al_lx_4['身份'])

In [ ]:
al_lx_4

### 汇总

In [ ]:
# 再按分组 求和（这时候 sum 就是你要的“销售额正、其他负”的汇总结果）
al_lx_5 = al_lx_4.groupby(['店铺名称（聚水潭）', '店铺ID', '新主体', '身份', '平台', '合伙人', '品牌', '类型', '来源文件'])[['金额']].sum().round(2).reset_index()

# 重命名列名
al_lx_5 = al_lx_5.rename(columns={'店铺名称（聚水潭）':'店铺名称', '调整金额':'金额'})

In [ ]:
al_lx_5

### 处理后

In [ ]:
# 定义需要保存的DataFrame和对应的文件路径
data_to_save = [

    (al_lx_wz, fr'../../结果/未知/阿里未知类型.csv'),
    (al_lx_5, fr'../../结果/2026年{int(month)}月/阿里/阿里2.0(待处理).csv'),
]
# 分离DataFrame和文件路径列表
dataframes = [item[0] for item in data_to_save]
file_paths = [item[1] for item in data_to_save]
# 调用函数保存数据
try:
    save_dataframes(dataframes, file_paths)
except Exception as e:
    print(f"保存数据时出错: {e}")